# When Foundation Models Fall Short: From Off-the-Shelf AI to Problem-Specific Adaptation

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#4A148C;color:#FFFFFF;border-radius:6px;padding:18px 20px;margin:6px 0;width:100%;box-sizing:border-box;font-family:inherit;line-height:1.5"><div style="font-size:2em;font-weight:bold;line-height:1.2;margin-bottom:6px">When Foundation Models Fall Short</div><p><strong>From Off-the-Shelf AI to Problem-Specific Adaptation</strong></p>
<p>AMLD 2027 · EPFL, 11 March 2027</p>
<p><strong>The question:</strong> what can modern AI solve out of the box, when do its limitations
start to matter, and what should we do next?</p>
<p><strong>At the end of this notebook you will be able to:</strong></p>
<ol>
<li>Judge when a vision foundation model is good enough out of the box, and recognise
   the signs that it is not.</li>
<li>Adapt it with cheap, incomplete supervision (scribbles, missing labels,
   pseudo-labels, explanations, prompts), and explain what each approach uses and assumes.</li>
<li>Compare approaches on accuracy, annotation effort, reliability and compute, and
   decide what to do next on a new problem, including your own images.</li>
</ol>
<p>Levels: 🟢 beginner · 🟡 intermediate (the default path) · 🔴 advanced, optional.
&nbsp; ✍️ = you write a few lines &nbsp; 🖍️ = you draw &nbsp; 👥 = compare with your neighbour.</p>
<p><strong>Run <code>setup_check.ipynb</code> first.</strong> It was shared with you the week
before: it puts the models, images and precomputed files in your Google Drive, so nothing is downloaded
in the room. If you have not run it, do it now (about 5 minutes), then come back here.</p></div>"""))

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#E8F5E9;border-left:6px solid #1B5E20;border-radius:6px;padding:12px 14px;margin:6px 0;color:#1B5E20;font-family:inherit;line-height:1.5"><b>🎯 Objective</b><p>Load the cached models and data; see the question and outcomes.</p></div>"""))

**You need** basic ML (training, validation, evaluation) and some Python, no computer-vision background, and
this notebook in Colab with **Runtime → Change runtime type → T4 GPU**. Every step also runs on a CPU, just
slower, and each training cell offers a precomputed checkpoint.

**The plan.** Block 1 (sections 1 to 5.2): a foundation model segments pets almost perfectly, meets a
specialised task and falls short, then learns from scribbles and from half-finished labels. Block 2 (5.3 to
10), after the coffee break: the PU loss, pseudo-labels, prompting the model directly, all compared on the
same hidden ground truth, then your own images.

## 0. Setup & the question

This cell detects Colab and the GPU, fixes the seed, creates the `RESULTS` dictionary every section writes
into, loads the `tqdm` progress bars used by the long steps, and defines `use()`, so your own code never
blocks the notebook. Run it once.

In [ ]:
# @title Setup (run me first) { display-mode: "form" }
import ast
import inspect
import json
import os
import random
import sys
import time
import weakref
from pathlib import Path

import numpy as np
from tqdm.auto import tqdm  # noqa: F401 - progress bars for the long loops of later sections

IN_COLAB = "google.colab" in sys.modules
SMOKE = os.environ.get("NB_SMOKE") == "1"  # automated test: tiny data, CPU, seconds. Never set this by hand.
SEED = 42

# Where the cache lives. setup_check.ipynb filled MyDrive/fm-workshop-cache/ in Colab; the smoke run uses the
# small fixtures committed in data/smoke/; a local checkout uses data/cache/ (scripts/prepare_data.py).
if IN_COLAB and not SMOKE:
    CACHE_DIR = Path("/content/drive/MyDrive/fm-workshop-cache")
    try:
        from google.colab import drive

        drive.mount("/content/drive")
    except Exception as err:  # noqa: BLE001 - one plain line, then a plain message below
        print(f"(Drive mount error: {type(err).__name__}: {err})")
    if not CACHE_DIR.is_dir():
        print(
            "⚠️ Google Drive is not mounted or the cache folder is missing. Run this cell again and click "
            "'Allow' in the pop-up. If the folder is still missing, run setup_check.ipynb first (5 minutes)."
        )
    if (CACHE_DIR / "models").is_dir():
        os.environ["HF_HUB_OFFLINE"] = "1"  # nothing is downloaded in the room (N-030)
    DATA_DIR, PRECOMPUTED = CACHE_DIR / "data", CACHE_DIR / "precomputed"
    SPECIALISED_JSON = CACHE_DIR / "specialised.json"
else:
    CACHE_DIR = Path("data/smoke" if SMOKE else "data/cache")
    DATA_DIR = CACHE_DIR if SMOKE else CACHE_DIR / "workshop"
    PRECOMPUTED = CACHE_DIR / "precomputed"
    SPECIALISED_JSON = Path("data/specialised.json")

# Model weights: plain files in <cache>/models/<org>--<name>/, written by setup_check.ipynb (no symlinks,
# so they survive Drive). `.fm-complete.json` is written last, once the folder is verified. No folder
# (smoke run, local checkout): the models load from the Hugging Face hub id instead.
MODELS_DIR = CACHE_DIR / "models"
MODEL_DONE = ".fm-complete.json"


def model_folder(repo_id: str) -> Path | None:
    """The cached plain-file folder of a Hugging Face model, or None when setup_check has not made one."""
    folder = MODELS_DIR / repo_id.replace("/", "--")
    return folder if (folder / MODEL_DONE).is_file() else None


def weights_file(folder: Path) -> Path:
    """The weight file inside a model folder (timm: model.safetensors)."""
    default = folder / "model.safetensors"
    return default if default.is_file() else sorted(folder.glob("*.safetensors"))[0]


if IN_COLAB:
    try:
        import pytorch_grad_cam  # noqa: F401 - the only package Colab lacks (SPEC §5); installed if missing
    except ImportError:
        get_ipython().run_line_magic("pip", "install -q grad-cam")  # noqa: F821 - %pip, inside a cell

import torch

if SMOKE:
    DEVICE = "cpu"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"
    print(
        "ℹ️ No GPU in this session, so everything runs on the CPU: fine, but slower. For a GPU, use "
        "Runtime → Change runtime type → T4 GPU, then run this cell again. Every training cell also "
        "offers our precomputed checkpoint."
    )


def seed_everything(seed: int = SEED) -> None:
    """Fix every random source, so re-running a section from its first cell gives the same result (N-013)."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything()

RESULTS: dict = {"use_fallbacks": {}}  # every section writes its named numbers here (SPEC §9)


def _is_stub(fn) -> bool:
    """True when the function body is still the `...` placeholder (optionally after a docstring)."""
    try:
        tree = ast.parse(inspect.getsource(fn).strip() or "pass")
    except (OSError, TypeError, SyntaxError):
        return False
    node = tree.body[0]
    if not isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
        return False
    def const(n, kind):
        is_const = isinstance(n, ast.Expr) and isinstance(n.value, ast.Constant)
        return is_const and isinstance(n.value.value, kind)

    body = [n for n in node.body if not const(n, str)]  # drop the docstring
    return all(const(n, type(...)) for n in body)


def _same(a, b) -> bool:
    """Compare two results of any common type (numbers, arrays, tensors, dicts, sequences)."""
    if isinstance(a, torch.Tensor) or isinstance(b, torch.Tensor):
        a = a.detach().cpu().numpy() if isinstance(a, torch.Tensor) else a
        b = b.detach().cpu().numpy() if isinstance(b, torch.Tensor) else b
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(_same(a[k], b[k]) for k in a)
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        return len(a) == len(b) and all(_same(x, y) for x, y in zip(a, b, strict=True))
    try:
        return bool(np.allclose(np.asarray(a, dtype=float), np.asarray(b, dtype=float), atol=1e-5))
    except (TypeError, ValueError):
        return a == b


def use(fn, reference, check=None, name: str | None = None):
    """Return the function the notebook continues with: yours if it matches the reference, else the reference.

    `check(f)` runs `f` on a small play example and returns what to compare (any numbers, arrays or dicts).
    Nothing here is mandatory: an untouched stub (`...` body) silently selects the reference (N-029).
    """
    name = name or fn.__name__
    if _is_stub(fn):
        RESULTS["use_fallbacks"][name] = True
        print(f"ℹ️ {name}: using the reference solution. Fill the stub above to use your own version.")
        return reference
    if check is None:
        RESULTS["use_fallbacks"][name] = False
        print(f"✅ {name}: using your version (no automatic check for this one).")
        return fn
    try:
        yours, theirs = check(fn), check(reference)
    except Exception as err:  # noqa: BLE001 - the participant's code may fail in any way; keep going
        RESULTS["use_fallbacks"][name] = True
        print(f"❌ {name}: your version raised {type(err).__name__}: {err}. Continuing with the reference.")
        return reference
    if _same(yours, theirs):
        RESULTS["use_fallbacks"][name] = False
        print(f"✅ {name}: matches the reference. Continuing with your version.")
        return fn
    RESULTS["use_fallbacks"][name] = True
    print(f"❌ {name}: {yours} vs reference {theirs}. Continuing with the reference.")
    return reference


if not SPECIALISED_JSON.is_file():
    raise SystemExit(f"{SPECIALISED_JSON} is missing. Run setup_check.ipynb first, then run this cell again.")
specialised: dict = json.loads(SPECIALISED_JSON.read_text())  # the specialised task; never hard-coded
specialised["dir"] = DATA_DIR / "specialised"
pets_task: dict = {"key": "pets", "name": "Oxford-IIIT Pet", "task": "pet vs background"}
pets_task |= {"licence": "CC BY-SA 4.0", "dir": DATA_DIR / "pets"}

print(f"Colab: {IN_COLAB} · smoke mode: {SMOKE} · device: {DEVICE} · seed: {SEED}")
print(f"cache: {CACHE_DIR} · specialised task: {specialised['name']} ({specialised['task']})")

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#EDE7F6;border-left:6px solid #311B92;border-radius:6px;padding:12px 14px;margin:6px 0;color:#311B92;font-family:inherit;line-height:1.5"><b>📘 Theory: how use() works</b><p>Some cells ask you to fill a function whose body is <code>...</code> (the first is <code>my_union</code> in section 1). The
line after it, <code>union_pixels = use(my_union, reference=_ref_union, ...)</code>, runs your version and ours on a
small play example:</p>
<ul>
<li>Stub untouched: the notebook continues with ours.</li>
<li>✅ Yours matches: it continues with yours.</li>
<li>❌ Yours differs or crashes: it prints both results and continues with ours.</li>
</ul>
<p>Writing the code is where the learning happens; skipping it never breaks anything.</p></div>"""))

### The theme

This cell sets the colours used in every figure: $\color{#6A1B9A}{\blacksquare}$ violet means "it works",
$\color{#E65100}{\blacksquare}$ orange "it fails, we adapt",
$\color{#2E7D32}{\blacksquare}$ green is the hidden ground truth, $\color{#00838F}{\blacksquare}$ teal
marks what a person draws (strokes, clicks, boxes) and $\color{#1565C0}{\blacksquare}$ blue is a model's
score before any threshold (a probability or similarity map), the same in every plot. Every panel carries a
tag in its corner: INPUT (what the model is given), PREDICTION, TRUTH (scoring only) or SCORE MAP.
Masks are drawn on a
$\color{#9E9E9E}{\blacksquare}$ grey copy of the photo, so every colour you see belongs to a mask and none
to the photo.

In [ ]:
# @title Theme and image helpers { display-mode: "form" }
import matplotlib as mpl
import matplotlib.pyplot as plt
from PIL import Image

THEME = {
    "works": "#6A1B9A",  # violet: the foundation model does the job
    "fails": "#E65100",  # orange: it falls short / we adapt
    "truth": "#2E7D32",  # green: hidden ground truth, evaluation only
    "scribble": "#00838F",  # teal: strokes a person drew
    "score": "#1565C0",  # blue: a model's score before the threshold (probability, similarity, heat map)
    "background": "#9E9E9E",  # grey overlay
    "banner": "#4A148C",
}
mpl.rcParams.update(
    {
        "figure.dpi": 100,
        "figure.facecolor": "white",
        "axes.titlesize": 10,
        "axes.titlecolor": "#311B92",
        "axes.prop_cycle": mpl.cycler(color=[THEME[k] for k in ("works", "fails", "truth", "scribble")]),
    }
)
# one scale for every score map: dark grey = 0 (sure it is not), white = 0.5 (unsure), blue = 1 (sure it is)
score_cmap = mpl.colors.LinearSegmentedColormap.from_list("score", ["#424242", "white", THEME["score"]])


def hex_rgb(colour: str) -> np.ndarray:
    """'#RRGGBB' -> float array in [0, 1]."""
    return np.array([int(colour[i : i + 2], 16) for i in (1, 3, 5)]) / 255


_OVERLAID = weakref.WeakValueDictionary()  # images overlay() made, so stacking a second mask never re-greys


def overlay(
    image: np.ndarray,
    mask: np.ndarray,
    colour: str = THEME["truth"],
    alpha: float = 0.75,
    unscored: np.ndarray | None = None,
    dim: bool | None = None,
):
    """Draw a binary mask in one theme colour on a grey, dimmed copy of an RGB uint8 image.

    The photo is turned grey (0.35 to 0.85 brightness) so an orange or green mask stays visible on a red
    retina or a brown dog. `unscored` pixels (not evaluated) are painted light grey. Stacking a second mask
    on the result of a first call keeps the first mask's colour (`dim` is then False unless you set it).
    """
    if dim is None:
        dim = _OVERLAID.get(id(image)) is not image
    out = image.astype(float) / 255
    if dim:
        grey = out @ np.array([0.299, 0.587, 0.114])
        out = np.repeat((0.35 + 0.5 * grey)[..., None], 3, axis=2)
    if unscored is not None:
        out[unscored] = 0.3 * out[unscored] + 0.7 * 0.88
    out[mask] = (1 - alpha) * out[mask] + alpha * hex_rgb(colour)
    result = (out * 255).astype(np.uint8)
    _OVERLAID[id(result)] = result
    return result


def region(item: dict) -> np.ndarray:
    """Where we score: the field of view if the dataset has one, else the whole image."""
    return item["fov"] if item.get("fov") is not None else np.ones(item["image"].shape[:2], bool)


def read_png(path: Path) -> np.ndarray:
    return np.array(Image.open(path).convert("L"))


def load_item(dataset: dict, item: dict) -> dict:
    """Read one image with its full mask (evaluation only!), scribbles, visible labels and FOV."""
    root = dataset["dir"]
    out = {
        "name": item["name"],
        "split": item["split"],
        "image": np.array(Image.open(root / item["image"]).convert("RGB")),
        "mask": read_png(root / item["mask"]) > 127,
        "scribbles": read_png(root / item["scribbles"]),
        "visible": read_png(root / item["visible"]) > 127,
        "fov": read_png(root / item["fov"]) > 127 if "fov" in item else None,
    }
    out["hidden"] = out["mask"] & ~out["visible"]
    return out


def load_index(dataset: dict) -> dict:
    """Attach the dataset's index.json (facts + one entry per image) and return it."""
    index_file = dataset["dir"] / "index.json"
    if not index_file.is_file():
        raise SystemExit(f"{index_file} is missing. Run setup_check.ipynb first, then run this cell again.")
    dataset["index"] = json.loads(index_file.read_text())
    dataset["items"] = dataset["index"]["items"]
    dataset["fit"] = [it for it in dataset["items"] if it["split"] == "fit"]
    dataset["held"] = [it for it in dataset["items"] if it["split"] == "held"]
    return dataset["index"]

### The data

This cell reads the two datasets (pets, and the specialised task) and shows two images of each with the
ground truth in $\color{#2E7D32}{\blacksquare}$ green and its share of the pixels in the title; the ground
truth is for evaluation only.
$\color{#E0E0E0}{\blacksquare}$ Light grey marks pixels nobody scores: on the pets, the "not sure" band
that the annotators drew around the
outline of each animal.

In [ ]:
# @title Load both datasets and show two examples of each { display-mode: "form" }
for ds in (pets_task, specialised):
    idx = load_index(ds)
    print(
        f"{ds['name']}: {len(ds['items'])} images ({len(ds['fit'])} fit, {len(ds['held'])} held-out) · "
        f"{idx['task']}"
    )
PRECOMPUTED_HEADS = json.loads((PRECOMPUTED / "heads" / "index.json").read_text())["heads"]
print(f"precomputed: {len(PRECOMPUTED_HEADS)} head checkpoints in {PRECOMPUTED}")

fig, axes = plt.subplots(2, 4, figsize=(12, 6.4))
for k, ds in enumerate((pets_task, specialised)):
    for j, item in enumerate(ds["fit"][:2]):
        it = load_item(ds, item)
        axes[k, 2 * j].imshow(it["image"])
        axes[k, 2 * j].set_title(f"{ds['name']}: {it['name']}")
        not_scored = ~region(it) if ds is pets_task else None  # the pets' "not sure" band
        axes[k, 2 * j + 1].imshow(overlay(it["image"], it["mask"], unscored=not_scored))
        axes[k, 2 * j + 1].set_title(f"ground truth · {it['mask'].mean():.0%} of the pixels")
for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F3E5F5;border-left:6px solid #4A148C;border-radius:6px;padding:12px 14px;margin:6px 0;color:#4A148C;font-family:inherit;line-height:1.5"><b>🤔 Your prediction</b><p>Look at the images above. Which task do you expect the foundation model to solve out of the box, and which
one will give it trouble? Write one reason; section 2 will tell you whether you were right.</p>
<p><strong>Q1.</strong> 🟢 What is different about the specialised task that could matter for a model trained on everyday
photos? Look at the size, shape and contrast of the objects, not the domain name.</p>
<details>
<summary>One possible answer</summary>
<p>Pets are big and clear; the specialised targets are thin, faint and small.</p>
</details></div>"""))

### The models

This cell defines two wrappers: `dino.features(image, long_side)` gives a grid of image features and
`sam.probability(image, points, labels)` gives a foreground map from point prompts.

In [ ]:
# @title Model loaders { display-mode: "form" }
import timm
import torch.nn.functional as F

DINO_ID = "vit_small_patch16_dinov3.lvd1689m"
DINO_SOURCE = "timm/vit_small_patch16_dinov3.lvd1689m"
DINO_LICENCE = "DINOv3 License (Meta)"
SAM_ID = "facebook/sam2.1-hiera-small"  # SPEC §5: SAM 2.1 hiera-small, Apache-2.0, no account needed
SAM_LICENCE = "Apache-2.0"


class Dino:
    """Frozen DINOv3 ViT-S/16: an image in, a grid of 384-dim patch features out."""

    def __init__(self, device: str = DEVICE):
        t = time.monotonic()
        self.device = device
        folder = model_folder(DINO_SOURCE)  # plain files on Drive, else the hub id
        extra = {"pretrained_cfg_overlay": {"file": str(weights_file(folder))}} if folder else {}
        model = timm.create_model(DINO_ID, pretrained=True, num_classes=0, dynamic_img_size=True, **extra)
        self.model = model.eval().to(device)
        cfg = timm.data.resolve_data_config({}, model=self.model)
        self.mean = torch.tensor(cfg["mean"]).view(1, 3, 1, 1).to(device)
        self.std = torch.tensor(cfg["std"]).view(1, 3, 1, 1).to(device)
        self.n_prefix = self.model.num_prefix_tokens
        self.dim = self.model.num_features
        dt = time.monotonic() - t
        print(f"DINOv3 ViT-S/16 · {DINO_SOURCE} · {DINO_LICENCE} · {dt:.0f} s")

    @torch.no_grad()
    def features(self, image: np.ndarray, long_side: int) -> torch.Tensor:
        """Patch features [D, h, w] for the image resized so its long side is `long_side` (multiple of 16)."""
        x = torch.from_numpy(np.ascontiguousarray(image)).permute(2, 0, 1).float().unsqueeze(0) / 255
        h, w = image.shape[:2]
        s = long_side / max(h, w)
        H, W = max(16, round(h * s / 16) * 16), max(16, round(w * s / 16) * 16)
        x = F.interpolate(x.to(self.device), size=(H, W), mode="bilinear", align_corners=False)
        x = (x - self.mean) / self.std
        f = self.model.forward_features(x)[0, self.n_prefix :]
        return f.reshape(H // 16, W // 16, -1).permute(2, 0, 1).contiguous().float()


class Sam:
    """SAM 2.1 (promptable segmentation): an image plus points or a box in, a foreground probability out."""

    def __init__(self, device: str = DEVICE):
        from transformers import Sam2Model, Sam2Processor

        t = time.monotonic()
        self.device = device
        src = str(model_folder(SAM_ID) or SAM_ID)  # plain files on Drive, else the hub id
        self.proc = Sam2Processor.from_pretrained(src)
        self.model = Sam2Model.from_pretrained(src).eval().to(device)
        self._cache: tuple[np.ndarray, torch.Tensor] | None = None  # (the last image, its embeddings)
        print(f"SAM 2.1 hiera-small · {SAM_ID} · {SAM_LICENCE} · {time.monotonic() - t:.0f} s")

    @torch.no_grad()
    def probability(
        self,
        image: np.ndarray,
        points: list[list[float]] | None = None,
        labels: list[int] | None = None,
        box: list[float] | None = None,
    ) -> np.ndarray:
        """Foreground probability (H x W float) for points (x, y; label 1 = fg, 0 = bg) and/or a box."""
        kw = {}
        if points:
            kw["input_points"], kw["input_labels"] = [[points]], [[labels]]
        if box is not None:
            kw["input_boxes"] = [[box]]
        inputs = self.proc(images=image, return_tensors="pt", **kw)
        if self._cache is None or self._cache[0] is not image:
            self._cache = (image, self.model.get_image_embeddings(inputs["pixel_values"].to(self.device)))
        out = self.model(
            image_embeddings=self._cache[1],
            input_points=inputs["input_points"].to(self.device) if points else None,
            input_labels=inputs["input_labels"].to(self.device) if points else None,
            input_boxes=inputs["input_boxes"].to(self.device) if box is not None else None,
            multimask_output=False,
        )
        masks = out.pred_masks.cpu()
        logits = self.proc.post_process_masks(masks, inputs["original_sizes"], binarize=False)[0]
        return torch.sigmoid(logits[0, 0]).numpy()

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#FDECEA;border-left:6px solid #8B1A10;border-radius:6px;padding:12px 14px;margin:6px 0;color:#8B1A10;font-family:inherit;line-height:1.5"><b>⚠️ Pitfall: the first slow step</b><p>Loading and the first call of a model are the slow steps: PyTorch compiles kernels the first time a model
sees a new image size. On a T4 this takes seconds; on a CPU it is much slower. Later calls are faster.</p></div>"""))

This cell loads DINOv3 and SAM from the cache and prints one line each: name · source · licence · load time.

In [ ]:
# @title Load DINOv3 and SAM { display-mode: "form" }
dino = Dino()
sam = Sam()
RESULTS["models"] = {"dino": DINO_SOURCE, "sam": SAM_ID, "device": DEVICE}

This cell defines the interactive SAM tool used in sections 1, 2 and 7: click a dot, drag a box, or (in
section 7) type a word, and SAM answers with a mask you can compare with the ground truth. Run it once.

In [ ]:
# @title Interactive SAM: click, box, or a word { display-mode: "form" }
import base64
import io
import uuid
from string import Template

from IPython.display import HTML, JSON, display

PLAY_PX = 560  # long side of the playground image on screen
SAM3_ID = "facebook/sam3"
HEAD_STEPS = 30 if SMOKE else 200  # training steps for every linear head in the notebook
_SAM3: dict = {}  # SAM 3 is loaded on the first text request and kept here


# what transformers needs from the SAM 3 repo, not the duplicate sam3.pt
SAM3_FILES = ["*.json", "*.txt", "model.safetensors", "*.model"]


def hf_token() -> str | None:
    """Your Hugging Face token: the HF_TOKEN environment variable, else Colab's 🔑 Secrets, else None."""
    if os.environ.get("HF_TOKEN"):
        return os.environ["HF_TOKEN"]
    if IN_COLAB:
        try:
            from google.colab import userdata

            return userdata.get("HF_TOKEN")
        except Exception:  # noqa: BLE001 - no secret, or notebook access switched off
            return None
    return None


def fetch_sam3() -> Path | None:
    """SAM 3 not in the cache but a token is set: download it once into the cache, as setup_check would."""
    import hashlib

    from huggingface_hub import snapshot_download

    token = hf_token()
    if SMOKE or token is None:
        return None
    folder = MODELS_DIR / SAM3_ID.replace("/", "--")
    print(f"SAM 3 is not in your cache yet: downloading it once (~3.4 GB, a few minutes) into {folder}")
    snapshot_download(SAM3_ID, local_dir=folder, token=token, allow_patterns=SAM3_FILES)

    def sha256_of(path: Path) -> str:
        h = hashlib.sha256()
        with open(path, "rb") as f:
            for block in iter(lambda: f.read(1 << 20), b""):
                h.update(block)
        return h.hexdigest()

    files = [
        p for p in sorted(folder.rglob("*")) if p.is_file() and ".cache" not in p.relative_to(folder).parts
    ]
    meta = {
        p.relative_to(folder).as_posix(): {"bytes": p.stat().st_size, "sha256": sha256_of(p)} for p in files
    }
    (folder / MODEL_DONE).write_text(json.dumps(meta, indent=1))  # the same marker setup_check writes
    return model_folder(SAM3_ID)


def load_sam3() -> dict:
    """SAM 3 from the cache, downloaded once if a token is set: {"model", "proc", "dtype"} or {"error"}."""
    if _SAM3:
        return _SAM3
    folder = model_folder(SAM3_ID)
    if folder is None:
        try:
            folder = fetch_sam3()
        except Exception as err:  # noqa: BLE001 - 403 while approval is pending, no network, a full Drive
            pending = "403" in str(err) or "authorized" in str(err) or "awaiting" in str(err)
            _SAM3["error"] = (
                f"SAM 3 could not be downloaded: Meta has not approved your access to {SAM3_ID} yet."
                if pending
                else f"SAM 3 could not be downloaded ({type(err).__name__}). SAM 2.1 still works."
            )
            return _SAM3
    try:
        from transformers import Sam3Model, Sam3Processor

        dtype = torch.float16 if DEVICE == "cuda" else torch.float32  # fp16 on a T4, never bf16
        src = str(folder or SAM3_ID)
        proc = Sam3Processor.from_pretrained(src, local_files_only=True)
        model = Sam3Model.from_pretrained(src, local_files_only=True, dtype=dtype).eval().to(DEVICE)
        _SAM3.update(model=model, proc=proc, dtype=dtype)
    except Exception as err:  # noqa: BLE001 - not cached, no access, an older transformers: all mean "no SAM 3"
        if folder is None:
            _SAM3["error"] = (
                f"SAM 3 is not in your cache and no HF_TOKEN is set: request access at "
                f"huggingface.co/{SAM3_ID} (Meta approves by hand), add your token in Colab's 🔑 Secrets as "
                "HF_TOKEN with notebook access on, and run this cell again."
            )
        else:
            _SAM3["error"] = (
                f"SAM 3 is in your cache but did not load ({type(err).__name__}: "
                f"{str(err).splitlines()[0][:120] if str(err) else 'no details'}). SAM 2.1 still works."
            )
    return _SAM3


def sam3_masks(image: np.ndarray, text: str, threshold: float = 0.3) -> np.ndarray:
    """Every instance SAM 3 finds for a short noun phrase: bool array [n, H, W] (n may be 0)."""
    s3 = load_sam3()
    if "error" in s3:
        raise RuntimeError(s3["error"])
    inputs = s3["proc"](images=image, text=text.strip(), return_tensors="pt").to(DEVICE)
    inputs["pixel_values"] = inputs["pixel_values"].to(s3["dtype"])
    with torch.no_grad():
        out = s3["model"](**inputs)
    found = s3["proc"].post_process_instance_segmentation(
        out, threshold=threshold, mask_threshold=0.5, target_sizes=[image.shape[:2]]
    )[0]
    return found["masks"].bool().cpu().numpy()


def sam_playground(items: list[dict], title: str, text_prompt: bool = False, default_text: str = "") -> None:
    """Click, box or word prompts on `items` (dicts from load_item); SAM 2.1 answers, with IoU vs the truth.

    Left click = object dot, shift-click = background dot, drag = box. With `text_prompt=True` a text box asks
    SAM 3 (if it is in your cache). "Show ground truth" (off by default, only for images with a mask) turns
    the overlay into an error map: violet found, orange invented, green missed, light grey not scored.
    Outside Colab and in the smoke run it shows one fixed click and one fixed box on the first image
    instead, so nothing blocks. All the code lives in this function on purpose.
    """
    page = Template("""
<div style="font-family:sans-serif"><b>$title</b><br>
<div id="st$uid" style="position:relative;margin:4px 0">
<img id="im$uid" style="position:absolute;left:0;top:0">
<img id="mk$uid" style="position:absolute;left:0;top:0;display:none;pointer-events:none">
<canvas id="cv$uid" style="position:absolute;left:0;top:0;cursor:crosshair;touch-action:none"></canvas></div>
<div id="ctl$uid"><button id="pv$uid">◀</button> <button id="nx$uid">▶</button>
<button id="pen$uid">➕ object dot</button> <button id="clr$uid">clear</button> $ask
<button id="gt$uid" style="display:none">👁 Show ground truth</button>
<span id="nm$uid"></span></div>
<div id="msg$uid" style="margin-top:4px;min-height:1.4em">Click = object dot · shift-click = background dot ·
drag = box.</div>
<div id="lg$uid" style="display:none;margin:2px 0;font-size:13px">
<span style="background:$c_found;color:#fff;padding:1px 6px;border-radius:3px">found (SAM and truth)</span>
<span style="background:$c_invented;color:#fff;padding:1px 6px;border-radius:3px">invented (SAM only)</span>
<span style="background:$c_missed;color:#fff;padding:1px 6px;border-radius:3px">missed (truth only)</span>
<span style="background:$c_unscored;color:#333;padding:1px 6px;border-radius:3px">not scored</span></div>
<div style="color:#666;font-size:12px">covers N % = how much of the scored area (the field of view if the
image has one, else the whole image) SAM's mask covers; "truth M %" = how much the true object
covers.</div></div>
<script>
(function () {
  var D = $data, uid = "$uid", k = 0, pts = [], box = null, bg = false, seq = 0, start = null, cur = null,
      gt = false, last = "";
  function el(s) { return document.getElementById(s + uid); }
  var im = el("im"), mk = el("mk"), cv = el("cv"), ctx = cv.getContext("2d"), msg = el("msg");
  function size(node, w, h) { node.style.width = w + "px"; node.style.height = h + "px"; }
  function xy(e) {
    var r = cv.getBoundingClientRect();
    return [(e.clientX - r.left) * cv.width / r.width, (e.clientY - r.top) * cv.height / r.height];
  }
  function draw() {
    ctx.clearRect(0, 0, cv.width, cv.height); ctx.lineWidth = 2;
    pts.forEach(function (p) {
      ctx.beginPath(); ctx.arc(p[0], p[1], 6, 0, 6.3);
      ctx.fillStyle = p[2] ? "#fff" : "#000"; ctx.strokeStyle = p[2] ? "$fg" : "#fff";
      ctx.fill(); ctx.stroke();
    });
    var b = start ? [start[0], start[1], cur[0], cur[1]] : box;
    if (b) {
      ctx.setLineDash([6, 4]); ctx.strokeStyle = "#fff"; ctx.strokeRect(b[0], b[1], b[2] - b[0], b[3] - b[1]);
      ctx.strokeStyle = "$fg"; ctx.lineDashOffset = 5; ctx.strokeRect(b[0], b[1], b[2] - b[0], b[3] - b[1]);
      ctx.setLineDash([]); ctx.lineDashOffset = 0;
    }
  }
  function reset() {
    var d = D[k]; pts = []; box = null; seq++;
    im.src = "data:image/jpeg;base64," + d.b64; im.style.filter = ""; mk.style.display = "none";
    cv.width = d.w; cv.height = d.h; size(im, d.w, d.h); size(mk, d.w, d.h); size(cv, d.w, d.h);
    size(el("st"), d.w, d.h); el("nm").textContent = d.name + " (" + (k + 1) + "/" + D.length + ")";
    msg.textContent = "Click = object dot · shift-click = background dot · drag = box."; draw();
    last = ""; el("gt").style.display = d.gt ? "" : "none";
    el("lg").style.display = (gt && d.gt) ? "" : "none";
    if (gt && d.gt) run("");
  }
  function run(text) {
    if (!(window.google && google.colab && google.colab.kernel)) {
      msg.textContent = "Needs a live Colab session."; return;
    }
    var mine = ++seq; msg.textContent = "SAM is thinking..."; last = text || "";
    google.colab.kernel.invokeFunction("fm_workshop.sam_play_" + uid,
                                       [k, pts, box, text || "", gt && D[k].gt], {})
      .then(function (r) {
        if (mine !== seq) return;
        var o = r.data["application/json"]; msg.textContent = o.line;
        if (o.png) {
          mk.src = o.png; mk.style.display = "block"; im.style.filter = "grayscale(1) brightness(.75)";
        }
        else { mk.style.display = "none"; im.style.filter = ""; }
      })
      .catch(function (err) { msg.textContent = "Error: " + err; });
  }
  cv.addEventListener("pointerdown", function (e) {
    start = cur = xy(e); cv.setPointerCapture(e.pointerId);
  });
  cv.addEventListener("pointermove", function (e) { if (start) { cur = xy(e); draw(); } });
  cv.addEventListener("pointerup", function (e) {
    if (!start) return;
    var s = start, c = xy(e); start = null;
    if (Math.hypot(c[0] - s[0], c[1] - s[1]) < 8) pts.push([s[0], s[1], (e.shiftKey || bg) ? 0 : 1]);
    else box = [Math.min(s[0], c[0]), Math.min(s[1], c[1]), Math.max(s[0], c[0]), Math.max(s[1], c[1])];
    draw(); run("");
  });
  el("pv").onclick = function () { k = (k + D.length - 1) % D.length; reset(); };
  el("nx").onclick = function () { k = (k + 1) % D.length; reset(); };
  el("pen").onclick = function () {
    bg = !bg; this.textContent = bg ? "➖ background dot" : "➕ object dot";
  };
  el("clr").onclick = reset;
  el("gt").onclick = function () {
    gt = !gt; this.textContent = gt ? "🙈 Hide ground truth" : "👁 Show ground truth";
    el("lg").style.display = gt ? "" : "none";
    if (gt || pts.length || box || last) run(last);
    else { seq++; mk.style.display = "none"; im.style.filter = ""; }
  };
  if (el("ask")) el("ask").onclick = function () {
    var t = el("txt").value.trim();
    if (t) run(t); else msg.textContent = "Type a word first, for example the object you want.";
  };
  reset();
})();
</script>
""")

    UNSCORED = "#D0D0D0"  # light grey: outside the scored region

    def score(item: dict, pred: np.ndarray, truth: bool = False) -> str:
        """'IoU vs ground truth 0.83 · covers 12 % of the image (truth 10 %)'; `truth` adds 'found 90 % of the
        truth'."""
        r = region(item)
        line = f"covers {(pred & r).sum() / max(1, r.sum()):.0%} of the image"
        if item.get("mask") is not None:
            line += f" (truth {(item['mask'] & r).sum() / max(1, r.sum()):.0%})"
            union = ((pred | item["mask"]) & r).sum()
            line = f"IoU vs ground truth {((pred & item['mask']) & r).sum() / max(1, union):.2f} · " + line
            if truth:
                found = (pred & item["mask"] & r).sum() / max(1, (item["mask"] & r).sum())
                line += f" · found {found:.0%} of the truth"
        return line

    def error_classes(item: dict, pred: np.ndarray | None) -> list[tuple[np.ndarray, str]]:
        """(mask, colour) pairs of the error map; `pred=None` shows just the truth, in green."""
        r, truth = region(item), item["mask"]
        if pred is None:
            hit, extra, miss = np.zeros_like(truth), np.zeros_like(truth), truth & r
        else:
            hit, extra, miss = pred & truth & r, pred & ~truth & r, ~pred & truth & r
        return [(hit, THEME["works"]), (extra, THEME["fails"]), (miss, THEME["truth"]), (~r, UNSCORED)]

    def predict(item: dict, pts: list, box: list | None, text: str, scale: float) -> np.ndarray:
        """The binary mask SAM 3 (a word) or SAM 2.1 (dots and box, in image pixels) gives."""
        if text.strip():
            found = sam3_masks(item["image"], text)
            return found.any(0) if len(found) else np.zeros(item["image"].shape[:2], bool)
        prob = sam.probability(
            item["image"],
            points=[[x * scale, y * scale] for x, y, _ in pts] or None,
            labels=[int(label) for _, _, label in pts] or None,
            box=[v * scale for v in box] if box else None,
        )
        return prob > 0.5

    if not IN_COLAB or SMOKE:  # no browser to talk to: one fixed click and one fixed box, as a figure
        item = items[0]
        ys, xs = np.nonzero(item["mask"]) if item.get("mask") is not None else (np.array([]), np.array([]))
        if len(xs):
            near = np.argmin((xs - xs.mean()) ** 2 + (ys - ys.mean()) ** 2)
            click = [float(xs[near]), float(ys[near])]
            box = [float(xs.min()), float(ys.min()), float(xs.max()), float(ys.max())]
        else:
            h, w = item["image"].shape[:2]
            click, box = [w / 2, h / 2], [w / 4, h / 4, 3 * w / 4, 3 * h / 4]
        panels = [
            ("click", sam.probability(item["image"], points=[click], labels=[1]) > 0.5),
            ("box", sam.probability(item["image"], box=box) > 0.5),
        ]
        has_truth = item.get("mask") is not None
        fig, axes = plt.subplots(1, 3, figsize=(10, 3.7))
        axes[0].imshow(item["image"])
        axes[0].plot(*click, "o", color="white", mec=THEME["works"])
        wh = (box[2] - box[0], box[3] - box[1])
        axes[0].add_patch(plt.Rectangle(box[:2], *wh, fill=False, ec="white", ls="--"))
        axes[0].set_title(f"{title}\n{item['name']}\none click (dot), one box (dashed)", fontsize=8)
        for ax, (how, pred) in zip(axes[1:], panels, strict=True):
            if has_truth:  # the error map: violet found, orange invented, green missed, light grey not scored
                shown_map = item["image"]
                for mask, colour in error_classes(item, pred)[:3]:
                    shown_map = overlay(shown_map, mask, colour)
                shown_map = overlay(shown_map, ~region(item), UNSCORED, alpha=0.7, dim=False)
                ax.imshow(shown_map)
                parts = score(item, pred, truth=True).split(" · ")
                lines = "\n".join([" · ".join(parts[:2]), *parts[2:]])
                ax.set_title(f"SAM, one {how}\n{lines}", fontsize=8)
            else:
                ax.imshow(overlay(item["image"], pred, THEME["works"]))
                ax.set_title(f"SAM, one {how}\n{score(item, pred)}", fontsize=8)
        for ax in axes:
            ax.axis("off")
        if has_truth:
            from matplotlib.patches import Patch

            keys = [("found (SAM and truth)", THEME["works"]), ("invented (SAM only)", THEME["fails"])]
            keys += [("missed (truth only)", THEME["truth"]), ("not scored", UNSCORED)]
            fig.legend(
                handles=[Patch(fc=c, label=n) for n, c in keys],
                loc="lower center",
                bbox_to_anchor=(0.5, 0.06),
                ncol=4,
                fontsize=8,
                frameon=False,
            )
            note = "covers N % = how much of the scored area SAM's mask covers, next to the true object's."
            fig.text(0.5, 0.005, note, ha="center", fontsize=7)
        plt.tight_layout(rect=(0, 0.16 if has_truth else 0, 1, 0.97))
        plt.show()
        word = ", or ask SAM 3 with a word" if text_prompt else ""
        print(f"In Colab this is a live canvas: click, shift-click, drag a box{word}.")
        return

    uid = uuid.uuid4().hex[:8]
    shown = []  # per image: the JPEG the canvas shows and its on-screen size
    for item in items:
        h, w = item["image"].shape[:2]
        s = PLAY_PX / max(h, w)
        size = (round(w * s), round(h * s))
        buf = io.BytesIO()
        Image.fromarray(item["image"]).resize(size).save(buf, format="JPEG", quality=85)
        b64 = base64.b64encode(buf.getvalue()).decode()
        shown.append(
            {"name": item["name"], "w": size[0], "h": size[1], "b64": b64, "gt": item.get("mask") is not None}
        )
    colour = hex_rgb(THEME["works"]) * 255

    def to_png(layers: list[tuple[np.ndarray, np.ndarray]], d: dict) -> str:
        """A transparent PNG from (mask, rgb colour) layers, with soft edges after shrinking to the screen."""
        acc, wsum = np.zeros((d["h"], d["w"], 3)), np.zeros((d["h"], d["w"]))
        for mask, rgb in layers:
            small = Image.fromarray(mask.astype(np.uint8) * 255).resize((d["w"], d["h"]), Image.BILINEAR)
            a = np.asarray(small)
            acc += (a / 255.0)[..., None] * rgb
            wsum += a / 255.0
        rgb_out = np.where(wsum[..., None] > 0, acc / np.maximum(wsum, 1e-6)[..., None], 0)
        rgba = np.dstack([rgb_out.astype(np.uint8), (np.minimum(wsum, 1) * 0.7 * 255).astype(np.uint8)])
        buf = io.BytesIO()
        Image.fromarray(rgba).save(buf, format="PNG")
        return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()

    def answer(k, pts, box, text, truth=False) -> JSON:
        """Called by the canvas: run SAM for image k and send back a transparent overlay PNG plus one line.

        With `truth` on (and a mask for this image) the overlay is the error map, else the plain violet mask.
        """
        item, d = items[int(k)], shown[int(k)]
        truth = bool(truth) and item.get("mask") is not None
        asked = bool(text.strip() or pts or box is not None)
        if not asked and not truth:
            return JSON({"png": "", "line": "Click, shift-click or drag first."})
        if not asked:  # the truth alone, in green
            layers = [(m, hex_rgb(c) * 255) for m, c in error_classes(item, None)]
            line = "Ground truth in green. Click, shift-click or drag to ask SAM."
            return JSON({"png": to_png(layers, d), "line": line})
        try:
            pred = predict(item, pts, box, text, item["image"].shape[1] / d["w"])
        except Exception as err:  # noqa: BLE001 - the widget shows one plain line, never a traceback
            line = f"SAM could not run: {type(err).__name__}: {err}"
            line = str(err) if isinstance(err, RuntimeError) else line
            return JSON({"png": "", "line": line})
        layers = [(m, hex_rgb(c) * 255) for m, c in error_classes(item, pred)] if truth else [(pred, colour)]
        who = "SAM 3" if text.strip() else "SAM 2.1"
        if pred.any():
            line = f"{who} · {score(item, pred, truth)}"
        else:
            line = f"{who} found nothing" + (" · found 0 % of the truth" if truth else ".")
        return JSON({"png": to_png(layers, d) if (pred.any() or truth) else "", "line": line})

    from google.colab import output

    output.register_callback(f"fm_workshop.sam_play_{uid}", answer)
    ask = (
        f'<input id="txt{uid}" value="{default_text}" size="18"> <button id="ask{uid}">Ask SAM 3</button>'
        if text_prompt
        else ""
    )
    data = json.dumps(shown).replace("</", "<\\/")
    display(
        HTML(
            page.substitute(
                title=title,
                uid=uid,
                data=data,
                ask=ask,
                fg=THEME["works"],
                c_found=THEME["works"],
                c_invented=THEME["fails"],
                c_missed=THEME["truth"],
                c_unscored=UNSCORED,
            )
        )
    )

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#FFF8E1;border-left:6px solid #5D4037;border-radius:6px;padding:12px 14px;margin:6px 0;color:#5D4037;font-family:inherit;line-height:1.5"><b>🧾 If something goes wrong</b><ul>
<li>"Google Drive is not mounted" or "... is missing" → run the setup cell again and click Allow; if the
  folder is still missing, run <code>setup_check.ipynb</code> first.</li>
<li>"No GPU in this session" → Runtime → Change runtime type → T4 GPU, then run the setup cell again.</li>
<li>A cell is slow on a CPU → normal; each training cell has its own <code>TRAIN_…</code> box: untick it to load our
  checkpoint.</li>
<li>A cell with a <code>...</code> stub → fill it or skip it; <code>use()</code> picks yours or ours and the notebook goes on.</li>
</ul></div>"""))

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F5F5F5;border-left:6px solid #616161;border-radius:6px;padding:12px 14px;margin:6px 0;color:#616161;font-family:inherit;line-height:1.5"><details><summary><b>🔍 Recipe and sources</b></summary><ul>
<li>Feature extractor: <strong>DINOv3</strong> ViT-S/16 (Meta, 2025, via <code>timm</code>).</li>
<li>Promptable segmenter: <strong>SAM 2.1</strong> hiera-small (Meta).</li>
<li>Scribble idea: train a tiny classifier on frozen DINO patch features, using only the pixels a person
  marked.</li>
</ul></details></div>"""))

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#E0F2F1;border-left:6px solid #004D40;border-radius:6px;padding:12px 14px;margin:6px 0;color:#004D40;font-family:inherit;line-height:1.5"><b>✅ Takeaway</b><ul>
<li>When a cell needs data → <code>pets_task</code> and <code>specialised</code> are loaded from the cache, with ground truth for
  evaluation only.</li>
<li>When a cell needs a model → <code>dino</code> and <code>sam</code> are already loaded, nothing is downloaded.</li>
<li>When you skip or fail a <code>...</code> stub → <code>use()</code> continues with our version, nothing breaks.</li>
<li>When you read a figure → <span style="color:#6A1B9A">■</span> violet works,
  <span style="color:#E65100">■</span> orange falls short, <span style="color:#2E7D32">■</span> green is
  ground
  truth, <span style="color:#00838F">■</span> teal is a scribble.</li>
</ul></div>"""))

## 1. Out of the box: pets

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#E8F5E9;border-left:6px solid #1B5E20;border-radius:6px;padding:12px 14px;margin:6px 0;color:#1B5E20;font-family:inherit;line-height:1.5"><b>🎯 Objective</b><p>See a foundation model segment pets almost perfectly, zero-shot (with no training on this task).</p></div>"""))

Here SAM, DINOv3 and a linear head (explained below the first figure) meet the easy task; keep your
numbers, because section 2 runs the same three cells on the specialised task.

This cell defines the prompt, figure and scoring helpers used all through the notebook.

In [ ]:
# @title Prompts and figure helpers { display-mode: "form" }
from safetensors import safe_open
from safetensors.torch import load_file

_FEATURE_FILES: dict = {}


def click_point(item: dict) -> list[float]:
    """One (x, y) where a person would click: a pixel of the first foreground scribble stroke."""
    ys, xs = np.nonzero(item["scribbles"] == 1)
    rng = np.random.default_rng([SEED, len(ys)])
    k = int(rng.integers(len(ys)))
    return [float(xs[k]), float(ys[k])]


def box_prompt(item: dict) -> list[float]:
    """The box a person would draw around the object: the full mask's bounding box, as x0, y0, x1, y1."""
    ys, xs = np.nonzero(item["mask"])
    return [float(xs.min()), float(ys.min()), float(xs.max()), float(ys.max())]


def _key_handle(entry):
    """One legend handle for a colour-key entry: "found", or (key, label), or (text, colour)."""
    from matplotlib.lines import Line2D
    from matplotlib.patches import Patch

    known = {
        "found": ("found (prediction and truth)", THEME["works"]),
        "invented": ("invented (prediction only)", THEME["fails"]),
        "missed": ("missed (truth only)", THEME["truth"]),
        "unscored": ("not scored (grey band)", "#E0E0E0"),
        "mask": ("the model's mask", THEME["works"]),
        "truth": ("ground truth", THEME["truth"]),
    }
    if isinstance(entry, str):
        entry = (entry, None)
    key, label = entry
    teal = THEME["scribble"]
    if key == "click":
        return Line2D([], [], marker="*", ls="", ms=11, color=teal, mec="white", label=label or "click")
    if key == "box":
        return Patch(fc="none", ec=teal, lw=2, label=label or "box")
    if key in known:
        default, colour = known[key]
        return Patch(fc=colour, ec="#888888", lw=0.5, label=label or default)
    return Patch(fc=label, ec="#888888", lw=0.5, label=key)  # (text, colour)


ROLES = {  # the tag in a panel's corner: what the panel is (the colours follow the theme)
    "input": ("INPUT", THEME["scribble"]),
    "prediction": ("PREDICTION", THEME["works"]),
    "errors": ("PREDICTION vs TRUTH", THEME["works"]),
    "truth": ("TRUTH (scoring only)", THEME["truth"]),
    "map": ("SCORE MAP", THEME["score"]),
}


def tag_panel(ax, role: str | None) -> None:
    """Write the panel's role (input, prediction, errors, truth, map) in a coloured tag in its top-left
    corner and draw a frame in the same colour, so an input is never mistaken for a prediction."""
    if role is None:
        return
    text, colour = ROLES[role]
    ax.text(0.02, 0.98, text, transform=ax.transAxes, ha="left", va="top", fontsize=7, color="white",
            fontweight="bold", bbox={"boxstyle": "round,pad=0.25", "fc": colour, "ec": "none"})
    ax.axis("on")
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_edgecolor(colour)
        spine.set_linewidth(2.5)


def browse_figures(figures: list, labels: list[str], title: str) -> None:
    """Show one figure at a time with ◀ ▶ buttons and a slider: `figures` are matplotlib figures (closed
    here), `labels` one line each. Pure HTML + JavaScript, so it needs no running Python and works in Colab
    and Jupyter; without JavaScript only the first figure shows."""
    import base64
    import io
    import uuid

    from IPython.display import HTML, display

    pngs = []
    for fig in figures:
        buf = io.BytesIO()
        fig.savefig(buf, format="png", dpi=80)
        plt.close(fig)
        pngs.append("data:image/png;base64," + base64.b64encode(buf.getvalue()).decode())
    uid, n = uuid.uuid4().hex[:8], len(pngs)
    data = json.dumps({"png": pngs, "label": labels})
    display(HTML(f"""
<div style="font-family:sans-serif"><b>{title}</b><br>
<button id="prev{uid}">◀</button> <input id="pick{uid}" type="range" min="0" max="{n - 1}" value="0"
 style="width:240px;vertical-align:middle"> <button id="next{uid}">▶</button>
<span id="lab{uid}" style="margin-left:8px"></span><br>
<img id="img{uid}" src="{pngs[0]}" style="max-width:100%"></div>
<script>(function() {{
const d = {data}, pick = document.getElementById("pick{uid}");
function show(k) {{ k = (k + {n}) % {n}; pick.value = k;
  document.getElementById("img{uid}").src = d.png[k];
  document.getElementById("lab{uid}").textContent = (k + 1) + " / {n} · " + d.label[k]; }}
document.getElementById("prev{uid}").onclick = () => show(+pick.value - 1);
document.getElementById("next{uid}").onclick = () => show(+pick.value + 1);
pick.oninput = () => show(+pick.value);
show(0); }})();</script>"""))


def add_key(fig, entries: list | None, note: str | None = None) -> None:
    """Lay out `fig`, then add a colour key and an optional one-line note under it (it grows a little).

    Entries: "found" | "invented" | "missed" | "unscored" | "mask" | "truth" | "click" | "box", or a pair
    (key, label) to reword one, or (text, colour) for any other colour. Call it last, instead of
    tight_layout()."""
    fig.tight_layout()
    handles = [_key_handle(e) for e in entries or []]
    width, height = fig.get_size_inches()
    cols = max(1, min(len(handles), int(width // 2.3)))
    rows = -(-len(handles) // cols) if handles else 0
    note_h, key_h = (0.3 if note else 0.0), 0.28 * rows
    extra = note_h + key_h + (0.1 if handles or note else 0.0)
    if not extra:
        return
    new_height = height + extra
    for ax in fig.axes:  # keep every axes the same size in inches, and make room underneath
        pos = ax.get_position(original=True)
        y0, h = (pos.y0 * height + extra) / new_height, pos.height * height / new_height
        ax.set_position([pos.x0, y0, pos.width, h])
    fig.set_size_inches(width, new_height)
    if note:
        fig.text(0.5, 0.1 / new_height, note, ha="center", va="bottom", fontsize=8, style="italic")
    if handles:
        fig.legend(handles=handles, loc="lower center", ncol=cols, fontsize=8, frameon=False,
                   bbox_to_anchor=(0.5, (0.05 + note_h) / new_height))


def show_masks(
    panels: list[tuple[str, np.ndarray]],
    prompts: dict | list | None = None,
    width: float = 3.2,
    legend: list | None = None,
    note: str | None = None,
    ncols: int | None = None,
    roles: list | None = None,
    show: bool = True,
):
    """One row of RGB panels (title, image), or several rows of `ncols` panels each.

    `prompts` = {"point": [x, y], "box": [x0, y0, x1, y1]} drawn on every panel, or a list with one such dict
    (or None) per panel. `legend` = colour-key entries for add_key(); `note` = one italic line under it.
    `roles` = one tag per panel ("input", "prediction", "errors", "truth", "map" or None), see tag_panel().
    `show=False` returns the figure instead of drawing it (for browse_figures)."""
    ncols = ncols or len(panels)
    nrows = -(-len(panels) // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(width * ncols, width * nrows))
    axes = np.atleast_1d(axes).ravel()
    for ax in axes[len(panels):]:
        ax.axis("off")
    for ax, (title, img), role in zip(axes, panels, roles or [None] * len(panels), strict=False):
        ax.imshow(img)
        ax.set_title(title, fontsize=8 if ncols > 3 else 9)
        ax.axis("off")
        tag_panel(ax, role)
    per_panel = prompts if isinstance(prompts, list) else [prompts] * len(panels)
    for ax, p in zip(axes, per_panel, strict=False):
        if not p:
            continue
        if p.get("point") is not None:
            x, y = p["point"]
            ax.plot(x, y, marker="*", markersize=14, color=THEME["scribble"], markeredgecolor="white")
        if p.get("box") is not None:
            x0, y0, x1, y1 = p["box"]
            rect = mpl.patches.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, lw=2)
            rect.set_edgecolor(THEME["scribble"])
            ax.add_patch(rect)
    add_key(fig, legend, note)
    if not show:
        return fig
    plt.show()


def pet_iou(pred: np.ndarray, item: dict) -> float:
    """IoU of a boolean mask with the pet, counted only inside the scored area (the grey band is left out)."""
    r = region(item)
    p, t = pred & r, item["mask"] & r
    union = (p | t).sum()
    return float((p & t).sum() / union) if union else 1.0  # same formula as iou() in 1.2, usable before it


def truth_panel(item: dict) -> np.ndarray:
    """The photo with the ground truth in green; the not-scored band is drawn grey."""
    return overlay(item["image"], item["mask"], THEME["truth"], unscored=~region(item))


def error_panel(item: dict, pred: np.ndarray) -> np.ndarray:
    """Violet = pet found, orange = false alarm, green = missed, grey = not scored."""
    r = region(item)
    pred, truth = pred & r, item["mask"] & r
    img = overlay(item["image"], pred & truth, THEME["works"], unscored=~r)
    img = overlay(img, pred & ~truth, THEME["fails"])
    return overlay(img, truth & ~pred, THEME["truth"])


def features_for(dataset: dict, item: dict, long_side: int) -> torch.Tensor:
    """DINOv3 patch features [384, h, w]: from the precomputed file when it exists, else computed now."""
    prefix = "pets" if dataset is pets_task else "specialised"
    path = PRECOMPUTED / "features" / f"{prefix}_{long_side}.safetensors"
    if path.is_file():
        if path not in _FEATURE_FILES:
            _FEATURE_FILES[path] = safe_open(path, framework="pt")
        store = _FEATURE_FILES[path]
        if item["name"] in store.keys():
            return store.get_tensor(item["name"]).to(DEVICE).float()
    return dino.features(item["image"], long_side)


def pca_rgb(feat: torch.Tensor, shape: tuple[int, int]) -> np.ndarray:
    """Project the 384-dim patch features onto their 3 main directions and show them as red, green, blue."""
    d, h, w = feat.shape
    x = feat.reshape(d, h * w).T.cpu().numpy().astype(np.float64)
    x = x - x.mean(0)
    _, _, vt = np.linalg.svd(x, full_matrices=False)
    rgb = x @ vt[:3].T
    rgb = (rgb - rgb.min(0)) / (rgb.max(0) - rgb.min(0) + 1e-9)
    img = torch.from_numpy(rgb.reshape(h, w, 3)).permute(2, 0, 1)[None].float()
    img = F.interpolate(img, size=shape, mode="bilinear", align_corners=False)[0].permute(1, 2, 0)
    return (img.numpy() * 255).astype(np.uint8)

This cell loads the held-out pets that every approach in this notebook is scored on.

In [ ]:
# @title Load the held-out pets { display-mode: "form" }
pets_task["feat_side"] = 256 if SMOKE else pets_task["index"]["feature_long_side"]
pets_task["eval"] = pets_task["held"] if (SMOKE or DEVICE == "cuda") else pets_task["held"][:12]
pets_eval = [load_item(pets_task, it) for it in pets_task["eval"]]
print(
    f"evaluation set: {len(pets_eval)} held-out pets · "
    f"DINOv3 features at {pets_task['feat_side']} px long side"
)

### 1.1 SAM: one click, one box

**What the $\color{#E0E0E0}{\blacksquare}$ grey band is.** The Oxford pet photos come with a ground truth
drawn by annotators. Around every
outline they added a "not sure" strip, about 10 to 20 pixels wide. A pixel in that strip is neither pet nor
floor: the annotator said "unknown". We do not score it, so ground-truth pictures show it in
$\color{#9E9E9E}{\blacksquare}$ grey and a mask
is judged only on the rest. The difference between "unknown" and "background" is the idea behind the whole
workshop.

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F3E5F5;border-left:6px solid #4A148C;border-radius:6px;padding:12px 14px;margin:6px 0;color:#4A148C;font-family:inherit;line-height:1.5"><b>🤔 Your prediction</b><p>The next cell lets you drive SAM yourself on the held-out pets. Before you start: will one click be enough
for a clean mask of the whole animal, and what could go wrong (fur against floor, the tail, a second
animal)? Say it to your neighbour or write it down.</p></div>"""))

The cell below shows a held-out pet and takes a click, a box or a word, then draws SAM's mask with its IoU
against the ground truth (IoU = overlap ÷ union of the two masks: 1 is a perfect match, 0 no overlap; 1.2
explains it). The ◀ ▶ buttons change pet. 👁 Show ground truth colours the result:
$\color{#6A1B9A}{\blacksquare}$ violet found, $\color{#E65100}{\blacksquare}$ orange
invented, $\color{#2E7D32}{\blacksquare}$ green missed, $\color{#E0E0E0}{\blacksquare}$ light grey not
scored. Without Colab it shows one fixed click and one box instead.

**Q2.** 🟢 Drive SAM on three or four pets.

- Click on a pet, shift-click on the floor, then draw a box. What does each give you?
- Add a second click on a part SAM missed. What changes?
- Press 👁 Show ground truth. Does the mask differ from the truth at the outline, or is a whole part
  missing or added?
- Where does one click fail, and does a box fix it?
- Optional: type `dog` or `cat` and press "Ask SAM 3" (only if SAM 3 is in your cache).

Look at the floor click: SAM still returns a clean-looking mask. A clean mask is not a correct one, so
always compare it with the ground truth (section 2 shows why).

In [ ]:
# @title Q2 — Click, box or a word: drive SAM yourself { display-mode: "form" }
sam_playground(pets_eval, "Pets: click, box, or a word", text_prompt=True, default_text="dog")

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#EDE7F6;border-left:6px solid #311B92;border-radius:6px;padding:12px 14px;margin:6px 0;color:#311B92;font-family:inherit;line-height:1.5"><b>📘 Theory: the three tools</b><ul>
<li><strong>SAM</strong> (Segment Anything) is <em>promptable</em>. You give it an image and a hint (a click, a box on a pet);
  it returns the mask of the object under the hint. Nothing is trained: it learnt "what an object is" from
  a billion masks of everyday photos.</li>
<li><strong>DINOv3</strong> is a <em>feature extractor</em>. It cuts the image into 16 × 16 px patches and returns 384 numbers
  per patch, learnt without labels. Patches with similar numbers look alike to the model (two cat ears,
  say).</li>
<li>A <strong>linear head</strong> is the tiny classifier that turns those features into a mask: the features segment
  nothing on their own, and the head learns a new class from a handful of masks in seconds (section 1.4).</li>
</ul></div>"""))

### 1.2 One number: IoU

Looking at masks is the honest way to judge a segmentation; one number lets us compare approaches on a
hundred images. The whole workshop uses one metric, **IoU** (intersection over union):

`IoU = |pred ∩ truth| / |pred ∪ truth|`

- Example: the predicted mask and the true mask each cover 100 pixels and share 80. The union is 120
  pixels, so IoU = 80 / 120 = 0.67.
- 1.0 is a perfect match, 0.0 means no overlap. Section 2 adds one extra metric for thin structures.
- On the pets, both masks are first cut to the scored area, so the $\color{#E0E0E0}{\blacksquare}$ grey
  "not sure" band counts for nothing.

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#FFE8D6;border-left:6px solid #7A2E00;border-radius:6px;padding:12px 14px;margin:6px 0;color:#7A2E00;font-family:inherit;line-height:1.5"><b>✏️ Your turn (optional, 1 min)</b><p>IoU = intersection ÷ union. We wrote the intersection. Fill <code>my_union</code>: how many pixels are True in at
least one of two boolean masks? The cell below checks your line on this 4 × 4 play example (1 = True) and
continues with ours if you skip it:</p>
<pre><code>pred   truth
0000   0110
0111   0110
0111   0110
0000   0000
</code></pre>
<p>The intersection is 4 and the union is 8, so the answer is 8 and the IoU is 0.5.</p></div>"""))

In [ ]:
# @title Reference solution (open only if stuck) { display-mode: "form" }
def _ref_union(pred: np.ndarray, truth: np.ndarray) -> int:
    """Reference: the number of pixels where pred or truth is True."""
    return int((pred | truth).sum())


_play_pred = np.zeros((4, 4), bool)
_play_pred[1:3, 1:4] = True  # 6 pixels
_play_truth = np.zeros((4, 4), bool)
_play_truth[0:3, 1:3] = True  # 6 pixels, 4 shared -> union 8, IoU = 4 / 8

In [ ]:
# ✍️ YOUR CODE
def my_union(pred: np.ndarray, truth: np.ndarray) -> int:
    """Number of pixels that are True in pred, in truth, or in both.
    e.g. 8"""
    ...


union_pixels = use(
    my_union, reference=_ref_union, check=lambda f: int(f(_play_pred, _play_truth)), name="iou"
)


def iou(pred: np.ndarray, truth: np.ndarray) -> float:
    """Intersection over union of two boolean masks of the same shape; 1.0 when both are empty.
    e.g. 0.5"""
    inter, union = float((pred & truth).sum()), float(union_pixels(pred, truth))
    return inter / union if union else 1.0

This cell scores SAM on every held-out pet with the IoU above and keeps the mean as the zero-shot number
for section 2. The click is one pixel of a foreground scribble and the box is the tight box around the pet,
both taken from the ground truth: what a person would give SAM.

In [ ]:
# @title Score SAM on every held-out pet { display-mode: "form" }
t0 = time.monotonic()
zero_shot_all = {"point": [], "box": []}
for it in tqdm(pets_eval, desc="SAM on the held-out pets"):
    pt = click_point(it)
    by_point = sam.probability(it["image"], points=[pt], labels=[1]) >= 0.5
    by_box = sam.probability(it["image"], box=box_prompt(it)) >= 0.5
    zero_shot_all["point"].append(pet_iou(by_point, it))
    zero_shot_all["box"].append(pet_iou(by_box, it))
zero_shot = {k: float(np.mean(v)) for k, v in zero_shot_all.items()}
RESULTS["pets_zero_shot_iou"] = zero_shot["box"]  # the headline: SAM's best single prompt, one box
RESULTS["pets_zero_shot_point_iou"] = zero_shot["point"]
RESULTS["pets_zero_shot_box_iou"] = zero_shot["box"]
RESULTS["pets_zero_shot_seconds"] = round(time.monotonic() - t0, 1)
print(
    f"SAM zero-shot on {len(pets_eval)} held-out pets · mean IoU one click {zero_shot['point']:.3f} · "
    f"one box {zero_shot['box']:.3f} · {RESULTS['pets_zero_shot_seconds']} s"
)

### 1.3 What DINOv3 sees

This cell keeps the **3 main directions (a PCA)** of each patch's 384 numbers and shows them as
$\color{#E53935}{\blacksquare}$ red, $\color{#43A047}{\blacksquare}$ green
and $\color{#1E88E5}{\blacksquare}$ blue, next to each photo. The same colour means the model sees those
patches as similar (both cat
ears, say). What to look for: does the animal get colours that differ from the floor, and do its parts
(face, body) differ from each other? The blocks are the 16 px patches.

In [ ]:
# @title Show the patch features as colours { display-mode: "form" }
fig_items = [load_item(pets_task, it) for it in pets_task["fit"][:3]]
panels = []
for it in fig_items:
    feat = features_for(pets_task, it, pets_task["feat_side"])
    title = f"DINOv3 PCA · {feat.shape[2]}×{feat.shape[1]} patches"
    panels += [(it["name"], it["image"]), (title, pca_rgb(feat, it["image"].shape[:2]))]
show_masks(
    panels,
    width=2.6,
    note="Colours = the 3 main directions of the 384 numbers per patch (red, green, blue). They are not "
    "classes: similar colour = patches the model finds similar.",
)

**Q3.** 🟡 Look at the PCA images above.

- Which pixels share a colour: the whole animal, parts of it (head, fur, legs), or the floor too?
- Does the colour follow the object's outline?
- How fine is the grid behind it?

<details><summary>What to look for</summary>

- The colours are not classes: they are the 3 directions along which the features vary most in this image.
- Typically the animal separates from the floor in one direction and its parts (face, body) in another.
- The blocky edges are the 16 px grid: the features know <i>what</i> is where, not <i>exactly where</i> the
  boundary is. Whether that is enough depends on how big the thing you segment is.

</details>

### 1.4 A linear head on five pets

**How the head works.** Each patch gets one score from its 384 feature numbers `f`:

`score = w · f + b` (384 weights + 1 bias = 385 parameters), the same `w` and `b` for every patch (a
"1 × 1 convolution"). Score above 0: the patch is "pet"; below 0: "background".

A sigmoid `σ` squeezes the score into a probability `p = σ(score)` between 0 and 1 (score 2.2 gives
p = 0.90, score 0 gives 0.5), so "score above 0" and "p above 0.5" say the same thing. The per-patch map is
then upsampled to the image.

Training uses a **balanced loss**: pet pixels and background pixels each get half of the total weight, so
a small pet is not drowned out by the floor.

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F3E5F5;border-left:6px solid #4A148C;border-radius:6px;padding:12px 14px;margin:6px 0;color:#4A148C;font-family:inherit;line-height:1.5"><b>🤔 Your prediction</b><p>The next cells train a classifier with 385 parameters on the patch features of just <strong>five</strong> pets with
full masks, then score it on the held-out ones. Will its IoU land above, near or below SAM's one-click
number from 1.2? Write down a number before you run.</p></div>"""))

This cell defines the linear head that every later training step reuses.

In [ ]:
# @title Linear head: train, predict, load a checkpoint { display-mode: "form" }
def train_head(
    feats: list[torch.Tensor], labels: list[np.ndarray], steps: int, lr: float = 0.05, balanced: bool = True
) -> tuple[dict, float]:
    """Linear head (384 -> 1) on standardised patch features. Labels per pixel: 1 fg, 0 bg, -1 ignored.

    Pixels labelled -1 are skipped. `balanced=True`: each class gets half of the total loss weight.
    Returns the head's tensors (weight, bias, feat_mean, feat_std) and the training seconds.
    """
    torch.manual_seed(SEED)
    allf = torch.cat([f.flatten(1) for f in feats], 1)
    mu, sd = allf.mean(1), allf.std(1) + 1e-6
    feats = [(f - mu.view(-1, 1, 1)) / sd.view(-1, 1, 1) for f in feats]
    labs = [torch.from_numpy(np.asarray(lab, dtype=np.int8)).to(DEVICE) for lab in labels]
    n_pos = sum(int((lab == 1).sum()) for lab in labs)
    n_neg = sum(int((lab == 0).sum()) for lab in labs)
    if balanced:
        w_pos, w_neg = 0.5 / max(n_pos, 1), 0.5 / max(n_neg, 1)
    else:
        w_pos = w_neg = 1.0 / max(n_pos + n_neg, 1)
    head = torch.nn.Conv2d(feats[0].shape[0], 1, 1).to(DEVICE)
    opt = torch.optim.Adam(head.parameters(), lr=lr)
    t0 = time.monotonic()
    for _ in range(steps):
        opt.zero_grad()
        loss = 0.0
        for f, lab in zip(feats, labs, strict=True):
            logit = F.interpolate(head(f[None]), size=lab.shape, mode="bilinear", align_corners=False)[0, 0]
            w = torch.where(lab == 1, w_pos, torch.where(lab == 0, w_neg, 0.0))
            bce = F.binary_cross_entropy_with_logits(logit, (lab == 1).float(), reduction="none")
            loss = loss + (bce * w).sum()
        loss.backward()
        opt.step()
    tensors = {
        "weight": head.weight.detach().cpu(),
        "bias": head.bias.detach().cpu(),
        "feat_mean": mu.cpu(),
        "feat_std": sd.cpu(),
    }
    return tensors, time.monotonic() - t0


@torch.no_grad()
def predict_head(tensors: dict, feat: torch.Tensor, shape: tuple[int, int]) -> np.ndarray:
    """Apply a head to one feature grid; returns the boolean foreground mask at `shape` (logit > 0)."""
    dev = feat.device
    f = (feat - tensors["feat_mean"].to(dev).view(-1, 1, 1)) / tensors["feat_std"].to(dev).view(-1, 1, 1)
    logit = F.conv2d(f[None], tensors["weight"].to(dev), tensors["bias"].to(dev))
    logit = F.interpolate(logit, size=shape, mode="bilinear", align_corners=False)[0, 0]
    return (logit > 0).cpu().numpy()


def load_head(dataset: dict, regime: str, long_side: int | None = None) -> tuple[dict, dict]:
    """Our precomputed checkpoint for (dataset, regime): its tensors and its index entry."""
    long_side = long_side or dataset["feat_side"]
    key = "pets" if dataset is pets_task else specialised["key"]
    wanted = (key, regime, long_side)
    heads = PRECOMPUTED_HEADS
    entry = next((h for h in heads if (h["dataset"], h["regime"], h["long_side"]) == wanted), None)
    if entry is None:
        raise SystemExit(f"no precomputed head for {wanted}. Run setup_check.ipynb first.")
    return load_file(PRECOMPUTED / entry["file"]), entry


def fit_pet_head(train_yourself: bool, n_pets: int, steps: int) -> tuple[list[dict], dict, float, str]:
    """Train the pet head on the full masks of the first `n_pets` fit pets, or load ours.

    The "not sure" band is labelled -1, so the loss skips it. Returns pets, head, seconds, source line.
    """
    if not isinstance(train_yourself, bool):
        raise SystemExit("TRAIN_PET_HEAD must be True or False.")
    items = [load_item(pets_task, it) for it in pets_task["fit"][:n_pets]]
    feats = [features_for(pets_task, it, pets_task["feat_side"]) for it in items]
    if not train_yourself:
        head, entry = load_head(pets_task, "full5")
        src = f"our checkpoint {entry['file']} ({len(entry['train_names'])} pets)"
        return items, head, entry["train_sec"], src
    labels = []
    for it in items:  # full masks: allowed in section 1 only
        lab = it["mask"].astype(np.int8)
        lab[~region(it)] = -1
        labels.append(lab)
    head, sec = train_head(feats, labels, steps)
    return items, head, sec, f"trained here on {len(items)} pets, {steps} steps"

This cell trains the head on the full masks of five pets, leaving out the
$\color{#E0E0E0}{\blacksquare}$ grey band. Untick
`TRAIN_PET_HEAD` to load our checkpoint instead.

In [ ]:
# @title Train the linear head on five pets { display-mode: "form" }
TRAIN_PET_HEAD = True  # @param {type:"boolean"}
N_TRAIN_PETS = 5
train_items, head_full5, head_sec, head_src = fit_pet_head(TRAIN_PET_HEAD, N_TRAIN_PETS, HEAD_STEPS)
print(f"linear head · {head_src} · {head_sec:.1f} s on {DEVICE}")
if head_sec > 60:
    print("ℹ️ Slow session? Untick TRAIN_PET_HEAD above to load our checkpoint next time.")

This cell runs the head you just trained on one training pet and shows each step (patches, one probability
per patch, the upsampled map, the final mask), then works through one patch by hand. The
$\color{#00838F}{\blacksquare}$ teal square marks
that patch; the probability p of "pet" runs from $\color{#424242}{\blacksquare}$ dark grey (0) through
$\square$ white (0.5) to $\color{#1565C0}{\blacksquare}$ blue (1). What to look for: the map in
panel 2 is coarse (one value per patch), and panels 3 and 4 only smooth and threshold it.

In [ ]:
# @title The linear head step by step on one pet { display-mode: "form" }
hx = train_items[0]
hfeat = features_for(pets_task, hx, pets_task["feat_side"])  # [384, h, w]: 384 numbers per patch
dev = hfeat.device
with torch.no_grad():
    hp = {k: v.to(dev) for k, v in head_full5.items()}
    hz = (hfeat - hp["feat_mean"].view(-1, 1, 1)) / hp["feat_std"].view(-1, 1, 1)
    hscore = F.conv2d(hz[None], hp["weight"], hp["bias"])[0, 0]  # w·f + b, one score per patch
    hprob = torch.sigmoid(hscore)
    hup = F.interpolate(hprob[None, None], size=hx["mask"].shape, mode="bilinear", align_corners=False)[0, 0]
gh, gw = hprob.shape
py, px = np.unravel_index(int(hprob.argmax()), hprob.shape)  # the most pet-like patch
f_vec, w_vec = hz[:, py, px], hp["weight"].reshape(-1)
f_head = ", ".join(f"{v:.2f}" for v in f_vec[:4].tolist())
dot, bias = float((w_vec * f_vec).sum()), float(hp["bias"])
total, p_patch = float(hscore[py, px]), float(hprob[py, px])
print(f"patch (row {py}, col {px}) · f = [{f_head}, … 384 numbers]")
print(f"w · f + b = {dot:.2f} + ({bias:.2f}) = {total:.2f} → p = σ({total:.2f}) = {p_patch:.2f}")

H, W = hx["mask"].shape
fig, axes = plt.subplots(1, 4, figsize=(11.6, 3.4))
axes[0].imshow(hx["image"])
axes[0].vlines(np.arange(0, W + 1, W / gw), 0, H, colors="white", linewidths=0.4)
axes[0].hlines(np.arange(0, H + 1, H / gh), 0, W, colors="white", linewidths=0.4)
cell_box = plt.Rectangle((px * W / gw, py * H / gh), W / gw, H / gh, fill=False, ec=THEME["scribble"], lw=2)
axes[0].add_patch(cell_box)
axes[0].set_title("1 · image cut into patches\n(white lines = the grid)", fontsize=9)
axes[1].imshow(hprob.cpu().numpy(), cmap=score_cmap, vmin=0, vmax=1, interpolation="nearest")
axes[1].add_patch(plt.Rectangle((px - 0.5, py - 0.5), 1, 1, fill=False, ec=THEME["scribble"], lw=2))
axes[1].set_title("2 · one p per patch\n(grey 0, white 0.5, blue 1)", fontsize=9)
axes[2].imshow(hup.cpu().numpy(), cmap=score_cmap, vmin=0, vmax=1)
axes[2].set_title("3 · upsampled\nto the image", fontsize=9)
axes[3].imshow(overlay(hx["image"], predict_head(head_full5, hfeat, hx["mask"].shape), THEME["works"]))
axes[3].set_title("4 · p > 0.5 → pet\n(violet = predicted pet)", fontsize=9)
for ax, role in zip(axes, ("input", "map", "map", "prediction"), strict=True):
    ax.axis("off")
    tag_panel(ax, role)
add_key(
    fig,
    [("box", "patch worked through in the printout"), ("16 px patch grid (panel 1)", "#FFFFFF"),
     ("p near 1 = pet (panels 2-3)", THEME["score"]), ("p = 0.5: unsure", "#FFFFFF"),
     ("p near 0 = background", "#424242"), ("mask: p > 0.5 (panel 4)", THEME["works"])],
    note="Panel 4 is the head's mask: violet = pixels with p above 0.5.",
)
plt.show()

This cell runs the head on the held-out pets it never saw and scores it with the same IoU. The figure shows
SAM from the box (row 1) and the head (row 2) on the same three pets: the worst, median and best by SAM's
IoU. The $\color{#00838F}{\blacksquare}$ teal rectangle is the box given to SAM. What to look for: is the
spread across pets as large for
the head as for SAM, and are the head's edges blockier (16 px patches)?

In [ ]:
# @title Score the head, and SAM next to it { display-mode: "form" }
t0 = time.monotonic()
head_preds = [
    predict_head(head_full5, features_for(pets_task, it, pets_task["feat_side"]), it["mask"].shape)
    for it in pets_eval
]
head_ious = [pet_iou(pred, it) for pred, it in zip(head_preds, pets_eval, strict=True)]
RESULTS["pets_head_iou"] = float(np.mean(head_ious))
RESULTS["pets_head"] = {
    "iou": RESULTS["pets_head_iou"],
    "n_train": len(train_items),
    "train_sec": round(float(head_sec), 2),
    "infer_sec": round(time.monotonic() - t0, 1),
}
n_eval = len(pets_eval)
print(f"Linear head: trained in 1.4 on {len(train_items)} pets with full masks, run on {n_eval} unseen pets.")
print(f"Score: per-pet IoU with the ground truth (grey band not scored), averaged over the {n_eval} pets.")
print(f"  linear head       mean IoU {RESULTS['pets_head_iou']:.3f}")
print(f"  SAM, one click    mean IoU {RESULTS['pets_zero_shot_point_iou']:.3f}  (the 1.2 masks)")
print(f"  SAM, one box      mean IoU {RESULTS['pets_zero_shot_box_iou']:.3f}  (the 1.2 masks)")
pet_order = np.argsort(zero_shot_all["box"])  # the same three pets in both rows
picked = [("worst", pet_order[0]), ("median", pet_order[n_eval // 2]), ("best", pet_order[-1])]
sam_row, head_row, boxes = [], [], []
for label, pos in picked:
    it = pets_eval[int(pos)]
    by_box = sam.probability(it["image"], box=box_prompt(it)) >= 0.5
    sam_iou, head_iou = zero_shot_all["box"][int(pos)], head_ious[int(pos)]
    sam_row.append((f"{label} pet · SAM from the box\nIoU {sam_iou:.2f}", error_panel(it, by_box)))
    head_row.append((f"same pet · linear head\nIoU {head_iou:.2f}", error_panel(it, head_preds[int(pos)])))
    boxes.append({"box": box_prompt(it)})
show_masks(
    sam_row + head_row,
    prompts=boxes + [None] * len(head_row),
    ncols=3,
    legend=["found", "invented", "missed", "unscored", ("box", "box given to SAM")],
    note="Pets chosen by SAM's box IoU (worst, median, best); the head is shown on the same three.",
)

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#E8EAF6;border-left:6px solid #1A237E;border-radius:6px;padding:12px 14px;margin:6px 0;color:#1A237E;font-family:inherit;line-height:1.5"><b>🔀 Across approaches</b><p>Two ways to the same mask, scored on the same hidden ground truth:</p>
<ul>
<li><strong>SAM</strong> needs <em>one hint per image</em> and no training.</li>
<li><strong>The head</strong> needs <em>five full masks once</em>, then runs on every image with no hint.</li>
</ul>
<p>Keep both numbers: the comparison table in section 8 starts with them, and section 2 reruns both on the
specialised task.</p></div>"""))

**Q4.** 🟡 Compare the two numbers you got with the prediction you wrote down.

- Which approach would you pick for 10 000 pet photos, and which for 10 photos?
- What does each one ask of a person?

<details><summary>One possible answer</summary>

- 10 photos: a click each is nothing, and SAM's masks have sharp boundaries.
- 10 000 photos: nobody clicks them all, while the head, trained once on a few masks, needs no hint. Its
  edges follow the 16 px grid, so it loses a little IoU, more as objects get smaller.
- Hardware or seed may change the numbers; the trade-off stays.

</details>

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#E0F2F1;border-left:6px solid #004D40;border-radius:6px;padding:12px 14px;margin:6px 0;color:#004D40;font-family:inherit;line-height:1.5"><b>✅ Takeaway</b><ul>
<li>When you have a few photos and can click each one → SAM with a point or a box gives sharp masks, with no
  training.</li>
<li>When you have thousands of photos and no one to click → train the linear head on a few full masks.</li>
<li>When a mask looks clean → still check it against the ground truth before trusting it.</li>
<li>When the head's edges look blocky (16 px grid) → expect lower IoU on small or thin objects; section 2
  tests it.</li>
</ul></div>"""))

## 2. Where it falls short

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#E8F5E9;border-left:6px solid #1B5E20;border-radius:6px;padding:12px 14px;margin:6px 0;color:#1B5E20;font-family:inherit;line-height:1.5"><b>🎯 Objective</b><p>Run the same tools on the specialised task and probe SAM with escalating prompts.</p></div>"""))

Same three tools, same cells, new task. Section 1 gave you three numbers on pets: SAM with one click,
SAM with one box, and a linear head on five full masks.

Here they meet retina images (a *fundus* is a photo of the back of the eye), where the objects are thin,
branching and low-contrast. Then we push SAM with more and more prompts, and give DINOv3 twice the
resolution, to see whether the limit is the prompt or the patch size.

This helper cell defines the scores, clicks and panel titles used in this section.

In [ ]:
# @title Thin-structure helpers: skeleton, score, clicks, figures { display-mode: "form" }
import hashlib

from scipy import ndimage as ndi
from skimage.morphology import skeletonize

_SKEL_CACHE: dict = {}  # prediction skeletons by mask content: {(shape, hash): flat pixel indices}
_SKEL_CACHE_MAX = 300


def skeleton(mask: np.ndarray) -> np.ndarray:
    """The one-pixel-wide centre line of a boolean mask."""
    return skeletonize(np.asarray(mask, bool))


def blob_skeleton(mask: np.ndarray) -> np.ndarray:
    """Centre line of a mask that fills most of the image, found at half resolution. Thinning a near-solid
    1024 px mask takes ~0.8 s and its centre line is a short stub either way; at half size it takes ~0.1 s and
    gave the same clDice on our retina crops."""
    out = np.zeros_like(mask)
    ys, xs = np.nonzero(skeleton(mask[::2, ::2]))
    out[ys * 2, xs * 2] = True
    return out & mask


def cached_skeleton(mask: np.ndarray) -> np.ndarray:
    """skeleton(mask), but a mask seen before is not skeletonised again, and a near-solid one goes faster."""
    mask = np.asarray(mask, bool)
    key = (mask.shape, hashlib.blake2b(np.packbits(mask), digest_size=16).digest())
    if key not in _SKEL_CACHE:
        if len(_SKEL_CACHE) >= _SKEL_CACHE_MAX:
            _SKEL_CACHE.pop(next(iter(_SKEL_CACHE)))  # drop the oldest
        sk = blob_skeleton(mask) if mask.mean() > 0.6 else skeleton(mask)
        _SKEL_CACHE[key] = np.flatnonzero(sk)
    out = np.zeros(mask.shape, bool)
    out.flat[_SKEL_CACHE[key]] = True
    return out


def truth_skeleton(item: dict) -> np.ndarray:
    """The true centre line inside the evaluation region, computed once per item and kept on it."""
    if "_skel" not in item:
        item["_skel"] = skeleton(item["mask"] & region(item))
    return item["_skel"]


def score(pred: np.ndarray, item: dict) -> dict:
    """IoU and clDice of a boolean prediction, both inside the evaluation region."""
    r = region(item)
    p, t = pred & r, item["mask"] & r
    return {"iou": iou(p, t), "cldice": cldice(p, t, cached_skeleton(p), truth_skeleton(item))}


def everything_score(item: dict) -> dict:
    """Score of "everything in the evaluation region is foreground", computed once per item."""
    if "_everything" not in item:
        item["_everything"] = score(region(item).copy(), item)
    return item["_everything"]


def mean_scores(scores: list[dict]) -> dict:
    return {k: float(np.mean([s[k] for s in scores])) for k in scores[0]}


def marked_pct(pred: np.ndarray, item: dict) -> str:
    """How much of the evaluation region a prediction calls vessel, next to the truth, e.g.
    "calls 98 % vessel (really 11 %)"."""
    r = region(item)
    return f"calls {100 * pred[r].mean():.0f} % vessel (really {100 * item['mask'][r].mean():.0f} %)"


def recall_of(pred: np.ndarray, item: dict) -> float:
    """Recall: the share of the true vessel pixels (inside the evaluation region) that the mask covers."""
    truth = item["mask"] & region(item)
    return float((pred & truth).sum() / truth.sum()) if truth.any() else 1.0


MARKED_NOTE = (
    "calls … vessel = share of the fundus (the photo inside the black ring) the mask calls vessel, "
    "next to the share that really is · recall = share of the true vessel pixels it finds"
)


def pred_title(label: str, pred: np.ndarray, item: dict) -> str:
    """Panel title on three short lines: what made the mask, IoU and recall, then the share called vessel."""
    r = region(item)
    return (
        f"{label}\nIoU {iou(pred & r, item['mask'] & r):.2f} · recall {recall_of(pred, item):.2f}\n"
        f"{marked_pct(pred, item)}"
    )


def scribble_points(item: dict, n: int) -> list[list[float]]:
    """n (x, y) clicks along the foreground scribbles: the section-1 click first, then spread out as far as
    possible from each other (farthest-point sampling), the way a person would click along the vessels."""
    ys, xs = np.nonzero(item["scribbles"] == 1)
    pts = np.stack([xs, ys], 1).astype(float)
    chosen = [np.array(click_point(item))]
    dist = np.linalg.norm(pts - chosen[0], axis=1)
    while len(chosen) < min(n, len(pts)):
        k = int(dist.argmax())
        chosen.append(pts[k])
        dist = np.minimum(dist, np.linalg.norm(pts - pts[k], axis=1))
    return [[float(x), float(y)] for x, y in chosen]


def show_clicks(
    panels: list[tuple[str, np.ndarray, list | None]],
    width: float = 2.6,
    legend: list | None = None,
    note: str | None = None,
):
    """One row of panels (title, RGB image, clicks or None[, box]); stars = clicks, frame = box.
    `legend` and `note` are the colour key of add_key() under the figure."""
    fig, axes = plt.subplots(1, len(panels), figsize=(width * len(panels), width))
    for ax, (title, img, pts, *extra) in zip(np.atleast_1d(axes), panels, strict=True):
        ax.imshow(img)
        ax.set_title(title, fontsize=8)
        ax.axis("off")
        if extra and extra[0] is not None:  # an optional box (x0, y0, x1, y1)
            bx0, by0, bx1, by1 = extra[0]
            ax.add_patch(plt.Rectangle((bx0, by0), bx1 - bx0, by1 - by0, fill=False, lw=2,
                                       ec=THEME["scribble"]))
        for x, y in pts or []:
            ax.plot(x, y, marker="*", markersize=10, color=THEME["scribble"], markeredgecolor="white")
    add_key(fig, legend, note)
    plt.show()


def model_key(name: str) -> tuple:
    """Legend entry for a model's plain mask, drawn in violet like everywhere else in this notebook."""
    return (name, THEME["works"])

This cell loads the held-out retina images (1024 px crops of a 3504 × 2336 fundus), the evaluation set for
the rest of the notebook. Keep the printed share of the fundus the vessels cover; it comes back as a
baseline.

In [ ]:
# @title Load the held-out retina images { display-mode: "form" }
specialised["feat_side"] = 256 if SMOKE else specialised["feature_long_side"]
specialised["eval"] = specialised["held"] if (SMOKE or DEVICE == "cuda") else specialised["held"][:12]
spec_eval = [load_item(specialised, it) for it in specialised["eval"]]
demo = spec_eval[0]  # the image the figures of sections 2 to 8 show
fg_share = float(np.mean([it["mask"][region(it)].mean() for it in spec_eval]))
print(
    f"evaluation set: {len(spec_eval)} held-out images · {specialised['task']} · the foreground covers "
    f"{fg_share:.0%} of the evaluation region · DINOv3 features at {specialised['feat_side']} px long side"
)

This cell shows one retina photo, its vessels ($\color{#2E7D32}{\blacksquare}$ green) and a zoom with the
16 px grid DINOv3 sees ($\square$ white
squares), and measures a vessel's width against one patch. What to look for: in the zoom, a vessel is
thinner than one $\square$ white square, so no square can be "all vessel".

In [ ]:
# @title Meet the retina images: vessels next to the DINOv3 patch grid { display-mode: "form" }
ph, pw = demo["mask"].shape
patch_px = 16 * max(ph, pw) / specialised["feat_side"]  # one DINOv3 patch, in image pixels
centre_line = skeleton(demo["mask"] & region(demo))
vessel_px = float(demo["mask"].sum() / max(centre_line.sum(), 1))  # area ÷ centre-line length = mean width
win = int(min(8 * patch_px, ph, pw))  # the zoom window: about 8 × 8 patches
y0, x0 = (ph - win) // 2, (pw - win) // 2
zoom = (slice(y0, y0 + win), slice(x0, x0 + win))
truth_img = overlay(demo["image"], demo["mask"], THEME["truth"])

fig, axes = plt.subplots(1, 3, figsize=(9.6, 3.4))
axes[0].imshow(demo["image"])
axes[0].set_title("fundus photo: bright round patch = optic disc,\nblack ring = outside the view", fontsize=8)
axes[1].imshow(truth_img)
axes[1].set_title("vessels (ground truth, green)", fontsize=9)
axes[2].imshow(truth_img[zoom])
grid = np.arange(0, win + 1, patch_px)
axes[2].vlines(grid, 0, win, colors="white", linewidths=0.8)
axes[2].hlines(grid, 0, win, colors="white", linewidths=0.8)
axes[2].set_xlim(0, win)
axes[2].set_ylim(win, 0)
axes[2].set_title(f"zoom: each white square is one {patch_px:.0f} × {patch_px:.0f} px patch", fontsize=9)
for ax in axes:
    ax.axis("off")
add_key(fig, [("vessels (ground truth)", THEME["truth"]), ("16 px patch grid", "#FFFFFF")])
plt.show()
print(
    f"{demo['name']}: vessels are about {vessel_px:.1f} px wide on average, one patch is {patch_px:.0f} px, "
    f"so a typical vessel fills about {vessel_px / patch_px:.0%} of a patch's width"
)

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#EDE7F6;border-left:6px solid #311B92;border-radius:6px;padding:12px 14px;margin:6px 0;color:#311B92;font-family:inherit;line-height:1.5"><b>📘 Theory: why thin structures are hard</b><p>Each tool has its own trouble with the vessels you just saw.</p>
<ul>
<li><strong>DINOv3 looks at 16 × 16 px patches.</strong> A vessel a few pixels wide fills a fraction of a patch (see the
  zoom), so the patch is mostly "background with a dark line in it". The head can only say "vessel" or
  "not" for the whole square.</li>
<li><strong>SAM learnt from a billion masks of <em>objects</em>:</strong> compact things with a clear inside and outside, like a
  pet. A vessel tree has no inside, and one click is ambiguous: this vessel? the whole tree? the eye?</li>
</ul></div>"""))

### 2.1 SAM on vessels: click, box, or a word

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F3E5F5;border-left:6px solid #4A148C;border-radius:6px;padding:12px 14px;margin:6px 0;color:#4A148C;font-family:inherit;line-height:1.5"><b>🤔 Your prediction</b><p>Before you open the playground, guess. What will SAM mark from one click on a thin vessel? And from a box
that covers all the vessels, which on a fundus is nearly the whole photo?</p></div>"""))

**Try it yourself.** Section 1's SAM could mark a pet. Now see whether you can make it mark only the
vessels. What to look for: the $\color{#6A1B9A}{\blacksquare}$ violet mask is what SAM returned. Press
"Show ground truth" to colour what
it found ($\color{#6A1B9A}{\blacksquare}$ violet), invented ($\color{#E65100}{\blacksquare}$ orange) and
missed ($\color{#2E7D32}{\blacksquare}$ green); the key under the picture names the colours.

**Q5.** 🟡 Can you get SAM to mark only the vessels? Try these four and note after each what SAM marked: a
vessel, the whole tree, a blob, or the whole fundus.

1. One click on a vessel.
2. Several clicks along one vessel.
3. A small box around a few vessels.
4. A box around the whole fundus.

Then compare with the pets of section 1. If SAM 3 is in your cache, type `retinal vessels` in the text box
and compare its mask with your clicks.

In [ ]:
# @title Q5 — Try it: make SAM mark only the vessels { display-mode: "form" }
sam_playground(spec_eval, "Retina: click, box, or a word", text_prompt=True, default_text="retinal vessels")

### 2.2 A second number for thin structures: clDice

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#EDE7F6;border-left:6px solid #311B92;border-radius:6px;padding:12px 14px;margin:6px 0;color:#311B92;font-family:inherit;line-height:1.5"><b>📘 Theory: clDice</b><p>IoU counts pixels, and thick vessels hold most of them. So a prediction that draws every vessel twice as
wide loses a lot of IoU although nothing is missed, and one that misses all the capillaries loses little
IoU although a large part of the tree's length is gone.</p>
<p><strong>clDice</strong> compares <em>centre lines</em> (each vessel thinned to a one-pixel line) instead:</p>
<ol>
<li><code>tprec</code> = the part of the prediction's centre line that lies inside the true vessels. Invented
   branches lower it.</li>
<li><code>tsens</code> = the part of the true centre line that lies inside the prediction. Missed branches lower it.</li>
<li>clDice = 2 · tprec · tsens ÷ (tprec + tsens). Example: tprec 1.0 and tsens 0.25 give 0.40.</li>
</ol>
<p>The workshop is not about vessels, so this is the only place clDice gets this much room. The figure below
shows all of it on a made-up vessel tree.</p></div>"""))

This collapsed cell is clDice as code (the three steps above); you do not need to read it.

In [ ]:
# @title Reference: clDice (tprec, tsens, harmonic mean) { display-mode: "form" }
def cldice_parts(pred: np.ndarray, truth: np.ndarray, sk_pred=None, sk_truth=None) -> tuple:
    """tprec, tsens and the two centre lines of two boolean masks (skeletons are computed when not given)."""
    sk_pred = skeleton(pred) if sk_pred is None else sk_pred
    sk_truth = skeleton(truth) if sk_truth is None else sk_truth
    if not sk_pred.any() or not sk_truth.any():
        return 0.0, 0.0, sk_pred, sk_truth
    tprec = float((sk_pred & truth).sum() / sk_pred.sum())
    tsens = float((sk_truth & pred).sum() / sk_truth.sum())
    return tprec, tsens, sk_pred, sk_truth


def cldice(pred: np.ndarray, truth: np.ndarray, sk_pred=None, sk_truth=None) -> float:
    """clDice: harmonic mean of topology precision and topology sensitivity (0.0 when a skeleton is empty)."""
    tprec, tsens, _, _ = cldice_parts(pred, truth, sk_pred, sk_truth)
    return 2 * tprec * tsens / (tprec + tsens) if tprec + tsens else 0.0

This collapsed cell builds the made-up vessel tree and the clDice figure.

In [ ]:
# @title Helper: the made-up vessel tree and the clDice figure { display-mode: "form" }
TOY_CASES = ["too wide", "capillaries missed", "blob"]


def vessel_toy(scale: float = 1.0, min_radius: int = 0, size: int = 128) -> np.ndarray:
    """A made-up vessel tree: a thick trunk, two branches, six capillaries. `scale` widens every vessel;
    `min_radius` leaves out the vessels thinner than that (2 drops the capillaries)."""
    from skimage.draw import line

    tree = [  # (x, y) corners of each vessel and its radius in pixels
        ([(0, 25), (30, 32), (60, 40), (100, 36), (127, 48)], 6),  # trunk
        ([(60, 40), (95, 20), (115, 4)], 3),  # branch up
        ([(60, 40), (48, 80), (70, 110), (75, 127)], 3),  # branch down
        ([(30, 32), (20, 10), (25, 2)], 1),  # capillaries from here on
        ([(48, 80), (25, 90), (8, 110)], 1),
        ([(48, 80), (30, 60), (8, 55)], 1),
        ([(100, 36), (112, 65), (105, 95)], 1),
        ([(100, 36), (85, 60), (95, 85)], 1),
        ([(70, 110), (100, 115), (125, 105)], 1),
    ]
    out = np.zeros((size, size), bool)
    for corners, radius in tree:
        if radius < min_radius:
            continue
        centre = np.zeros((size, size), bool)
        for (x0, y0), (x1, y1) in zip(corners[:-1], corners[1:], strict=True):
            rr, cc = line(y0, x0, y1, x1)
            centre[rr, cc] = True
        out |= ndi.distance_transform_edt(~centre) <= radius * scale
    return out


def toy_prediction(case: str, size: int = 128) -> np.ndarray:
    """The prediction of one case: every vessel twice as wide, no capillaries, or one big disc."""
    if case == "too wide":
        return vessel_toy(scale=2.0)
    if case == "capillaries missed":
        return vessel_toy(min_radius=2)
    yy, xx = np.mgrid[:size, :size]
    return (yy - size / 2) ** 2 + (xx - size / 2) ** 2 <= (0.7 * size) ** 2


def explain_cldice(case: str) -> None:
    """Draw the four steps of clDice on one case, and IoU next to clDice for all three cases."""
    if case not in TOY_CASES:
        raise SystemExit(f"CASE must be one of {TOY_CASES}.")
    truth = vessel_toy()

    def paint(img, mask, colour, alpha=1.0):
        img = img.copy()
        img[mask] = (1 - alpha) * img[mask] + alpha * hex_rgb(colour)
        return img

    def thick(mask):  # a 1 px line is drawn 3 px wide so it shows up
        return ndi.binary_dilation(mask, np.ones((3, 3), bool))

    blank = np.full(truth.shape + (3,), 0.97)
    results = {}
    for name in TOY_CASES:
        pred = toy_prediction(name)
        tprec, tsens, sk_pred, sk_truth = cldice_parts(pred, truth)
        both = (iou(pred, truth), cldice(pred, truth, sk_pred, sk_truth))
        results[name] = (pred, tprec, tsens, sk_pred, sk_truth, *both)
    pred, tprec, tsens, sk_pred, sk_truth, iou_c, cld_c = results[case]
    white = "#FFFFFF"
    n_in, n_all = int((sk_pred & truth).sum()), int(sk_pred.sum())
    m_in, m_all = int((sk_truth & pred).sum()), int(sk_truth.sum())

    fig, axes = plt.subplots(
        1, 5, figsize=(15, 3.8), gridspec_kw={"width_ratios": [1, 1, 1, 1, 1.25]}
    )
    img = paint(paint(blank, truth, THEME["truth"], 0.6), sk_truth, white)
    axes[0].imshow(img)
    axes[0].set_title("1 · truth + centre line", fontsize=9)
    img = paint(paint(blank, pred, THEME["background"], 0.85), sk_pred, white)
    axes[1].imshow(img)
    axes[1].set_title(f"2 · prediction ({case})\n+ centre line", fontsize=9)
    img = paint(blank, truth, THEME["truth"], 0.35)
    img = paint(paint(img, thick(sk_pred & ~truth), THEME["fails"]), thick(sk_pred & truth), THEME["works"])
    axes[2].imshow(img)
    axes[2].set_title(f"3 · tprec = {tprec:.2f}\n{n_in} of {n_all} counted", fontsize=9)
    img = paint(blank, pred, THEME["background"], 0.5)
    img = paint(paint(img, thick(sk_truth & ~pred), THEME["truth"]), thick(sk_truth & pred), THEME["works"])
    axes[3].imshow(img)
    axes[3].set_title(f"4 · tsens = {tsens:.2f}\n{m_in} of {m_all} counted", fontsize=9)
    for ax in axes[:4]:
        ax.axis("off")

    x = np.arange(len(TOY_CASES))
    vals = [(results[c][5], results[c][6]) for c in TOY_CASES]
    axes[4].bar(x - 0.2, [v[0] for v in vals], 0.4, color=THEME["background"], label="IoU")
    axes[4].bar(x + 0.2, [v[1] for v in vals], 0.4, color=THEME["works"], label="clDice")
    for xi, (a, b) in zip(x, vals, strict=True):
        axes[4].text(xi - 0.2, a + 0.02, f"{a:.2f}", ha="center", fontsize=7)
        axes[4].text(xi + 0.2, b + 0.02, f"{b:.2f}", ha="center", fontsize=7)
    axes[4].set_xticks(x, [c.replace(" ", "\n", 1) for c in TOY_CASES], fontsize=8)
    axes[4].get_xticklabels()[TOY_CASES.index(case)].set_fontweight("bold")
    axes[4].set_ylim(0, 1.15)
    axes[4].set_title("IoU vs clDice, all three cases\n(bold = the case drawn)", fontsize=9)
    axes[4].legend(fontsize=7, frameon=False, loc="upper right")
    add_key(
        fig,
        [
            ("truth", "green: truth, and truth line the prediction misses"),
            ("prediction (grey)", THEME["background"]),
            ("centre line (white)", "#FFFFFF"),
            ("counted (violet)", THEME["works"]),
            ("orange: prediction line outside the truth (invented)", THEME["fails"]),
        ],
    )
    plt.show()
    print(
        f"clDice = 2 · tprec · tsens ÷ (tprec + tsens) = 2 · {tprec:.2f} · {tsens:.2f} ÷ "
        f"({tprec:.2f} + {tsens:.2f}) = {cld_c:.2f}   (IoU is {iou_c:.2f})"
    )

This cell draws clDice on a made-up vessel tree (a thick trunk, two branches and six capillaries). Pick a
case in the `CASE` field and the figure redraws. The four steps below match the four left panels, with the
numbers of the default case, "too wide" (a prediction that draws every vessel twice as wide). The fifth
panel puts IoU next to clDice for all three cases. A blob's centre line is a short stub in its middle: here
the stub misses every vessel, so clDice is 0.

1. **Thin the true vessels to their centre line** ($\square$ white line on the
   $\color{#2E7D32}{\blacksquare}$ green truth, panel 1).
2. **Thin the prediction the same way** ($\square$ white line on the $\color{#9E9E9E}{\blacksquare}$ grey
   prediction, panel 2).
3. **tprec**: of the prediction's centre line, how much lies inside the true vessels?
   $\color{#6A1B9A}{\blacksquare}$ Violet pixels count,
   $\color{#E65100}{\blacksquare}$ orange ones do not. 490 of 529 pixels = 0.93.
4. **tsens**: of the true centre line, how much is covered by the prediction? 534 of 534 = 1.00.
   Then clDice = 2 · 0.93 · 1.00 ÷ (0.93 + 1.00) = 0.96, while IoU for the same masks is only 0.57.

What to look for: in "too wide" the prediction is fat but its centre line still follows the truth, so
clDice stays high and IoU drops. Try "capillaries missed": now tsens drops, because the thin branches are
gone.

In [ ]:
# @title See clDice on a made-up vessel tree { display-mode: "form", run: "auto" }
CASE = "too wide"  # @param ["too wide", "capillaries missed", "blob"]
explain_cldice(CASE)

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F5F5F5;border-left:6px solid #616161;border-radius:6px;padding:12px 14px;margin:6px 0;color:#616161;font-family:inherit;line-height:1.5"><details><summary><b>🔍 Where clDice comes from</b></summary><p>clDice is from Shit et al., "clDice: a novel topology-preserving loss function for tubular structure
segmentation", CVPR 2021.</p></details></div>"""))

### 2.3 Escalating prompts: 1 → 5 → 20 clicks, and a box

If one click is ambiguous, maybe more clicks tell SAM what we mean. Each step tells SAM something new:

1. **1 point**: one spot. SAM must guess: this vessel, the tree, or the eye?
2. **5 points**: five spots along different vessels. SAM looks for one object containing all the clicks.
3. **20 points**: dense along the tree. More information, but still only *where*, never *what kind*.
4. **box**: a box around the whole fundus says "everything in here".

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F3E5F5;border-left:6px solid #4A148C;border-radius:6px;padding:12px 14px;margin:6px 0;color:#4A148C;font-family:inherit;line-height:1.5"><b>🤔 Your prediction</b><p>Write down three guesses before you run the ladder.</p>
<ul>
<li>Will IoU rise steadily from 1 to 20 clicks, jump at some point, or stay flat?</li>
<li>Will the box beat 20 clicks?</li>
<li>Where will the <span style="color:#9E9E9E">■</span> grey line sit? It scores a mask that says "everything
  is foreground": if vessels cover 12 %
  of the fundus, overlap 12 % ÷ union 100 % gives IoU 0.12. The load cell printed your share.</li>
</ul></div>"""))

This cell runs the ladder on every evaluation image and prints three numbers per setting. **IoU** is
overlap ÷ union. **Recall** is the share of the true vessel pixels the mask finds. **Marked** is the share
of the fundus the mask covers (100 % = the whole eye).

Then two figures. The bars show mean IoU for each setting ($\color{#E65100}{\blacksquare}$ orange), the
dashed $\color{#9E9E9E}{\blacksquare}$ grey line is the score of
marking everything, and the dotted $\color{#6A1B9A}{\blacksquare}$ violet line is SAM's one-box score on
the pets. The pictures show the
four prompts ($\color{#00838F}{\blacksquare}$ teal stars and box) on the demo image, with SAM's mask in
$\color{#6A1B9A}{\blacksquare}$ violet next to the $\color{#2E7D32}{\blacksquare}$ green truth.
What to look for: bars that do not rise above the $\color{#9E9E9E}{\blacksquare}$ grey dashed line, and
masks that grow without getting
thinner.

In [ ]:
# @title Run the SAM prompt ladder on every evaluation image { display-mode: "form" }
LADDER = {"1 point": 1, "5 points": 5, "20 points": 20, "box": None}  # setting -> number of clicks
t0 = time.monotonic()
ladder_scores = {name: [] for name in LADDER}
ladder_cover = {name: [] for name in LADDER}
ladder_recall = {name: [] for name in LADDER}
ladder_demo = {}  # the demo image's masks and clicks, for the figure below
sam_box_masks = []  # every image's box mask, for the gallery in 2.5
everything = []
for it in tqdm(spec_eval, desc="SAM prompt ladder"):
    r = region(it)
    for name, n in LADDER.items():
        pts = scribble_points(it, n) if n else None
        if n:
            pred = sam.probability(it["image"], points=pts, labels=[1] * len(pts)) >= 0.5
        else:
            pred = sam.probability(it["image"], box=box_prompt(it)) >= 0.5
        ladder_scores[name].append(score(pred, it))
        ladder_cover[name].append(float(pred[r].mean()))
        ladder_recall[name].append(recall_of(pred, it))
        if it is demo:
            ladder_demo[name] = (pred, pts, box_prompt(it) if not n else None)
    sam_box_masks.append(pred)  # the last setting is the box
    everything.append(everything_score(it))
ladder_mean = {name: mean_scores(s) for name, s in ladder_scores.items()}
everything_mean = mean_scores(everything)
spec_sec = round(time.monotonic() - t0, 1)

RESULTS["sam_prompt_ladder"] = {name: v["iou"] for name, v in ladder_mean.items()}
RESULTS["sam_prompt_ladder_cldice"] = {name: v["cldice"] for name, v in ladder_mean.items()}
RESULTS["specialised_zero_shot_iou"] = ladder_mean["box"]["iou"]  # same headline as pets: one box
RESULTS["specialised_zero_shot_point_iou"] = ladder_mean["1 point"]["iou"]
RESULTS["specialised_zero_shot_box_iou"] = ladder_mean["box"]["iou"]
RESULTS["specialised_everything_iou"] = everything_mean["iou"]
RESULTS["specialised_zero_shot_seconds"] = spec_sec

vessel_share = float(np.mean([it["mask"][region(it)].mean() for it in spec_eval]))
print(f"SAM prompt ladder on {len(spec_eval)} held-out images · {spec_sec} s on {DEVICE}")
print(
    "calls vessel = share of the fundus the mask calls vessel "
    f"(100 % = the whole eye; the vessels really cover about {vessel_share:.0%})"
)
print("recall = share of the true vessel pixels the mask finds")
print(f"{'setting':<12}{'IoU':>7}{'recall':>8}{'calls vessel':>14}")
for name, v in ladder_mean.items():
    print(f"{name:<12}{v['iou']:>7.3f}{np.mean(ladder_recall[name]):>8.3f}{np.mean(ladder_cover[name]):>14.0%}")
print(f"{'everything':<12}{everything_mean['iou']:>7.3f}{1:>8.3f}{1:>14.0%}")
print(f"{'truth':<12}{'':>15}{vessel_share:>14.0%}")
box_marked = float(np.mean(ladder_cover["box"]))
if abs(ladder_mean["box"]["iou"] - everything_mean["iou"]) < 0.02:
    print(
        "'box' and 'everything' have the same IoU because SAM's box mask covers "
        f"{box_marked:.0%} of the fundus, so it scores like marking everything."
    )

fig, ax = plt.subplots(figsize=(6.4, 3))
names = list(ladder_mean)
ax.bar(names, [ladder_mean[k]["iou"] for k in names], color=THEME["fails"], label="retina, SAM")
ax.axhline(everything_mean["iou"], color=THEME["background"], ls="--", label='"everything is foreground"')
ax.axhline(RESULTS["pets_zero_shot_iou"], color=THEME["works"], ls=":", label="pets, SAM one box (section 1)")
ax.set_ylabel("mean IoU")
ax.set_ylim(0, 1)
ax.set_title("More prompts, same model: mean IoU on the retina")
ax.legend(fontsize=8, frameon=False, loc="center right")
plt.tight_layout()
plt.show()

show_clicks(
    [
        (pred_title(f"SAM, {k}", m, demo), overlay(demo["image"], m, THEME["works"]), p, b)
        for k, (m, p, b) in ladder_demo.items()
    ]
    + [("ground truth", overlay(demo["image"], demo["mask"], THEME["truth"]), None)],
    legend=[model_key("SAM's mask"), ("truth", "vessels (ground truth)"), "click", "box"],
    note=MARKED_NOTE,
)

**Q6.** 🟡 Compare the ladder with your prediction. Read the table row by row. Example: IoU 0.30 with 70 %
marked, against a $\color{#9E9E9E}{\blacksquare}$ grey line at 0.29, is a big mask that is barely better
than marking everything.

- Which rows beat the $\color{#9E9E9E}{\blacksquare}$ grey line by more than 0.05 IoU?
- Did more clicks bring the masks closer to the vessels, or just make them bigger?
- What does that say about what SAM thinks an object is?

<details><summary>What to look for</summary>

- Read IoU next to the $\color{#9E9E9E}{\blacksquare}$ grey line, not on its own. A mask that marks most of
  the fundus scores the
  "everything" baseline, whatever the prompt.
- If more clicks push "marked" up while IoU stays near the line, SAM is growing one big object around the
  clicks, and for a vessel tree that object is the eye.
- If clicks do help, it is along the thick vessels near the clicks, rarely the capillaries in between. A
  prompt says *where*, not *what kind of thing* (section 7 comes back to it).

</details>

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#FDECEA;border-left:6px solid #8B1A10;border-radius:6px;padding:12px 14px;margin:6px 0;color:#8B1A10;font-family:inherit;line-height:1.5"><b>⚠️ Pitfall: over-trusting SAM&#x27;s confident masks</b><p>SAM returns a crisp mask for every prompt. Its own confidence score is high for a mask of the whole eye,
because the whole eye <em>is</em> a good object. Nothing in the output says "this is the wrong kind of thing".</p>
<ul>
<li>Score against a baseline that knows nothing (the <span style="color:#9E9E9E">■</span> grey line).</li>
<li>Always look at the mask.</li>
</ul></div>"""))

### 2.4 The linear head, and the resolution it looks at

The head from section 1.4, trained the same way on the full masks of five retina images (the last time
training sees a full mask: a perfect vessel map costs an expert 3 to 5 hours per fundus). We train it
twice, on DINOv3 features at two resolutions:

- **512 px long side**: each 16 px patch covers 32 × 32 pixels of the 1024 px crop.
- **1024 px long side**: each patch covers 16 × 16 pixels. Four times the patches, and about 16 times the
  cost, because attention compares every patch with every other.

This collapsed cell trains (or loads) one head per resolution on the five fit images.

In [ ]:
# @title Helper: train one head per resolution, or load ours { display-mode: "form" }
N_TRAIN_SPEC = 5
sides = specialised["feature_long_sides"]  # [512, 1024]: the two resolutions we compare
sides = [side // 4 for side in sides] if SMOKE else sides  # smoke thumbnails are 256 px


def fit_spec_heads(train_yourself: bool) -> tuple[list[dict], dict]:
    """Train one linear head per resolution on the first N_TRAIN_SPEC fit images (or load ours).

    The border is labelled -1, so the loss skips it. Returns the fit images and {long side: (head, seconds)}.
    """
    if not isinstance(train_yourself, bool):
        raise SystemExit("TRAIN_SPEC_HEADS must be True or False.")
    items = [load_item(specialised, it) for it in specialised["fit"][:N_TRAIN_SPEC]]
    heads = {}
    for side in sides:
        if train_yourself:
            feats = [features_for(specialised, it, side) for it in items]
            labels = [np.where(region(it), it["mask"], -1).astype(np.int8) for it in items]
            heads[side] = train_head(feats, labels, HEAD_STEPS)
            src = f"trained here on {len(items)} images, {HEAD_STEPS} steps"
        else:
            tensors, entry = load_head(specialised, "full", side)
            heads[side] = (tensors, entry["train_sec"])
            src = f"our checkpoint {entry['file']} ({len(entry['train_names'])} images)"
        print(f"linear head @ {side} px · {src} · {heads[side][1]:.1f} s on {DEVICE}")
    if sum(sec for _, sec in heads.values()) > 60:
        print("ℹ️ Slow session? Untick TRAIN_SPEC_HEADS above to load our checkpoints next time.")
    return items, heads

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F3E5F5;border-left:6px solid #4A148C;border-radius:6px;padding:12px 14px;margin:6px 0;color:#4A148C;font-family:inherit;line-height:1.5"><b>🤔 Your prediction</b><p>With <em>perfect</em> labels, write two guesses.</p>
<ul>
<li>Will the head beat SAM's one-box IoU from the 2.3 table?</li>
<li>Will doubling the resolution add more than 0.05 to IoU, to recall, to both or to neither?</li>
</ul></div>"""))

This cell trains one head per resolution on the five fit images and prints the seconds it took. If it is
slow, untick `TRAIN_SPEC_HEADS` and run it again to load our checkpoints.

In [ ]:
# @title Train the two heads here (or load ours) { display-mode: "form" }
TRAIN_SPEC_HEADS = True  # @param {type:"boolean"}
spec_train, spec_heads = fit_spec_heads(TRAIN_SPEC_HEADS)

This cell scores both heads on the evaluation images and prints mean IoU, recall and "calls vessel" per
resolution, next to SAM's one box. What to look for: does doubling the resolution add more than 0.05 to
IoU, and does recall rise with it?

In [ ]:
# @title Score both heads on the evaluation images { display-mode: "form" }
t0 = time.monotonic()
spec_head_preds, res_scores, res_recall, res_marked = {}, {}, {}, {}
for side, (tensors, _) in spec_heads.items():
    spec_head_preds[side] = [
        predict_head(tensors, features_for(specialised, it, side), it["mask"].shape)
        for it in tqdm(spec_eval, desc=f"head @ {side} px")
    ]
    preds = spec_head_preds[side]
    res_scores[side] = mean_scores([score(p, it) for p, it in zip(preds, spec_eval, strict=True)])
    res_recall[side] = float(np.mean([recall_of(p, it) for p, it in zip(preds, spec_eval, strict=True)]))
    res_marked[side] = float(np.mean([p[region(it)].mean() for p, it in zip(preds, spec_eval, strict=True)]))
top_side = max(sides)  # the higher of the two resolutions
RESULTS["specialised_head_iou"] = res_scores[top_side]["iou"]
RESULTS["specialised_head"] = {
    "iou": res_scores[top_side]["iou"],
    "cldice": res_scores[top_side]["cldice"],
    "long_side": top_side,
    "n_train": len(spec_train),
    "train_sec": round(float(spec_heads[top_side][1]), 2),
    "infer_sec": round(time.monotonic() - t0, 1),
}
RESULTS["resolution_effect"] = {f"{side} px": v for side, v in res_scores.items()}
for side, v in res_scores.items():
    print(
        f"linear head @ {side:>4} px · mean IoU {v['iou']:.3f} · recall {res_recall[side]:.3f} · "
        f"calls {res_marked[side]:.0%} of the fundus vessel"
    )
print(
    f"SAM one box · mean IoU {ladder_mean['box']['iou']:.3f} · recall {np.mean(ladder_recall['box']):.3f} · "
    f"calls {np.mean(ladder_cover['box']):.0%} of the fundus vessel"
)
print(f"(really vessel: {fg_share:.0%} of the fundus; recall = share of the true vessel pixels a mask finds)")

This cell puts the section-1 pet numbers next to this section's vessel numbers, for the same two tools
(SAM and a linear head). The tools are the same, only the task changed.
$\color{#6A1B9A}{\blacksquare}$ Violet bars are the pets, $\color{#E65100}{\blacksquare}$ orange
bars the vessels, and the dashed $\color{#9E9E9E}{\blacksquare}$ grey line is the score of marking
everything on the vessels. What to look
for: every $\color{#E65100}{\blacksquare}$ orange bar sits far below its
$\color{#6A1B9A}{\blacksquare}$ violet neighbour and close to the dashed line.

In [ ]:
# @title Pets vs vessels: the same two tools, high on pets and low on vessels { display-mode: "form" }
rows = {
    "SAM,\none click": (RESULTS["pets_zero_shot_point_iou"], RESULTS["specialised_zero_shot_point_iou"]),
    "SAM,\none box": (RESULTS["pets_zero_shot_box_iou"], RESULTS["specialised_zero_shot_box_iou"]),
    "linear head on\nDINOv3 features": (RESULTS["pets_head_iou"], RESULTS["specialised_head_iou"]),
}
print(f"{'tool':<32}{'pets IoU':>10}{'vessels IoU':>13}")
for name, (p, v) in rows.items():
    print(f"{name.replace(chr(10), ' '):<32}{p:>10.3f}{v:>13.3f}")
print(f"{'everything is foreground':<32}{'':>10}{RESULTS['specialised_everything_iou']:>13.3f}")

x = np.arange(len(rows))
fig, ax = plt.subplots(figsize=(6.4, 3.2))
ax.bar(
    x - 0.2,
    [v[0] for v in rows.values()],
    0.4,
    color=THEME["works"],
    label=f"{pets_task['name']} (easy task)",
)
ax.bar(x + 0.2, [v[1] for v in rows.values()], 0.4, color=THEME["fails"],
       label=f"{specialised['name']} (specialised task)")
ax.axhline(RESULTS["specialised_everything_iou"], color=THEME["background"], ls="--",
           label='vessels, "everything is foreground"')
ax.set_xticks(x, list(rows), fontsize=8)
ax.set_ylabel("mean IoU on held-out images")
ax.set_ylim(0, 1)
ax.set_title("Same two tools, two tasks: pets vs vessels")
ax.legend(fontsize=8, frameon=False)
plt.tight_layout()
plt.show()

### 2.5 The failure gallery

Numbers average the failures away; the gallery shows them. It sorts the evaluation images from worst to
best (by the head's IoU at the higher resolution) and shows three at a time. Each row is one image: the
photo, SAM's mask from a box ($\color{#6A1B9A}{\blacksquare}$ violet), the head's mask
($\color{#6A1B9A}{\blacksquare}$ violet) and the ground truth ($\color{#2E7D32}{\blacksquare}$ green). What
to look
for: the kind of mistake, whole regions marked for SAM, missing thin vessels for the head.

This collapsed cell ranks the evaluation images by the head's IoU and draws the gallery.

In [ ]:
# @title Helper: the failure gallery, worst images first { display-mode: "form" }
def gallery_row(k: int, side: int) -> None:
    """One row for image k: the photo, SAM with a box, the head at `side` px, and the ground truth."""
    it = spec_eval[k]
    head_mask = spec_head_preds[side][k]
    show_masks(
        [
            (it["name"], it["image"]),
            (
                pred_title("SAM from a box", sam_box_masks[k], it),
                overlay(it["image"], sam_box_masks[k], THEME["works"]),
            ),
            (
                pred_title(f"linear head, features @ {side} px", head_mask, it),
                overlay(it["image"], head_mask, THEME["works"]),
            ),
            (
                f"ground truth\nvessels cover {100 * it['mask'][region(it)].mean():.0f} % of the fundus",
                overlay(it["image"], it["mask"], THEME["truth"]),
            ),
        ],
        width=2.8,
        legend=[model_key("the model's mask"), ("truth", "vessels (ground truth)")],
        note=MARKED_NOTE,
    )


def rank_images() -> tuple[list[dict], np.ndarray, int]:
    """Score the higher-resolution head on every evaluation image and rank the images, worst first.

    Returns the scores, the ranking (image indices) and the resolution used. The predictions are looked up by
    resolution (a key of `spec_head_preds`, such as 1024), never by image number."""
    side = max(spec_head_preds)
    scores = [score(p, it) for p, it in zip(spec_head_preds[side], spec_eval, strict=True)]
    return scores, np.argsort([s["iou"] for s in scores]), side


def show_gallery(start: int) -> None:
    """Draw the three images from rank `start` (0 = worst) of the ranking by the head's IoU."""
    if isinstance(start, bool) or not isinstance(start, int) or not 0 <= start < len(spec_eval):
        raise SystemExit(f"GALLERY_START must be a whole number from 0 to {len(spec_eval) - 1}.")
    scores, ranking, side = rank_images()
    for k in ranking[start : start + 3]:
        gallery_row(int(k), side)

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F3E5F5;border-left:6px solid #4A148C;border-radius:6px;padding:12px 14px;margin:6px 0;color:#4A148C;font-family:inherit;line-height:1.5"><b>👥 Pairs, 5 min</b><p>Run the next cell: it draws the failure gallery from the predictions you already have, three images at a
time, worst first (ranked by the head's IoU).</p>
<p><strong>Q7.</strong> 🟡 Each of you pages to a different place with the <code>GALLERY_START</code> field (0 = the three worst; try
3, 6, ...; change only that value) and finds the worst mask you can. Show it to your neighbour and name the
failure in three words. Pick from: missed thin vessels, vessels drawn too wide, the optic disc or the border
marked as vessel, whole regions marked.</p>
<ul>
<li>Is it the same failure for SAM and for the head?</li>
</ul></div>"""))

In [ ]:
# @title Q7 — The failure gallery, worst images first { display-mode: "form" }
GALLERY_START = 0  # @param {type:"integer"}
show_gallery(GALLERY_START)

<details><summary>What people usually find</summary>

- SAM tends to fail *globally*: it marks a whole region or one thick vessel, because it looks for an object.
- The head fails *locally*: it finds the thick vessels, draws them as wide as a patch, and loses the
  capillaries, because it labels 16 px squares.
- Neither fails by being unsure: both draw crisp masks. Always put the mask next to the truth.

</details>

**Got a photo of something thin or specialised?** Section 9 runs SAM, the ladder and the head on it.

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#E0F2F1;border-left:6px solid #004D40;border-radius:6px;padding:12px 14px;margin:6px 0;color:#004D40;font-family:inherit;line-height:1.5"><b>✅ Takeaway</b><ul>
<li><strong>Objects thinner than a 16 px patch</strong> → raise the resolution first, and check how far the numbers move.</li>
<li><strong>Score close to the "everything is foreground" line</strong> → the model learnt nothing about the task; always
  compare with a baseline that knows nothing, and look at the mask next to the truth.</li>
<li><strong>More clicks make SAM's mask bigger, not thinner</strong> → stop prompting: a prompt says <em>where</em>, not <em>what
  kind of thing</em>.</li>
<li><strong>IoU and clDice disagree</strong> (2.2) → trust the structure metric on thin objects, and look at the masks.</li>
</ul></div>"""))

## 3. Hide the labels: the naive baseline

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#E8F5E9;border-left:6px solid #1B5E20;border-radius:6px;padding:12px 14px;margin:6px 0;color:#1B5E20;font-family:inherit;line-height:1.5"><b>🎯 Objective</b><p>Train a head with "missing = background" and watch it fail differently.</p></div>"""))

A perfect vessel map of one fundus takes an expert 3 to 5 hours, so real label files are rarely complete.

Picture an annotator who traced the big vessels, ran out of time halfway through the tree, and saved the
file. Nothing in it says "I stopped here": every pixel they did not paint is stored as background, exactly
like the pixels that really are background.

We make such a file ourselves by hiding about half of the vessels from the ground truth (3.1), train the
section-2 head on it the obvious way (3.3), and score it on the vessels we hid (3.2). From here on the full
masks are used for scoring only.

The cheap labels of this notebook come in two kinds, both leaving pixels nobody looked at: **missing
labels** (sections 3 and 5) and **scribbles** (section 4). What should a model believe about those pixels?
A plain loss says "background".

This helper hides a random half of the vessel segments, the way the cached label files were made.

In [ ]:
# @title Missing labels: cut into segments, hide a share { display-mode: "form" }
import zlib

from scipy import ndimage as ndi


def label_units(mask: np.ndarray, thin: bool) -> np.ndarray:
    """Label map of the units we hide (0 = background): skeleton segments for thin structures or a single
    object, connected components otherwise. Every foreground pixel gets the label of its nearest segment."""
    lab, n = ndi.label(mask)
    if not thin and n >= 2:
        return lab
    sk = skeleton(mask)
    neighbours = ndi.convolve(sk.astype(np.uint8), np.ones((3, 3), np.uint8), mode="constant") - sk
    seg, _ = ndi.label(sk & ~(sk & (neighbours > 2)), structure=np.ones((3, 3)))  # cut at junctions
    if seg.max() == 0:
        return np.where(mask, 1, 0)
    _, (iy, ix) = ndi.distance_transform_edt(seg == 0, return_indices=True)
    return np.where(mask, seg[iy, ix], 0)


def hide_units(mask: np.ndarray, fraction: float, rng: np.random.Generator, thin: bool) -> np.ndarray:
    """The 'missing labels' file: the ground truth with ~fraction of its units removed. Returns the visible
    mask; the hidden foreground is mask & ~visible."""
    lab = label_units(mask, thin)
    ids = np.arange(1, int(lab.max()) + 1)
    rng.shuffle(ids)
    k = int(round(len(ids) * fraction))
    hidden_ids = ids[: max(1, k)] if len(ids) >= 2 else ids[:0]
    return mask & ~np.isin(lab, hidden_ids)


def hide_rng(name: str) -> np.random.Generator:
    """One fixed random generator per image, so the same segments are hidden on every run (N-013)."""
    return np.random.default_rng([SEED, zlib.crc32(f"{name}/hide".encode())])


def part_recall(preds: list[np.ndarray], items: list[dict], part: str) -> float:
    """Mean over images of the share of item[part] ("mask", "visible", "hidden"), inside the evaluation
    region, that the prediction marks. Uses `hidden_recall` from 3.2."""
    pairs = zip(preds, items, strict=True)
    return float(np.mean([hidden_recall(p, it[part] & region(it)) for p, it in pairs]))


def two_colour(
    image: np.ndarray, good: np.ndarray, bad: np.ndarray, bad_colour: str = THEME["fails"]
) -> np.ndarray:
    """Overlay `good` in violet ("works") and `bad` in `bad_colour` (default orange; pass THEME["truth"] for
    missed vessels, which are green) on one image."""
    return overlay(overlay(image, good, THEME["works"]), bad, bad_colour)

This helper draws a row of pictures with a colour key under them, so no colour is left unexplained.

In [ ]:
# @title show_keyed: pictures with a colour key { display-mode: "form" }
key_painted = (THEME["scribble"], "teal: painted (in the label file)")
key_hidden = (THEME["truth"], "green: hidden vessel")
key_found = (THEME["works"], "violet: hidden vessel the head found")
key_missed = (THEME["truth"], "green: hidden vessel it missed")


def show_keyed(
    panels: list[tuple[str, np.ndarray]],
    key: list[tuple[str, str]],
    width: float = 3.0,
    roles: list | None = None,
) -> None:
    """One row of RGB panels (title, image) with a colour key under them: `key` = [(colour, meaning), ...].
    Long titles wrap with a newline so neighbours never overlap. `roles` = one corner tag per panel."""
    fig, axes = plt.subplots(1, len(panels), figsize=(width * len(panels), width + 0.7))
    tags = roles or [None] * len(panels)
    for ax, (title, img), role in zip(np.atleast_1d(axes), panels, tags, strict=True):
        ax.imshow(img)
        ax.set_title(title, fontsize=9)
        ax.axis("off")
        tag_panel(ax, role)
    handles = [mpl.patches.Patch(color=c, label=m) for c, m in key]
    fig.legend(handles=handles, loc="lower center", ncol=len(handles), frameon=False, fontsize=8)
    fig.tight_layout(rect=(0, 0.1, 1, 1))
    plt.show()

### 3.1 Make a half-finished label file

This cell hides about half of the vessel segments (the share comes from the dataset config) of the five
retina training images of section 2.4 and shows one of them. Two words from here on: **painted** vessels
are the ones left in the label file, **hidden** vessels are in the full ground truth but the file leaves
them unpainted, so it calls them background.

The printout gives two shares that differ a little: the share of hidden *segments* is what we ask for, the
share of hidden vessel *pixels* is what results, because long segments hold more pixels than short ones.
What to look for: the $\color{#00838F}{\blacksquare}$ teal (painted) vessels in the third picture and the
$\color{#2E7D32}{\blacksquare}$ green (hidden) vessels in the
fourth together make the full ground truth in the second.

In [ ]:
# @title Hide half of the vessel segments and look at the result { display-mode: "form" }
missing_labels = specialised["missing_labels"]  # {"hidden_fraction": 0.5, "unit": ...}: never hard-coded here
miss_train = [load_item(specialised, it) for it in specialised["fit"][:N_TRAIN_SPEC]]
same_as_cache = []
for it in miss_train:
    rng = hide_rng(it["name"])
    visible = hide_units(it["mask"], missing_labels["hidden_fraction"], rng, specialised["thin_structures"])
    same_as_cache.append(float((visible == it["visible"]).mean()))
    it["visible"], it["hidden"] = visible, it["mask"] & ~visible  # from now on: what training may see
hidden_share = float(np.mean([it["hidden"].sum() / max(it["mask"].sum(), 1) for it in miss_train]))
print(
    f"{len(miss_train)} training images · {hidden_share:.0%} of the vessel pixels hidden "
    f"(asked for {missing_labels['hidden_fraction']:.0%} of the segments) · pixels identical to the cached "
    f"labels our checkpoints used: {np.mean(same_as_cache):.2%}"
)

ex = miss_train[0]
show_keyed(
    [
        (ex["name"], ex["image"]),
        ("full ground truth\n(evaluation only)", overlay(ex["image"], ex["mask"], THEME["truth"])),
        ("painted\n(what training may use)", overlay(ex["image"], ex["visible"], THEME["scribble"])),
        ('hidden\n(the file says "background")', overlay(ex["image"], ex["hidden"], THEME["truth"])),
    ],
    [(THEME["truth"], "green: true vessels (all of them, or the hidden ones)"), key_painted],
    width=2.8,
    roles=["input", "truth", "input", "truth"],
)

### 3.2 Two numbers: hidden vessels found, and precision

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#EDE7F6;border-left:6px solid #311B92;border-radius:6px;padding:12px 14px;margin:6px 0;color:#311B92;font-family:inherit;line-height:1.5"><b>📘 Theory: hidden vessels found, and precision</b><p><strong>Hidden vessels found</strong> = of the hidden vessel pixels (3.1), the share the model marks as vessel. It is
the usual <em>recall</em>, counted on the hidden pixels only. <strong>Precision</strong> = of the pixels the model marks, the
share that really are vessel.</p>
<ul>
<li>Hidden vessels found: 400 hidden vessel pixels, the model marks 120 of them: 120 ÷ 400 = 30%.</li>
<li>Precision: the model marked 150 pixels in total and 120 are vessels: 120 ÷ 150 = 80%.</li>
</ul>
<p>IoU mixes the two into one number. Here we want them apart, because the question is how many hidden
vessels the model finds.</p></div>"""))

This helper defines the two numbers above and the picture of what a head found and missed.

In [ ]:
# @title hidden_recall, marked_precision and the "found / missed" figure { display-mode: "form" }
def hidden_recall(pred: np.ndarray, hidden: np.ndarray) -> float:
    """Share of the `hidden` pixels that `pred` marks as foreground; 0.0 when `hidden` is empty."""
    hidden = np.asarray(hidden, bool)
    return float(np.asarray(pred, bool)[hidden].mean()) if hidden.any() else 0.0


def marked_precision(pred: np.ndarray, truth: np.ndarray) -> float:
    """Share of the pixels `pred` marks that are foreground in `truth`; 0.0 when `pred` marks nothing."""
    pred = np.asarray(pred, bool)
    return float(np.asarray(truth, bool)[pred].mean()) if pred.any() else 0.0


def found_panel(item: dict, pred: np.ndarray, label: str = "") -> tuple[str, np.ndarray]:
    """One picture: painted vessels in teal, hidden vessels the model found in violet, missed in green."""
    hidden = item["hidden"] & region(item)
    found, missed = hidden & pred, hidden & ~pred
    base = overlay(item["image"], item["visible"], THEME["scribble"])
    n = int(hidden.sum())
    title = (f"{label}\n" if label else "") + (
        f"found {int(found.sum())} of {n} hidden px = {hidden_recall(pred, hidden):.0%}"
    )
    return title, two_colour(base, found, missed, THEME["truth"])


def hidden_panels(item: dict, pred: np.ndarray) -> list[tuple[str, np.ndarray]]:
    """The three pictures: the label file (painted), the hidden vessels, and found / missed by `pred`."""
    base = overlay(item["image"], item["visible"], THEME["scribble"])
    hidden_pic = overlay(base, item["hidden"] & region(item), THEME["truth"])
    return [("label file:\npainted vessels", base), ("hidden:\nthe file says background", hidden_pic)] + [
        found_panel(item, pred)
    ]

### 3.3 The naive baseline: missing = background

"Naive" means: pass the label file straight into the textbook loss, with no thought about what the
unpainted pixels mean. We measure the hidden vessels found (3.2), next to the section-2 head's.

The next picture shows two vessel patches of a training image that look alike. The annotator painted one
and never reached the other. Each row is one patch: the photo around it, the same photo with the label
file on top, and the target the loss gets for the patch.

What to look for: the left pictures are almost the same, the right ones are opposite.

In [ ]:
# @title Picture: two look-alike patches, two targets { display-mode: "form" }
pic_patch = 16 * max(miss_train[0]["mask"].shape) / specialised["feat_side"]  # one head patch, in image px
pic_ph = max(2, int(round(pic_patch)))  # patch side, whole pixels
pic_big = 3 * pic_ph  # the crop reaches 3 patches to each side of the centre
pic_pair = None
for pic_item in miss_train:
    pic_h, pic_w = pic_item["mask"].shape
    cands = []
    for pic_part in ("hidden", "visible"):
        pys, pxs = np.nonzero(skeleton(pic_item[pic_part]))
        inside = (pys >= pic_big) & (pys < pic_h - pic_big) & (pxs >= pic_big) & (pxs < pic_w - pic_big)
        cands.append(np.stack([pys[inside], pxs[inside]], 1))
    if len(cands[0]) and len(cands[1]):
        pic_pair = (pic_item, cands)
        break
if pic_pair is None:
    print("no hidden and visible vessel segment far enough from the border to draw the picture")
else:
    pic_item, (pic_hid, pic_vis) = pic_pair
    pic_gray = pic_item["image"].astype(float).mean(axis=2)

    def pic_crop(arr: np.ndarray, centre: np.ndarray, half: int) -> np.ndarray:
        return arr[centre[0] - half : centre[0] + half, centre[1] - half : centre[1] + half]

    hid_centre = pic_hid[len(pic_hid) // 2]
    vis_pool = pic_vis[:: max(1, len(pic_vis) // 300)]
    gaps = [
        float(np.abs(pic_crop(pic_gray, c, pic_ph) - pic_crop(pic_gray, hid_centre, pic_ph)).mean())
        for c in vis_pool
    ]
    vis_centre = vis_pool[int(np.argmin(gaps))]  # the painted patch that looks most like the hidden one
    pic_seen = overlay(pic_item["image"], pic_item["hidden"], THEME["truth"])  # hidden vessels in green
    pic_seen = overlay(pic_seen, pic_item["visible"], THEME["scribble"])  # painted vessels in teal
    fig, axes = plt.subplots(2, 3, figsize=(8.4, 6.2))
    pic_rows = (
        (vis_centre, "painted patch", "vessel = 1"),
        (hid_centre, "hidden patch", "background = 0"),
    )
    for (centre, pic_name, target), row in zip(pic_rows, axes, strict=True):
        for ax, pic in zip(row[:2], (pic_item["image"], pic_seen), strict=True):
            ax.imshow(pic_crop(pic, centre, pic_big))
            for colour, width in (("white", 3.6), (THEME["scribble"], 1.8)):  # teal square with a white edge
                corner = pic_big - pic_ph / 2
                ax.add_patch(
                    mpl.patches.Rectangle((corner, corner), pic_ph, pic_ph, fill=False, ec=colour, lw=width)
                )
        row[0].set_title(f"{pic_name}\nthe photo", fontsize=9)
        row[1].set_title(f"{pic_name}\nwith the label file", fontsize=9)
        lab_patch = pic_crop(pic_item["visible"], centre, pic_ph // 2 + 1).astype(float)
        row[2].imshow(
            np.where(lab_patch[..., None] > 0, hex_rgb(THEME["scribble"]), hex_rgb(THEME["background"])),
            interpolation="nearest",
        )
        row[2].set_title("the target the loss\ngets for the patch", fontsize=9)
        row[2].text(
            0.5,
            0.5,
            target,
            transform=row[2].transAxes,
            ha="center",
            va="center",
            fontsize=12,
            fontweight="bold",
            color="white",
            bbox={"boxstyle": "round", "fc": "black", "alpha": 0.6},
        )
        for ax in row:
            ax.set_xticks([])
            ax.set_yticks([])
    add_key(
        fig,
        [
            ("teal: painted (target 1)", THEME["scribble"]),
            ("green: hidden vessel (target 0)", THEME["truth"]),
            ("grey: background target 0", THEME["background"]),
            ("box", "square: the patch the head sees"),
        ],
        note="Both patches show a vessel. The label file calls one vessel and the other background.",
    )
    plt.show()

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F3E5F5;border-left:6px solid #4A148C;border-radius:6px;padding:12px 14px;margin:6px 0;color:#4A148C;font-family:inherit;line-height:1.5"><b>🤔 Your prediction</b><p>The head is told that about half of the vessels in these five images are background. Write two guesses.</p>
<ul>
<li>Of the hidden vessel pixels, how many will it find? Near the section-2 head's share, about half of it, or
  close to zero?</li>
<li>On new images, will it mark more pixels than the section-2 head, or fewer?</li>
</ul></div>"""))

This helper trains the head on the label file as it is, or loads our checkpoint.

In [ ]:
# @title fit_naive: train the naive head, or load our checkpoint { display-mode: "form" }
def fit_naive(train_yourself: bool) -> tuple[dict, float, list[torch.Tensor], list[np.ndarray]]:
    """Naive head on the label file: returns (head, seconds, features, labels). The hidden ground truth never
    enters training (L-005)."""
    if not isinstance(train_yourself, bool):
        raise SystemExit("TRAIN_NAIVE_HEAD must be True or False.")
    side = specialised["feat_side"]
    feats = [features_for(specialised, it, side) for it in miss_train]
    labels = [np.where(region(it), it["visible"], -1).astype(np.int8) for it in miss_train]  # unpainted = 0
    if train_yourself:
        head, sec = train_head(feats, labels, HEAD_STEPS, balanced=False)
        src = f"trained here on {len(miss_train)} images, {HEAD_STEPS} steps, plain cross-entropy"
    else:
        head, entry = load_head(specialised, "hidden_bg", side)
        sec = entry["train_sec"]
        src = f"our checkpoint {entry['file']} ({len(entry['train_names'])} images)"
    print(f"naive head @ {side} px · {src} · {sec:.1f} s on {DEVICE}")
    if sec > 60:
        print("ℹ️ Slow session? Untick TRAIN_NAIVE_HEAD above to load our checkpoint next time.")
    return head, sec, feats, labels

This cell trains the naive head and prints the seconds it took; untick `TRAIN_NAIVE_HEAD` and run it again
to load our checkpoint.

In [ ]:
# @title Train the naive head { display-mode: "form" }
TRAIN_NAIVE_HEAD = True  # @param {type:"boolean"}
side = specialised["feat_side"]
naive_head, naive_sec, miss_feats, naive_labels = fit_naive(TRAIN_NAIVE_HEAD)

This cell scores the naive head and the section-2 head (full masks) on three sets of vessel pixels: the
ones we hid, the ones that were painted, and every vessel of the held-out images. Each bar is the share of
that set the head finds. The printout adds the held-out IoU, the precision, and **calls vessel**: the share
of the fundus (the scored area) the head calls vessel, next to the share that really is vessel.

What to look for (the numbers are from our run, yours will differ a little):

- The **hidden** bars: how far below the section-2 head is the naive head? On our run it finds about 6 %
  of the hidden vessels, the section-2 head about 88 %.
- The **painted** bars: does it find even the vessels it was told about? On our run it misses many.
- The printout: does the naive head mark more or less of the fundus, and is its precision higher or lower?

In [ ]:
# @title Score both heads on hidden, painted and held-out vessels { display-mode: "form" }
full_head = spec_heads[side][0]  # section 2.4: same features, trained on full masks
compared_heads = {"full masks (section 2)": full_head, "naive: missing = background": naive_head}
t0 = time.monotonic()
naive_report = {}
for name, tensors in compared_heads.items():
    fit_preds = [
        predict_head(tensors, f, it["mask"].shape) for f, it in zip(miss_feats, miss_train, strict=True)
    ]
    held_preds = [
        predict_head(tensors, features_for(specialised, it, side), it["mask"].shape) for it in spec_eval
    ]
    held = mean_scores([score(p, it) for p, it in zip(held_preds, spec_eval, strict=True)])
    held_pairs = list(zip(held_preds, spec_eval, strict=True))
    precision = [marked_precision(p & region(it), it["mask"]) for p, it in held_pairs]
    naive_report[name] = {
        "recall_hidden_fit": part_recall(fit_preds, miss_train, "hidden"),
        "recall_visible_fit": part_recall(fit_preds, miss_train, "visible"),
        "recall_held": part_recall(held_preds, spec_eval, "mask"),
        "precision_held": float(np.mean(precision)),
        "marked_held": float(np.mean([p[region(it)].mean() for p, it in held_pairs])),
        "iou_held": held["iou"],
        "cldice_held": held["cldice"],
        "fit_preds": fit_preds,
        "held_preds": held_preds,
    }
infer_sec = round(time.monotonic() - t0, 1)

naive = naive_report["naive: missing = background"]
RESULTS["naive_recall_hidden_fg"] = naive["recall_hidden_fit"]  # recall on the vessels hidden in training
RESULTS["naive_head"] = {k: v for k, v in naive.items() if not k.endswith("_preds")} | {
    "long_side": side,
    "n_train": len(miss_train),
    "hidden_share": hidden_share,
    "train_sec": round(float(naive_sec), 2),
    "infer_sec": infer_sec,
}

GROUPS = {
    "recall_hidden_fit": "hidden vessels\n(told: background)",
    "recall_visible_fit": "painted vessels\n(told: vessel)",
    "recall_held": "held-out images\n(all vessels)",
}
print(
    f"{'head':<30}{'hidden found':>13}{'painted':>9}{'held-out':>10}"
    "   held-out IoU / precision / calls vessel"
)
for name, r in naive_report.items():
    print(
        f"{name:<30}{r['recall_hidden_fit']:>13.3f}{r['recall_visible_fit']:>9.3f}{r['recall_held']:>10.3f}"
        f"   {r['iou_held']:.3f} / {r['precision_held']:.3f} / {r['marked_held']:.0%}"
    )
print(f"really vessel on the held-out images: {fg_share:.0%} of the fundus")

x = np.arange(len(GROUPS))
fig, ax = plt.subplots(figsize=(6.4, 3.2))
head_colours = (THEME["works"], "#BF360C")  # full-mask head, naive head: the naive colour of section 5
for shift, name, colour in zip((-0.14, 0.14), compared_heads, head_colours, strict=True):
    bars = ax.bar(x + shift, [naive_report[name][g] for g in GROUPS], 0.27, color=colour, label=name)
    ax.bar_label(bars, fmt="%.2f", fontsize=8)
ax.set_xticks(x, list(GROUPS.values()), fontsize=8)
ax.set_ylabel("share of vessel pixels found")
ax.set_ylim(0, 1.3)
ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
ax.set_title("Which vessels does each head find?")
ax.legend(fontsize=8, frameon=False, loc="upper right", ncol=1)
plt.tight_layout()
plt.show()

What the numbers say:

- **Stingy, not generous.** The section-2 head failed by marking too much (vessels as wide as a patch).
  This one fails by marking too little: it keeps the clearest vessels and often misses even the painted
  ones.
- **A close IoU can hide it.** IoU rewards the precision the naive head gained as much as it punishes the
  vessels it lost, so the two heads' held-out IoU can be close. One number would have hidden the failure.

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#EDE7F6;border-left:6px solid #311B92;border-radius:6px;padding:12px 14px;margin:6px 0;color:#311B92;font-family:inherit;line-height:1.5"><b>📘 Theory: why missing = background goes wrong</b><ol>
<li>The annotator painted about half of the vessels and stopped (the label file of 3.1).</li>
<li>The plain loss reads every unpainted pixel as "background", hidden vessels included.</li>
<li>So the head sees look-alike vessel patches with opposite labels (the picture above).</li>
<li>For two identical patches with opposite labels, the loss is lowest when the head answers "about 50 %
   vessel" for both.</li>
<li>50 % sits on the threshold, so faint and thin vessels fall just below it and are marked as background.</li>
<li>The head misses vessels, even painted ones. What it marks is mostly the clearest vessels, so it is
   mostly right: it looks precise.</li>
</ol>
<p>Check this against your guess and the numbers above.</p></div>"""))

This cell shows the naive head on its typical training image (the median by hidden vessels found).

What to look for: how much $\color{#2E7D32}{\blacksquare}$ green (missed) is left, and whether the
$\color{#6A1B9A}{\blacksquare}$ violet (found) vessels are the thick,
clear ones.

In [ ]:
# @title The typical training image: label file, hidden vessels, found and missed { display-mode: "form" }
per_image = [hidden_recall(p, it["hidden"]) for p, it in zip(naive["fit_preds"], miss_train, strict=True)]
typical = int(np.argsort(per_image)[len(per_image) // 2])
show_keyed(
    hidden_panels(miss_train[typical], naive["fit_preds"][typical]),
    [
        key_painted,
        (THEME["truth"], "green: hidden vessel (middle), missed by the head (right)"),
        key_found,
    ],
    width=3.6,
    roles=["input", "truth", "errors"],
)

**Try it: hide more or fewer vessels.** The next helper rebuilds the label file with a different share of
hidden segments, trains the naive head again on it (a few seconds), and prints the result next to the
default one.

In [ ]:
# @title naive_with_hidden_share: rebuild the label file, retrain, score { display-mode: "form" }
_HELD_FEATS: dict = {}


def naive_with_hidden_share(percent: int) -> None:
    """Hide `percent` % of the vessel segments (the 3.1 helper), retrain the plain head from scratch on that
    label file (the 3.3 helper), and print hidden vessels found and held-out precision next to the default."""
    if not isinstance(percent, int) or not 0 <= percent <= 90:
        raise SystemExit("HIDE_PERCENT must be a whole number from 0 to 90.")
    thin = specialised["thin_structures"]
    visibles = [
        it["mask"] if percent == 0 else hide_units(it["mask"], percent / 100, hide_rng(it["name"]), thin)
        for it in miss_train
    ]
    labels = [np.where(region(it), v, -1).astype(np.int8) for it, v in zip(miss_train, visibles, strict=True)]
    head, sec = train_head(miss_feats, labels, HEAD_STEPS, balanced=False)
    trial = [dict(it, visible=v, hidden=it["mask"] & ~v) for it, v in zip(miss_train, visibles, strict=True)]
    fit_preds = [predict_head(head, f, it["mask"].shape) for f, it in zip(miss_feats, trial, strict=True)]
    held_preds = []
    for it in spec_eval:
        if it["name"] not in _HELD_FEATS:
            _HELD_FEATS[it["name"]] = features_for(specialised, it, specialised["feat_side"])
        held_preds.append(predict_head(head, _HELD_FEATS[it["name"]], it["mask"].shape))
    shown = next((it for it in trial if it["hidden"].any()), None)  # first image that has hidden vessels
    has_hidden = shown is not None
    rows = {
        f"default ({missing_labels['hidden_fraction']:.0%} of segments)": (
            naive["recall_hidden_fit"],
            naive["recall_visible_fit"],
            naive["precision_held"],
        ),
        f"this run ({percent}% of segments)": (
            part_recall(fit_preds, trial, "hidden") if has_hidden else float("nan"),
            part_recall(fit_preds, trial, "visible"),
            float(
                np.mean(
                    [
                        marked_precision(p & region(it), it["mask"])
                        for p, it in zip(held_preds, spec_eval, strict=True)
                    ]
                )
            ),
        ),
    }
    share = np.mean([it["hidden"].sum() / max(it["mask"].sum(), 1) for it in trial])
    print(
        f"The naive head is retrained from scratch on the same {len(miss_train)} training images, with their "
        f"label files rebuilt so that {percent}% of the vessel segments are hidden (trained in {sec:.1f} s "
        f"on {DEVICE})."
    )
    print(
        f"That hides {share:.0%} of the vessel pixels: the two percentages differ because segments differ "
        f"in size, and long segments hold more pixels than short ones."
    )
    print(f"{'':<28}{'hidden found':>13}{'painted found':>15}{'held-out precision':>20}")
    for name, (hid, vis, prec) in rows.items():
        print(f"{name:<28}{hid:>13.2f}{vis:>15.2f}{prec:>20.2f}")
    if has_hidden:
        print(
            f"The three pictures show training image {trial.index(shown)}: its new label file, the vessels "
            f"hidden in it, and what the retrained head found of them."
        )
        show_keyed(
            hidden_panels(shown, fit_preds[trial.index(shown)]),
            [
                key_painted,
                (THEME["truth"], "green: hidden vessel (middle), missed by the head (right)"),
                key_found,
            ],
            width=3.0,
            roles=["input", "truth", "errors"],
        )
    else:
        print(
            "Nothing is hidden at 0%, so there is no 'hidden found' to report: the head sees the full masks."
        )

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F3E5F5;border-left:6px solid #4A148C;border-radius:6px;padding:12px 14px;margin:6px 0;color:#4A148C;font-family:inherit;line-height:1.5"><b>❓ Question</b><p><strong>Q8.</strong> 🟡 Write down a guess first: how will "hidden found" and "precision" change as you hide more of
the vessels? Then run the next cell with <code>HIDE_PERCENT</code> at 10, 50 and 90. For each of the two numbers,
say what happened, and what it means for an annotator who stops early.</p>
<p>What the cell trains: the naive head is retrained from scratch on the same training images, with their label
files rebuilt so that <code>HIDE_PERCENT</code> % of the vessel segments are hidden. The three pictures show training
image 0: its new label file, the hidden vessels, and what the retrained head found of them. The share of
hidden vessel <em>pixels</em> is a little different from <code>HIDE_PERCENT</code>, because segments differ in size.</p></div>"""))

In [ ]:
# @title Q8 — hide more or fewer vessels, retrain the naive head { display-mode: "form" }
HIDE_PERCENT = 70  # @param {type:"slider", min:0, max:90, step:10}
naive_with_hidden_share(HIDE_PERCENT)

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#E0F2F1;border-left:6px solid #004D40;border-radius:6px;padding:12px 14px;margin:6px 0;color:#004D40;font-family:inherit;line-height:1.5"><b>✅ Takeaway</b><ul>
<li><strong>A label file with only vessel and background</strong> → ask how it was made: an unpainted pixel may mean
  "unknown", not "background".</li>
<li><strong>A head trained with "missing = background"</strong> → expect a stingy model (it marks less, precision stays
  high, the vessels left out are mostly not found), so also measure the hidden vessels found on a small,
  fully labelled set.</li>
<li><strong>Three failures so far</strong> (SAM finds the eye, the head is limited by its 16 px patches, naive training
  believes the file) → none of these is a model-size problem we have tested; each needs different
  supervision. The hidden vessels found in this section is the baseline to beat; section 4 starts on how.</li>
</ul></div>"""))

## 4. The adaptation ladder

In section 3 the head believed a label file that said "background" wherever nobody had looked. Now the ways
out. The first is a ladder: give the foundation model a little more task information at each rung, and look
at what changes. Higher is not automatically better and each rung fails in its own way, so we look at the
masks, not only at the numbers.

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#E8F5E9;border-left:6px solid #1B5E20;border-radius:6px;padding:12px 14px;margin:6px 0;color:#1B5E20;font-family:inherit;line-height:1.5"><b>🎯 Objective</b><p>zero-shot → scribbles: see how predictions change when the model gets more task information. A third rung,
one annotated image, waits in a go-deeper box at the end.</p></div>"""))

The next cell draws what the model receives at each rung, with the time that annotation costs.

What to look for: the input is the only thing that changes, and one complete mask costs hours where a few
strokes cost minutes.

In [ ]:
# @title What each rung receives { display-mode: "form" }
def effort_minutes(n_strokes: int = 0, n_full_masks: int = 0) -> float:
    """Annotation minutes under the dataset's stated assumptions (seconds per stroke, hours per full mask)."""
    cost = specialised["annotation_effort"]
    return n_strokes * cost["seconds_per_stroke"] / 60 + n_full_masks * 60 * float(
        np.mean(cost["hours_per_full_mask"])
    )


_shot0 = load_item(specialised, specialised["fit"][0])
_scribs = [load_item(specialised, m) for m in specialised["fit"][:N_TRAIN_SPEC]]
_n_str = sum(m.get("n_fg_strokes", 0) + m.get("n_bg_strokes", 0) for m in specialised["fit"][:N_TRAIN_SPEC])


def _cost(minutes: float) -> str:
    if minutes < 1:
        return f"{minutes * 60:.0f} s"
    return f"{minutes:.0f} min" if minutes < 90 else f"{minutes / 60:.0f} h"


_ex = _scribs[0]
_thick = {v: ndi.binary_dilation(_ex["scribbles"] == v, iterations=2) for v in (1, 2)}
_strokes = overlay(_ex["image"], _thick[2], THEME["background"], 0.9)
_strokes = overlay(_strokes, _thick[1], THEME["scribble"], 0.9)
_c0, _c1 = _cost(effort_minutes(n_strokes=1)), _cost(effort_minutes(n_full_masks=1))
_c2 = _cost(effort_minutes(n_strokes=_n_str))
show_masks(
    [
        (f"Rung 0 · zero-shot\n1 image + 1 box · ≈ {_c0}", demo["image"]),
        (
            f"Rung 1 · one-shot (go-deeper)\n1 image + its complete mask · ≈ {_c1}",
            overlay(_shot0["image"], _shot0["mask"], THEME["scribble"]),
        ),
        (f"Rung 2 · scribbles\n{len(_scribs)} images + {_n_str} strokes · ≈ {_c2}", _strokes),
    ],
    prompts=[{"box": box_prompt(demo)}, None, None],
    width=3.4,
    roles=["input"] * 3,
    legend=[
        ("teal: what the person draws (box, mask, vessel strokes)", THEME["scribble"]),
        ("grey: background strokes", THEME["background"]),
    ],
    note="The model gets only these marks and nothing else about the task.",
)

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#EDE7F6;border-left:6px solid #311B92;border-radius:6px;padding:12px 14px;margin:6px 0;color:#311B92;font-family:inherit;line-height:1.5"><b>📘 Theory: what each rung tells the model</b><ul>
<li><strong>Rung 0 · zero-shot</strong> (section 2): SAM gets the image and one box, and nothing about the task. It
  returns its best guess of "the object": on the retina, the eye or one thick vessel.</li>
<li><strong>Rung 2 · scribbles:</strong> strokes on some vessels and some background in a few images; everything else is
  <em>unlabelled</em>. The section-2 head is trained with a loss that counts only the labelled pixels (4.2).</li>
<li><strong>Rung 1 · one-shot:</strong> one image with a complete vessel mask. It is the go-deeper box at the end of the
  section; its row joins the table when you run it.</li>
<li><strong>Cost:</strong> one complete mask takes hours, one scribble seconds.</li>
</ul></div>"""))

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F5F5F5;border-left:6px solid #616161;border-radius:6px;padding:12px 14px;margin:6px 0;color:#616161;font-family:inherit;line-height:1.5"><details><summary><b>🔍 Go deeper: where the rungs come from</b></summary><ul>
<li>Partial cross-entropy on scribbles: Tang et al., ECCV 2018.</li>
<li>The recipe of the MIT "DINOv2 + scribbles" work, here on DINOv3.</li>
<li>Prototypes from one example: PerSAM, Zhang et al., ICLR 2024.</li>
</ul></details></div>"""))

This helper defines the ladder's tools: one colour per approach, prototypes, error maps and a report.

In [ ]:
# @title Ladder helpers: colours, prototypes, error maps, report { display-mode: "form" }
approach_colours = {  # one colour per approach (L-003); later sections add theirs and keep these
    "zero_shot": "#E65100",
    "one_shot": "#C2185B",
    "scribbles": "#3949AB",
}
approach_names = {
    "zero_shot": "zero-shot: SAM, one box",
    "one_shot": "one-shot: nearest prototype",
    "scribbles": "scribbles: partial cross-entropy",
}


def patch_share(mask: np.ndarray, grid: tuple[int, int]) -> torch.Tensor:
    """Share of each feature patch covered by a boolean pixel mask: [h, w] floats in [0, 1]."""
    m = torch.from_numpy(np.asarray(mask, np.float32))[None, None]
    return F.interpolate(m, size=grid, mode="area")[0, 0].to(DEVICE)


def prototypes(feat: torch.Tensor, w_fg: torch.Tensor, w_bg: torch.Tensor) -> tuple[torch.Tensor, ...]:
    """Foreground and background prototypes: the weighted means of the unit-length patch features."""
    f = F.normalize(feat.flatten(1), dim=0)  # [D, h*w]
    return tuple(F.normalize(f @ w.flatten().to(f), dim=0) for w in (w_fg, w_bg))


@torch.no_grad()
def prototype_score(feat: torch.Tensor, protos: tuple[torch.Tensor, torch.Tensor]) -> torch.Tensor:
    """Per patch: cosine similarity to the foreground prototype minus to the background one ([h, w])."""
    return torch.einsum("dhw,d->hw", F.normalize(feat, dim=0), protos[0] - protos[1])


def score_to_mask(s: torch.Tensor, shape: tuple[int, int]) -> np.ndarray:
    """Upsample a patch score map to the image and mark the pixels nearer the foreground prototype."""
    up = F.interpolate(s[None, None], size=shape, mode="bilinear", align_corners=False)[0, 0]
    return (up > 0).cpu().numpy()


def heat_rgb(s: torch.Tensor, shape: tuple[int, int]) -> np.ndarray:
    """A score map as RGB on the shared score scale: blue = like the vessel prototype, white = in between
    (score 0, the cut), dark grey = like the background prototype."""
    up = F.interpolate(s[None, None], size=shape, mode="bilinear", align_corners=False)[0, 0].cpu().numpy()
    up = 0.5 + 0.5 * up / (np.abs(up).max() + 1e-9)
    return (score_cmap(up)[..., :3] * 255).astype(np.uint8)


def error_map(image: np.ndarray, pred: np.ndarray, item: dict) -> np.ndarray:
    """Inside the evaluation region: vessels found in violet, false alarms in orange, missed vessels in
    green (the ground truth the prediction did not cover); outside it, light grey (not scored)."""
    r = region(item)
    truth, p = item["mask"] & r, pred & r
    out = overlay(image, p & truth, THEME["works"], unscored=~r)
    out = overlay(out, p & ~truth, THEME["fails"])
    return overlay(out, truth & ~p, THEME["truth"])


def called_vessel(marked: float, truth_share: float | None = None) -> str:
    """The "marked" share in plain words: how much of the fundus a mask calls vessel, next to how much really
    is vessel (by default the evaluation images' share printed in section 2)."""
    truth_share = fg_share if truth_share is None else truth_share
    return f"calls {marked:.0%} of the fundus vessel (really vessel: {truth_share:.0%})"


def rung_title(label: str, pred: np.ndarray, item: dict) -> str:
    """Panel title: what made the mask, then IoU, recall and the share of this fundus it calls vessel."""
    r = region(item)
    truth, p = item["mask"] & r, pred & r
    numbers = f"IoU {iou(p, truth):.2f} · recall {hidden_recall(pred, truth):.2f}"
    return f"{label}\n{numbers}\ncalls {p[r].mean():.0%} vessel (really {truth[r].mean():.0%})"


def image_numbers(pred: np.ndarray, item: dict) -> dict:
    """IoU, recall, precision and marked share of one prediction on one image."""
    r = region(item)
    p = pred & r
    return {
        "iou": iou(p, item["mask"] & r),
        "recall": hidden_recall(pred, item["mask"] & r),
        "precision": hidden_recall(item["mask"], p),
        "marked": float(p[r].mean()),
    }


def rung_report(preds: list[np.ndarray]) -> dict:
    """Mean IoU, clDice, recall, precision and marked share of the evaluation region, over spec_eval."""
    pairs = tqdm(list(zip(preds, spec_eval, strict=True)), desc="scoring", leave=False)
    out = mean_scores([score(p, it) for p, it in pairs])
    out["recall"] = part_recall(preds, spec_eval, "mask")
    pairs = zip(preds, spec_eval, strict=True)
    out["precision"] = float(np.mean([hidden_recall(it["mask"], p & region(it)) for p, it in pairs]))
    out["marked"] = float(np.mean([p[region(it)].mean() for p, it in zip(preds, spec_eval, strict=True)]))
    return out

### 4.1 Rung 0: zero-shot, from section 2

Nothing new to run: the next cell measures section 2's SAM-with-one-box masks with the numbers every rung
is compared on (the table in 4.3 explains each one). The last line answers "how much of the photo does the
mask call vessel?": a mask that calls 99 % of the fundus vessel, when about a tenth really is, has simply
painted the whole eye.

In [ ]:
# @title Score rung 0 { display-mode: "form" }
k_demo = next(i for i, it in enumerate(spec_eval) if it is demo)  # the section-2 demo image
ladder_preds = {"zero_shot": sam_box_masks}  # section 2.3: SAM, one box, per evaluation image
ladder_report = {"zero_shot": rung_report(sam_box_masks)}
v = ladder_report["zero_shot"]
print(
    f"zero-shot (SAM, one box) on {len(spec_eval)} held-out images · IoU {v['iou']:.3f} · recall "
    f"{v['recall']:.3f} · precision {v['precision']:.3f}\n{called_vessel(v['marked'])}"
)

### 4.2 Rung 2: scribbles

The annotator from section 3 again, with a new instruction: "don't trace everything; draw a few strokes
along some vessels and a few on clear background, in each of five images, then stop". Every other pixel is
*unlabelled*, and this time the stroke map says so, so the loss can skip it.

In the stroke map 0 = unlabelled, 1 = vessel stroke, 2 = background stroke. In the training targets
−1 = ignored.

The next cell shows the strokes on one training image and, in the third picture, what the loss will see.

What to look for: almost the whole picture is dark, so the loss ignores almost every pixel.

In [ ]:
# @title Show the scribbles of one training image { display-mode: "form" }
scrib_train = [load_item(specialised, it) for it in specialised["fit"][:N_TRAIN_SPEC]]
labelled = float(np.mean([(it["scribbles"] > 0)[region(it)].mean() for it in scrib_train]))
scrib_meta = specialised["fit"][:N_TRAIN_SPEC]
n_strokes = sum(m.get("n_fg_strokes", 0) + m.get("n_bg_strokes", 0) for m in scrib_meta)
print(
    f"{len(scrib_train)} training images · {n_strokes} strokes · {labelled:.2%} of the fundus pixels "
    f"labelled · ≈ {effort_minutes(n_strokes=n_strokes):.1f} min at "
    f"{specialised['annotation_effort']['seconds_per_stroke']} s per stroke"
)
ex = scrib_train[0]
thick = {v: ndi.binary_dilation(ex["scribbles"] == v, iterations=2) for v in (1, 2)}
strokes = overlay(overlay(ex["image"], thick[2], THEME["background"], 0.9), thick[1], THEME["scribble"], 0.9)
counted = ndi.binary_dilation(ex["scribbles"] > 0, iterations=2)  # thickened for display only
loss_sees = np.where(counted[..., None], strokes, (ex["image"] * 0.25).astype(np.uint8))
show_masks(
    [
        (ex["name"], ex["image"]),
        ("scribbles: vessel · background", strokes),
        ("what the loss sees", loss_sees),
        ("ground truth · evaluation only", overlay(ex["image"], ex["mask"], THEME["truth"])),
    ],
    width=2.8,
    roles=["input", "input", "input", "truth"],
    legend=[
        ("teal: vessel stroke", THEME["scribble"]),
        ("grey: background stroke", THEME["background"]),
        ("dark: pixels the loss ignores", "#404040"),
        "truth",
    ],
    note="Strokes are thickened for the figure only.",
)

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#EDE7F6;border-left:6px solid #311B92;border-radius:6px;padding:12px 14px;margin:6px 0;color:#311B92;font-family:inherit;line-height:1.5"><b>📘 Theory: partial cross-entropy</b><p>The loss of section 2 adds a cross-entropy term for every pixel. The partial version adds it only over
the set S of pixels a person labelled:</p>
<p><code>L = −(1/|S|) · Σ_{i∈S} [ y_i·log p_i + (1−y_i)·log(1−p_i) ]</code></p>
<ul>
<li>A vessel stroke says y = 1, a background stroke says y = 0.</li>
<li>An unlabelled pixel is not in S: it adds nothing. In code it is a target of <code>-1</code>, which <code>train_head</code>
  already skips (it has skipped the fundus border that way since section 2).</li>
<li>In section 3 the unpainted pixels were targets of <code>0</code>: a thin vessel nobody painted counted as a
  "background" error.</li>
</ul></div>"""))

This helper turns a stroke map into training targets.

In [ ]:
# @title scribble_targets: stroke map → training targets { display-mode: "form" }
def scribble_targets(scribbles: np.ndarray, region: np.ndarray, unlabelled: int = -1) -> np.ndarray:
    """Targets (int8) from a stroke map: 1 vessel stroke, 0 background stroke, `unlabelled` everywhere else
    (-1 = ignored, 0 = background); -1 outside `region`."""
    targets = np.full(scribbles.shape, unlabelled, np.int8)
    targets[scribbles == 2] = 0
    targets[scribbles == 1] = 1
    targets[~np.asarray(region, bool)] = -1
    return targets

This helper trains the section-2 head on the strokes, or loads our checkpoint.

In [ ]:
# @title Scribble-head helper: train on the strokes, or load ours { display-mode: "form" }
def train_scribble_head(train_yourself: bool, unlabelled: int):
    """The scribble head of rung 2. Returns the head's tensors, the training seconds and the DINOv3
    features of the training images (section 6 trains on them too)."""
    if not isinstance(train_yourself, bool):
        raise SystemExit("TRAIN_SCRIBBLE_HEAD must be True or False.")
    if unlabelled not in (-1, 0):
        raise SystemExit("UNLABELLED_TARGET must be -1 (ignore) or 0 (background).")
    side = specialised["feat_side"]
    feats = [features_for(specialised, it, side) for it in scrib_train]
    # the rung's supervision: strokes only; unlabelled pixels and the border are ignored (-1) by default
    labels = [scribble_targets(it["scribbles"], region(it), unlabelled) for it in scrib_train]
    if unlabelled != -1 and not train_yourself:
        print("ℹ️ Our checkpoint ignores unlabelled pixels, so with 0 the head is trained here.")
        train_yourself = True
    if train_yourself:
        head, sec = train_head(feats, labels, HEAD_STEPS)
        what = "ignored" if unlabelled == -1 else "background"
        src = f"trained here on {len(scrib_train)} images, {HEAD_STEPS} steps, unlabelled = {what}"
    else:
        head, entry = load_head(specialised, "scribbles", side)
        sec = entry["train_sec"]
        src = f"our checkpoint {entry['file']} ({len(entry['train_names'])} images)"
    print(f"scribble head @ {side} px · {src} · {sec:.1f} s on {DEVICE}")
    if sec > 60:
        print("ℹ️ Slow session? Set TRAIN_SCRIBBLE_HEAD = False above to load our checkpoint next time.")
    return head, sec, feats

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F3E5F5;border-left:6px solid #4A148C;border-radius:6px;padding:12px 14px;margin:6px 0;color:#4A148C;font-family:inherit;line-height:1.5"><b>🤔 Your prediction</b><p>The head will see labels on fewer than two pixels in a hundred (see the printout above), and no vessel
will ever be called "background". Write two guesses:</p>
<ul>
<li>Will it find more of the held-out vessels (recall) than SAM's box, or fewer?</li>
<li>Will it mark more of the fundus, or less?</li>
</ul></div>"""))

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#FFE8D6;border-left:6px solid #7A2E00;border-radius:6px;padding:12px 14px;margin:6px 0;color:#7A2E00;font-family:inherit;line-height:1.5"><b>✏️ Your turn: unlabelled pixels, ignored or background?</b><ul>
<li>The next cell trains the section-2 head on the strokes of the five training images. <code>UNLABELLED_TARGET</code>
  says what an unlabelled pixel counts as: <strong>-1</strong> ignored (the default), <strong>0</strong> background.</li>
<li>Predict first: what happens to the vessels nobody stroked if you pick 0? (Section 3 has the answer.)</li>
<li>Run it with -1 and read the table in 4.3. If you have time, pick 0, run the cells down to that table
  again, and compare.</li>
<li><code>TRAIN_SCRIBBLE_HEAD = False</code> loads our checkpoint, which used -1. With 0 the head is always trained here.</li>
</ul></div>"""))

In [ ]:
# @title Train the scribble head on the strokes { display-mode: "form" }
TRAIN_SCRIBBLE_HEAD = True  # @param {type:"boolean"}
UNLABELLED_TARGET = -1  # @param [-1, 0] {type:"raw"}
scrib_head, scrib_sec, scrib_feats = train_scribble_head(TRAIN_SCRIBBLE_HEAD, UNLABELLED_TARGET)

This cell scores the scribble head on the evaluation images with the same numbers as rung 0. Compare its
"calls … vessel" share with the real one: everything above it is fundus the head paints by mistake.

In [ ]:
# @title Score the scribble head on the evaluation images { display-mode: "form" }
t0 = time.monotonic()
side = specialised["feat_side"]
ladder_preds["scribbles"] = [
    predict_head(scrib_head, features_for(specialised, it, side), it["mask"].shape) for it in spec_eval
]
ladder_report["scribbles"] = rung_report(ladder_preds["scribbles"])
scrib_infer = round(time.monotonic() - t0, 1)
v = ladder_report["scribbles"]
print(
    f"scribble head · IoU {v['iou']:.3f} · recall {v['recall']:.3f} · precision {v['precision']:.3f} · "
    f"{scrib_infer} s on {DEVICE}\n{called_vessel(v['marked'])}"
)

### 4.3 The ladder, side by side

The next cell prints one row per rung with the annotation it needed, then draws each rung's error map on
the demo image. The numbers:

- **IoU:** overlap of the mask with the true vessels.
- **Recall:** of the true vessel pixels, the share the mask finds.
- **Precision:** of the pixels the mask marks, the share that really are vessel.
- **Called vessel:** the share of the fundus (the scored area) the mask calls vessel. Compare it with the
  last line, the share that really is vessel: far above it means the mask paints background too.

What to look for: which rung misses vessels ($\color{#2E7D32}{\blacksquare}$ green) and which invents them
($\color{#E65100}{\blacksquare}$ orange).

In [ ]:
# @title Table and error maps of the rungs { display-mode: "form" }
effort = {
    "zero_shot": effort_minutes(n_strokes=len(spec_eval)),  # one box per image you segment
    "one_shot": effort_minutes(n_full_masks=1),
    "scribbles": effort_minutes(n_strokes=n_strokes),
}
effort_text = {
    "zero_shot": f"1 box per image ({len(spec_eval)} here)",
    "one_shot": "1 complete mask",
    "scribbles": f"{n_strokes} strokes, {len(scrib_train)} images",
}
RESULTS["ladder"] = {k: ladder_report[k]["iou"] for k in ("zero_shot", "scribbles")}  # one_shot: go-deeper
RESULTS["ladder_detail"] = dict(ladder_report)
RESULTS["ladder_effort_min"] = {k: round(v, 2) for k, v in effort.items()}
RESULTS["scribbles_head"] = ladder_report["scribbles"] | {
    "long_side": side,
    "n_train": len(scrib_train),
    "n_strokes": n_strokes,
    "labelled_fraction": labelled,
    "train_sec": round(float(scrib_sec), 2),
    "infer_sec": scrib_infer,
}

print(f"{'approach':<36}{'IoU':>6}{'recall':>8}{'precis.':>8}{'called vessel':>15}   annotation (≈ minutes)")
for k, v in ladder_report.items():
    print(
        f"{approach_names[k]:<36}{v['iou']:>6.3f}{v['recall']:>8.3f}{v['precision']:>8.3f}"
        f"{v['marked']:>15.0%}   {effort_text[k]} (≈ {effort[k]:.1f})"
    )
print(f"{'everything is foreground':<36}{RESULTS['specialised_everything_iou']:>6.3f}{'':>16}{1:>15.0%}")
print(f"{'really vessel (the ground truth)':<36}{'':>22}{fg_share:>15.0%}")

show_masks(
    [
        (
            rung_title(approach_names[k].split(":")[0], ladder_preds[k][k_demo], demo),
            error_map(demo["image"], ladder_preds[k][k_demo], demo),
        )
        for k in ladder_preds
    ]
    + [("ground truth", overlay(demo["image"], demo["mask"], THEME["truth"], unscored=~region(demo)))],
    width=3.2,
    roles=["errors"] * len(ladder_preds) + ["truth"],
    legend=["found", "invented", "missed", "unscored"],
    note="Each rung's mask on the demo image, compared with the ground truth.",
)

**Q9.** 🟡 Read the table and the two error maps:

- Which rung misses more vessels ($\color{#2E7D32}{\blacksquare}$ green), and which invents more
  ($\color{#E65100}{\blacksquare}$ orange)? Is the rung with the higher IoU
  also the one with fewer of both?
- You need a screening tool that must not miss a vessel, and a tool whose every mark must be right. Do you
  pick the same rung for both?

<details><summary>What to look for</summary>

- SAM's box tends to mark a region or the whole eye: high recall, many false alarms, an IoU near the
  "everything is foreground" row.
- The scribble head saw the fewest labelled pixels, yet from several images, and it was never told a vessel
  was background. Its errors are not section 3's: compare its recall with the naive head's.
- Which rung is best depends on whether a miss or a false alarm costs more, and on the minutes in the last
  column. Your order may differ from your neighbour's.

</details>

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F5F5F5;border-left:6px solid #616161;border-radius:6px;padding:12px 14px;margin:6px 0;color:#616161;font-family:inherit;line-height:1.5"><details><summary><b>🔍 Go deeper: rung 1, one fully annotated image</b></summary><p>Someone spends an afternoon tracing every vessel of <strong>one</strong> training image. That mask is all this rung
may use. No training, just one mean per class:</p>
<ol>
<li>Cut the annotated image into 16 px patches; DINOv3 gives one feature vector per patch.</li>
<li>Average the vessel patches into a vessel prototype, the background patches into a background prototype.</li>
<li>For a new image, score every patch: <code>cos(feature, vessel) − cos(feature, background)</code>.</li>
<li>Upsample the scores to pixels and mark those above 0.</li>
</ol>
<p>Before you run the next cell, guess: will the prototype mark more of the fundus than SAM's box, or less?
Will it find the thin vessels, or only the thick ones?</p>
<p>Then move <code>ONE_SHOT_IMAGE</code> and run the cell again. Does the held-out IoU move much, and what does that say
about relying on a single example?</p>
<p>The prototype is as coarse as a 16 px patch, like the section-2 head, and it depends on <em>which</em> example
you annotated: with one image, a lucky or unlucky choice is the whole training set.</p></details></div>"""))

This helper builds the prototypes from one annotated image and scores every evaluation image.

In [ ]:
# @title One-shot helper: prototypes from one annotated image { display-mode: "form" }
def run_one_shot(image_index: int):
    """Rung 1 from the annotated training image `image_index`: prints the numbers and draws the maps.
    Returns the annotated item, the similarity map of every evaluation image and the scoring seconds."""
    if not isinstance(image_index, int) or not 0 <= image_index < len(specialised["fit"]):
        raise SystemExit(f"ONE_SHOT_IMAGE must be a whole number from 0 to {len(specialised['fit']) - 1}.")
    side = specialised["feat_side"]
    shot = load_item(specialised, specialised["fit"][image_index])
    shot_feat = features_for(specialised, shot, side)
    grid, r = tuple(shot_feat.shape[1:]), region(shot)
    # the rung's supervision: one complete mask of one *training* image (L-005)
    protos = prototypes(shot_feat, patch_share(shot["mask"] & r, grid), patch_share(~shot["mask"] & r, grid))
    t0 = time.monotonic()
    proto_scores = [prototype_score(features_for(specialised, it, side), protos) for it in spec_eval]
    ladder_preds["one_shot"] = [
        score_to_mask(s, it["mask"].shape) for s, it in zip(proto_scores, spec_eval, strict=True)
    ]
    ladder_report["one_shot"] = rung_report(ladder_preds["one_shot"])
    proto_sec = round(time.monotonic() - t0, 1)
    v = ladder_report["one_shot"]
    RESULTS.setdefault("ladder", {})["one_shot"] = v["iou"]
    RESULTS.setdefault("ladder_detail", {})["one_shot"] = v
    RESULTS["one_shot"] = v | {"image": shot["name"], "infer_sec": proto_sec}
    print(
        f"one-shot prototype from training image {image_index} ({shot['name']}) · IoU {v['iou']:.3f} · "
        f"recall {v['recall']:.3f} · precision {v['precision']:.3f} · {proto_sec} s on {DEVICE}\n"
        f"{called_vessel(v['marked'])}"
    )
    print(f"for scale: the section-2 head on five complete masks · IoU {RESULTS['specialised_head_iou']:.3f}")
    show_masks(
        [
            (
                f"the one annotated image\n{shot['name']}",
                overlay(shot["image"], shot["mask"], THEME["scribble"]),
            ),
            ("similarity: vessel − background", heat_rgb(proto_scores[k_demo], demo["mask"].shape)),
            (
                rung_title("one-shot prototype", ladder_preds["one_shot"][k_demo], demo),
                error_map(demo["image"], ladder_preds["one_shot"][k_demo], demo),
            ),
        ],
        width=3.4,
        roles=["input", "map", "errors"],
        legend=[
            ("teal: the annotator's complete mask (input)", THEME["scribble"]),
            ("score map: looks like the vessel prototype", THEME["score"]),
            ("score map: looks like the background", "#424242"),
            "found",
            "invented",
            "missed",
            "unscored",
        ],
        note="Panels 2 and 3 show the held-out demo image; the next cell browses all held-out images.",
    )
    return shot, proto_scores, proto_sec

This cell runs the one-shot rung; it also runs under "Run all", so the table in section 8 has its row.

**`ONE_SHOT_IMAGE`** chooses *which* training image the annotator traced completely: 0 is the first of the
20 training images, 19 the last. That one mask is all the rung learns from; the left panel shows it. The
25 held-out images it is scored on never change.

In [ ]:
# @title One-shot: one annotated image, applied to every held-out image { display-mode: "form" }
ONE_SHOT_IMAGE = 0  # @param {type:"slider", min:0, max:19, step:1}
shot, proto_scores, proto_sec = run_one_shot(ONE_SHOT_IMAGE)

The next cell lets you flip through the one-shot result on every held-out image (◀ ▶ or the slider): the
photo the model gets, its score map, and its errors against the truth. It reuses the cell above, so run that
one first after changing `ONE_SHOT_IMAGE`. What to look for: does the prototype fail the same way on every
image, or only on some?

In [ ]:
# @title Browse the one-shot result on every held-out image { display-mode: "form" }
def one_shot_browser() -> None:
    """One figure per held-out image (photo, score map, error map) in a ◀ ▶ browser."""
    figs, labels = [], []
    for s, pred, it in zip(proto_scores, ladder_preds["one_shot"], spec_eval, strict=True):
        figs.append(
            show_masks(
                [
                    (f"{it['name']} · the photo", it["image"]),
                    ("similarity: vessel − background", heat_rgb(s, it["mask"].shape)),
                    (rung_title("one-shot prototype", pred, it), error_map(it["image"], pred, it)),
                ],
                width=3.0,
                roles=["input", "map", "errors"],
                legend=[
                    ("score: like the vessels", THEME["score"]),
                    ("score: like the background", "#424242"),
                    "found", "invented", "missed", "unscored",
                ],
                show=False,
            )
        )
        labels.append(f"{it['name']} · IoU {iou(pred & region(it), it['mask'] & region(it)):.2f}")
    browse_figures(figs, labels, f"One-shot prototype from {shot['name']}, on the held-out images")


one_shot_browser()

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#E0F2F1;border-left:6px solid #004D40;border-radius:6px;padding:12px 14px;margin:6px 0;color:#004D40;font-family:inherit;line-height:1.5"><b>✅ Takeaway</b><ul>
<li><strong>Zero-shot IoU looks fine but the mask covers a whole region</strong> → read the "marked" share and the error
  map, not only IoU.</li>
<li><strong>You have cheap scribbles</strong> → train with a loss that ignores unlabelled pixels. "Far from every stroke,
  so surely background" is a guess that brings section 3's mistake back.</li>
<li><strong>You have one complete mask</strong> → expect patch-wide vessels, and swap the annotated image to see how
  much the result depends on your choice.</li>
<li><strong>Two rungs score alike</strong> → compare their error maps and the minutes of annotation: they differ more
  in how they fail than in one number. Keep the rung colours; they return in section 8.</li>
</ul></div>"""))

## 5. Path 1 · Learn from less

Same trick as 4.2 ("unlabelled = ignored"), now measured on the half-painted label file of section 3. The
naive head failed because the loss got that file as it was, and the file calls every unpainted pixel
"background". Path 1 keeps the same file, features and head, and changes only what the loss is told about
the unpainted pixels:

- **Masked:** "unknown". The loss skips them; the only background it hears about is a few strokes on
  clear background. "Unknown" is the simplest way to make a loss uncertainty-aware.
- **PU** (positive-unlabelled: you know some positives, everything else is unknown): "unlabelled, and
  about this share of them is vessel". The loss uses every pixel and corrects for the vessels hiding
  among them.

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#E8F5E9;border-left:6px solid #1B5E20;border-radius:6px;padding:12px 14px;margin:6px 0;color:#1B5E20;font-family:inherit;line-height:1.5"><b>🎯 Objective</b><p>treat missing labels as unknown (write the masked loss), then tell the loss how many vessels hide among the
unpainted pixels (the PU loss).</p></div>"""))

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#EDE7F6;border-left:6px solid #311B92;border-radius:6px;padding:12px 14px;margin:6px 0;color:#311B92;font-family:inherit;line-height:1.5"><b>📘 Theory: the masked loss</b><p>A <strong>masked loss</strong> is the partial cross-entropy of 4.2: only the pixels that carry a label count, and an
ignored pixel adds nothing and sends no gradient.</p>
<ul>
<li><strong>Example, a thin vessel nobody painted.</strong> Naive (section 3): its pixels count as background, so the
  loss punishes the head for calling them vessel. Masked: they are ignored, so nothing punishes it.</li>
<li><strong>Catch:</strong> if the known pixels are almost only vessel, "everything is vessel" scores a perfect zero. Some
  known background must stay.</li>
</ul></div>"""))

The next cell defines a training loop that takes any loss, a scoring report, a view of what the loss
hears, and one colour for each head of this path.

In [ ]:
# @title Path 1 helpers: train with any loss, score a head, view the loss mask { display-mode: "form" }
approach_colours |= {
    "naive": "#BF360C",
    "masked": "#F9A825",
    "pu": "#6D4C41",
}
approach_names |= {
    "naive": "naive: missing = background",
    "masked": "masked: missing = unknown",
    "pu": "PU: missing = unlabelled",
}


def train_head_with(
    feats: list[torch.Tensor], labels: list[np.ndarray], loss_fn, steps: int, lr: float = 0.05
):
    """The linear head of `train_head`, trained with `loss_fn(logits, labels, ignore_mask)` on every pixel of
    every training image at once. Labels per pixel: 1 vessel, 0 background (or unlabelled), -1 ignored.
    Returns the head's tensors and the training seconds."""
    torch.manual_seed(SEED)
    allf = torch.cat([f.flatten(1) for f in feats], 1)
    mu, sd = allf.mean(1), allf.std(1) + 1e-6
    feats = [(f - mu.view(-1, 1, 1)) / sd.view(-1, 1, 1) for f in feats]
    lab = torch.cat([torch.from_numpy(np.asarray(x, dtype=np.int8)).flatten() for x in labels]).to(DEVICE)
    target, ignore = (lab == 1).float(), lab < 0
    shapes = [np.asarray(x).shape for x in labels]
    head = torch.nn.Conv2d(feats[0].shape[0], 1, 1).to(DEVICE)
    opt = torch.optim.Adam(head.parameters(), lr=lr)
    t0 = time.monotonic()
    for _ in range(steps):
        opt.zero_grad()
        logits = torch.cat(
            [
                F.interpolate(head(f[None]), size=s, mode="bilinear", align_corners=False).flatten()
                for f, s in zip(feats, shapes, strict=True)
            ]
        )
        loss_fn(logits, target, ignore).backward()
        opt.step()
    tensors = {
        "weight": head.weight.detach().cpu(),
        "bias": head.bias.detach().cpu(),
        "feat_mean": mu.cpu(),
        "feat_std": sd.cpu(),
    }
    return tensors, time.monotonic() - t0


def path_report(tensors: dict) -> dict:
    """Recall on the hidden and visible vessels of the training images (section 3) and the held-out
    numbers of section 4 (IoU, recall, precision, marked share), plus the predictions."""
    fit_preds = [
        predict_head(tensors, f, it["mask"].shape) for f, it in zip(miss_feats, miss_train, strict=True)
    ]
    held_preds = [
        predict_head(tensors, features_for(specialised, it, side), it["mask"].shape) for it in spec_eval
    ]
    out = {
        "recall_hidden_fit": part_recall(fit_preds, miss_train, "hidden"),
        "recall_visible_fit": part_recall(fit_preds, miss_train, "visible"),
    }
    return out | rung_report(held_preds) | {"fit_preds": fit_preds, "held_preds": held_preds}


def print_reports(reports: dict) -> None:
    """One line per head: on the training images, the share of the hidden and of the painted vessels it finds;
    on the held-out images, IoU, recall and the share of the fundus it calls vessel."""
    print(f"{'':<34}{'training images · found':^26}   {'held-out images':^30}")
    print(f"{'head':<34}{'hidden':>12}{'painted':>12}   {'IoU':>7}{'recall':>9}{'calls vessel':>14}")
    for name, r in reports.items():
        print(
            f"{name:<34}{r['recall_hidden_fit']:>12.0%}{r['recall_visible_fit']:>12.0%}   {r['iou']:>7.3f}"
            f"{r['recall']:>9.0%}{r['marked']:>14.0%}"
        )
    print(f"{'really vessel (the ground truth)':<34}{'':>24}   {'':>16}{fg_share:>14.0%}")


def public(r: dict) -> dict:
    """A report without its prediction arrays, for RESULTS."""
    return {k: v for k, v in r.items() if not k.endswith("_preds")}


def loss_view(item: dict, labels: np.ndarray, show_bg: bool = True) -> np.ndarray:
    """What the loss hears on one image: painted vessel in teal, background targets in grey, pixels the loss
    ignores (label -1) dark. Sparse labels are thickened for display only."""
    known = labels >= 0
    if known.mean() < 0.2:
        known = ndi.binary_dilation(known, iterations=4)
    out = np.where(known[..., None], item["image"], (item["image"] * 0.25).astype(np.uint8))
    if show_bg:
        bg = labels == 0
        if bg.mean() < 0.2:
            bg = ndi.binary_dilation(bg, iterations=4)
        out = overlay(out, bg, THEME["background"])
    return overlay(out, labels == 1, THEME["scribble"])

### 5.1 Three ways to read the same label file

The next cell shows what each recipe tells the loss on one of the five training images of section 3. The
painted vessels are the same every time; the rest differs:

- **Naive:** every unpainted pixel is a background target.
- **Masked:** unpainted pixels are ignored, except 3–5 background strokes per image (the annotator also
  drew a few strokes on clear background, as in section 4). The loss needs some known background.
- **PU:** every unpainted pixel counts, as "unlabelled".

What to look for: the dark areas of the third picture are free for the head to call vessel. No ground truth
goes in, only the painted vessels, the strokes and the fundus border.

In [ ]:
# @title What each recipe tells the loss { display-mode: "form" }
bg_strokes = [(it["scribbles"] == 2) & region(it) & ~it["visible"] for it in miss_train]
# masked: vessel where painted, background only on the strokes, everything else unknown (-1)
masked_labels = [
    np.where(it["visible"] & region(it), 1, np.where(bg, 0, -1)).astype(np.int8)
    for it, bg in zip(miss_train, bg_strokes, strict=True)
]
# PU: vessel where painted, "unlabelled" (0) everywhere else inside the fundus
pu_labels = [np.where(region(it), it["visible"], -1).astype(np.int8) for it in miss_train]
known = {
    "naive": np.mean([(lab >= 0).mean() for lab in naive_labels]),
    "masked": np.mean([(lab >= 0).mean() for lab in masked_labels]),
}
n_bg_strokes = sum(m.get("n_bg_strokes", 0) for m in specialised["fit"][: len(miss_train)])
print(
    f"pixels the loss hears about · naive {known['naive']:.1%} (all called background unless painted) · "
    f"masked {known['masked']:.1%} ({n_bg_strokes} background strokes over {len(miss_train)} images) · "
    "PU: all of them, as vessel or unlabelled"
)

ex, k_ex = miss_train[0], 0
show_masks(
    [
        (ex["name"], ex["image"]),
        ("naive: vessel / background", loss_view(ex, naive_labels[k_ex])),
        ("masked: vessel / strokes / ignored", loss_view(ex, masked_labels[k_ex])),
        ("PU: vessel / unlabelled", loss_view(ex, pu_labels[k_ex], show_bg=False)),
    ],
    width=2.8,
    legend=[
        ("painted vessel (target: vessel)", THEME["scribble"]),
        ("background target", THEME["background"]),
        ("ignored by the loss (dark)", "#404040"),
    ],
    note="PU panel: everything that is not teal counts as unlabelled, hidden vessels included.",
)

### 5.2 Write the masked loss

This cell holds our version of the one line you can write. Open it only if you get stuck.

In [ ]:
# @title Reference solution and play example (open only if stuck) { display-mode: "form" }
def _ref_known_pixels(ignore_mask: torch.Tensor) -> torch.Tensor:
    """Reference: a pixel is known when it is not ignored."""
    return ~ignore_mask


_play_logits = torch.tensor([[2.0, -1.0, 0.5], [-2.0, 3.0, 0.0]])
_play_labels = torch.tensor([[1.0, 0.0, 1.0], [0.0, 0.0, 1.0]])
_play_ignore = torch.tensor([[False, False, True], [False, True, False]])  # two unknown pixels


def _check_known(f):
    """What `use` compares: which pixels the function calls known on the play example."""
    return torch.as_tensor(f(_play_ignore)).int().tolist()


def show_masked_play(fn) -> None:
    """Print the loss `fn` gives on the 2 x 3 play example, with and without unknown pixels."""
    with_unknown, without = (
        float(fn(_play_logits, _play_labels, ig)) for ig in (_play_ignore, torch.zeros_like(_play_ignore))
    )
    print(f"play example · loss {with_unknown:.4f} with two pixels unknown, {without:.4f} with none")

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#FFE8D6;border-left:6px solid #7A2E00;border-radius:6px;padding:12px 14px;margin:6px 0;color:#7A2E00;font-family:inherit;line-height:1.5"><b>✏️ Your turn (optional, 1 min): one line of the masked loss</b><p>The masked loss below is the balanced loss of every head so far, with one change: ignored pixels count for
nothing. Everything is given except one line. Fill <code>known_pixels</code>: it gets <code>ignore_mask</code> (a bool tensor,
True = unknown) and returns a bool tensor of the same shape that is True where the label is <strong>known</strong>.</p>
<p>Return <code>torch.ones_like(ignore_mask)</code> and every pixel counts: that is the naive loss again.</p>
<p>The cell checks your line on a 2 × 3 play example and continues with ours if you skip it.</p></div>"""))

In [ ]:
# ✍️ YOUR CODE
def known_pixels(ignore_mask: torch.Tensor) -> torch.Tensor:
    """True where the label is known (the pixel is not ignored), same shape as `ignore_mask`.
    e.g. tensor([[True, True, False], [True, False, True]])"""
    ...


known_pixels_fn = use(known_pixels, reference=_ref_known_pixels, check=_check_known, name="known_pixels")

The masked loss itself, with your line (or ours) in the first row. The play example prints it with two
pixels unknown and with none.

In [ ]:
def masked_loss(logits: torch.Tensor, labels: torch.Tensor, ignore_mask: torch.Tensor) -> torch.Tensor:
    """Balanced cross-entropy over the known pixels: the mean loss on the known vessel pixels and the mean
    loss on the known background pixels, averaged. An empty group counts as 0."""
    known_mask = known_pixels_fn(ignore_mask)  # your line: which labels can be trusted
    bce = F.binary_cross_entropy_with_logits(logits, labels.float(), reduction="none")  # loss of each pixel
    pos, neg = known_mask & (labels == 1), known_mask & (labels == 0)  # known vessel, known background
    return 0.5 * bce[pos].sum() / pos.sum().clamp(min=1) + 0.5 * bce[neg].sum() / neg.sum().clamp(min=1)


show_masked_play(masked_loss)

This helper cell defines `train_masked_head`: it trains with the masked loss, or loads our checkpoint.

In [ ]:
# @title Masked-head helper: train with the masked loss, or load ours { display-mode: "form" }
def train_masked_head(train_yourself: bool):
    """The masked head of Path 1. Returns the head's tensors and the training seconds."""
    if not isinstance(train_yourself, bool):
        raise SystemExit("TRAIN_MASKED_HEAD must be True or False.")
    # the supervision of this path: painted vessels + background strokes; no hidden truth (L-005)
    if train_yourself:
        head, sec = train_head_with(miss_feats, masked_labels, masked_loss, HEAD_STEPS)
        src = f"trained here on {len(miss_train)} images, {HEAD_STEPS} steps, masked loss"
    else:
        head, entry = load_head(specialised, "masked", side)
        sec = entry["train_sec"]
        src = (
            f"our checkpoint {entry['file']} ({len(entry['train_names'])} images). ⚠️ It was told exactly "
            "which pixels next to the hidden vessels are unknown and that the rest is background: a rule "
            "that needs the hidden truth, so read its numbers as an upper bound for this path"
        )
    print(f"masked head @ {side} px · {src} · {sec:.1f} s on {DEVICE}")
    if sec > 60:
        print("ℹ️ Slow session? Set TRAIN_MASKED_HEAD = False above to load our checkpoint next time.")
    return head, sec

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F3E5F5;border-left:6px solid #4A148C;border-radius:6px;padding:12px 14px;margin:6px 0;color:#4A148C;font-family:inherit;line-height:1.5"><b>🤔 Your prediction</b><p>Same label file as the naive head (you saw its hidden-vessel recall in section 3), but the unpainted pixels
are now unknown. Write two guesses:</p>
<ul>
<li>On the hidden vessels of the training images, will its recall be closer to the naive head's or to the
  section-2 head's (full masks)?</li>
<li>On new images, will it mark more of the fundus than the naive head, or less?</li>
</ul></div>"""))

The next cell trains the head with the masked loss on the labels of 5.1. With `TRAIN_MASKED_HEAD = False`
it loads our checkpoint, made with a stricter rule that needs the hidden truth: its numbers are an upper
bound for this path.

In [ ]:
# @title Train the masked head (False = load our checkpoint) { display-mode: "form" }
TRAIN_MASKED_HEAD = True  # @param {type:"boolean"}
masked_head, masked_sec = train_masked_head(TRAIN_MASKED_HEAD)

The next cell scores the masked head next to the naive head and the full-mask head, on the hidden and
visible vessels of the training images and on the held-out images. "Marked" is the share of the fundus a
mask covers. What to look for: how much of the fundus each head marks, against the share that is really
vessel.

In [ ]:
# @title Score the masked head next to the earlier heads { display-mode: "form" }
t0 = time.monotonic()
masked = path_report(masked_head)
masked_infer = round(time.monotonic() - t0, 1)
path_reports = {
    approach_names["naive"]: path_report(naive_head),
    approach_names["masked"]: masked,
    "full masks (section 2)": path_report(full_head),
}
path_colours = {
    approach_names["naive"]: approach_colours["naive"],
    approach_names["masked"]: approach_colours["masked"],
    approach_names["pu"]: approach_colours["pu"],
    "full masks (section 2)": THEME["works"],
}

RESULTS["masked_recall_hidden_fg"] = masked["recall_hidden_fit"]  # same definition as section 3 (SPEC §9)
RESULTS["masked_head"] = public(masked) | {
    "long_side": side,
    "n_train": len(miss_train),
    "known_share": float(known["masked"]),
    "n_bg_strokes": int(n_bg_strokes),
    "trained_here": TRAIN_MASKED_HEAD,
    "train_sec": round(float(masked_sec), 2),
    "infer_sec": masked_infer,
}
print_reports(path_reports)
truth_share = float(np.mean([it["mask"][region(it)].mean() for it in spec_eval]))
print(
    f"the true vessels cover {truth_share:.1%} of the fundus · the masked head calls {masked['marked']:.1%} "
    f"of it vessel · the naive head {path_reports[approach_names['naive']]['marked']:.1%}"
)

# annotation effort of the three Path 1 heads, for the comparison table of section 8: the painted half of
# each vessel map (the dataset's share), plus seconds per background stroke
painted_masks = len(miss_train) * (1 - missing_labels["hidden_fraction"])
RESULTS["path1_effort_min"] = {
    "naive": round(effort_minutes(n_full_masks=painted_masks), 2),
    "masked": round(effort_minutes(n_strokes=n_bg_strokes, n_full_masks=painted_masks), 2),
    "pu": round(effort_minutes(n_full_masks=painted_masks), 2),
}
print(
    "annotation effort · "
    + " · ".join(
        f"{approach_names[k].split(':')[0]} {v:.0f} min" for k, v in RESULTS["path1_effort_min"].items()
    )
)

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#FDECEA;border-left:6px solid #8B1A10;border-radius:6px;padding:12px 14px;margin:6px 0;color:#8B1A10;font-family:inherit;line-height:1.5"><b>⚠️ Pitfall: &quot;ignore&quot; is not free</b><p>Remove the background strokes and the masked loss hears only "vessel": a head that paints the whole eye
scores a loss near zero. Watch how much of the fundus the head calls vessel, not the training loss.</p></div>"""))

In [ ]:
# @title A fake head that always answers "vessel" { display-mode: "form" }
_lab = torch.cat([torch.from_numpy(x.astype(np.int8)).flatten() for x in masked_labels])
_y, _ignored = (_lab == 1).float(), _lab < 0
_all_vessel = torch.full_like(_y, 8.0)  # a head that says "vessel" everywhere
_bce = F.binary_cross_entropy_with_logits(_all_vessel, _y, reduction="none")
print(
    "mean loss of 'vessel everywhere' on the labels the masked loss keeps: "
    f"{float(_bce[~_ignored].mean()):.3f} with the background strokes, "
    f"{float(_bce[~_ignored & (_lab == 1)].mean()):.5f} without them "
    "(≈ 0: a perfect score for a useless head)"
)

### ☕ Coffee break

End of block 1. The masked head found hidden vessels, and it marks too much.

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F3E5F5;border-left:6px solid #4A148C;border-radius:6px;padding:12px 14px;margin:6px 0;color:#4A148C;font-family:inherit;line-height:1.5"><b>❓ Question for the coffee break</b><p>No code, just think about it over coffee.</p>
<p><strong>Q10.</strong> 🟡 The masked head over-marks. The annotator says they painted about half of the vessels. How
could that one number help the training?</p>
<details>
<summary>A hint, for after the break</summary>
<ul>
<li>Half of the vessels are painted, so about as many again are hiding among the unpainted pixels.</li>
<li>If the loss knows roughly how many vessels to expect in that unpainted mix, it can stop punishing them,
  without any background strokes. That is 5.3.</li>
<li>Later in block 2: section 6 guesses the missing labels with SAM, section 7 skips training and prompts
  the foundation model.</li>
</ul>
</details></div>"""))

### 5.3 The PU loss

Welcome back. The masked loss threw away every unpainted pixel. The PU loss keeps them, as *unlabelled*: a
mix of background and the vessels nobody painted. The annotator tells us roughly what share of the vessels
they painted, here about half. Say the painted vessels cover 1.5 % of the fundus. Then about 3 % of the
fundus is vessel, and the other 1.5 % hides among the unlabelled pixels. The loss counts every unpainted
pixel as background, then subtracts that 1.5 % again, so it stops punishing the hidden vessels (and fixes
the wrong proportion of the naive head on the way). The cell below prints the real painted share and the
vessel share it implies.

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#EDE7F6;border-left:6px solid #311B92;border-radius:6px;padding:12px 14px;margin:6px 0;color:#311B92;font-family:inherit;line-height:1.5"><b>📘 Theory: the PU loss in three steps</b><p>The <strong>prior</strong> π is the share of fundus pixels that are vessel: (painted share of the fundus) ÷ (share of the
vessels the annotator painted). Example: painted 1.5 %, annotator did half, so π = 3 %.</p>
<ol>
<li>Vessel loss on the painted pixels, target "vessel": <code>L_v = mean_painted(−log p)</code>.</li>
<li>Background loss. Calling every pixel "background" mixes π vessels with (1−π) background, so subtract
   the vessels: <code>L_b = ( mean_all(−log(1−p)) − π · mean_painted(−log(1−p)) ) / (1−π)</code>.</li>
<li>Balance them: <code>L = ½·L_v + ½·max(L_b, 0)</code>. The max stops L_b dipping below zero when the head
   overfits the few painted vessels.</li>
</ol>
<p>It assumes the painted vessels are a random sample of all vessels: true here, because we hid segments at
random.</p></div>"""))

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F5F5F5;border-left:6px solid #616161;border-radius:6px;padding:12px 14px;margin:6px 0;color:#616161;font-family:inherit;line-height:1.5"><details><summary><b>🔍 Go deeper: where the PU loss comes from</b></summary><ul>
<li>Elkan &amp; Noto, KDD 2008: with positives drawn at random from all positives, plus their share, you can
  estimate the loss you would get with complete labels.</li>
<li>Kiryo et al., NeurIPS 2017, <a style="color:inherit;text-decoration:underline" href="https://arxiv.org/abs/1703.00593">arXiv:1703.00593</a>: the clipping at zero in
  step 3. We use it on a single pixel classifier, so we call ours PU-<em>inspired</em>.</li>
</ul></details></div>"""))

This cell holds our version of step 2. Open it only if you get stuck.

In [ ]:
# @title Reference solution for step 2 (open only if stuck) { display-mode: "form" }
def _ref_pu_background(all_as_bg: torch.Tensor, painted_as_bg: torch.Tensor, prior: float) -> torch.Tensor:
    """Reference: the background loss with the expected vessels subtracted."""
    return (all_as_bg - prior * painted_as_bg) / (1 - prior)


def _check_pu_background(f):
    """What `use` compares: step 2 on two made-up pairs of numbers."""
    pairs = ((0.9, 0.4, 0.2), (0.5, 0.5, 0.3))
    return [round(float(f(torch.tensor(a), torch.tensor(b), p)), 4) for a, b, p in pairs]

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#FFE8D6;border-left:6px solid #7A2E00;border-radius:6px;padding:12px 14px;margin:6px 0;color:#7A2E00;font-family:inherit;line-height:1.5"><b>🔴 Optional challenge (2 min): one line of the PU loss</b><p>Fill <code>pu_background</code>, step 2 above. In: <code>all_as_bg</code> (the mean loss of calling every pixel background),
<code>painted_as_bg</code> (the same on the painted vessels only) and <code>prior</code> (the vessel share π). Out: the
background loss <code>L_b</code>, one number. Skip it and the notebook continues with ours.</p></div>"""))

In [ ]:
# ✍️ YOUR CODE
def pu_background(all_as_bg: torch.Tensor, painted_as_bg: torch.Tensor, prior: float) -> torch.Tensor:
    """Background loss: every pixel as background, minus the expected vessels, rescaled by (1 - prior).
    e.g. tensor(0.7500)"""
    ...


pu_background_fn = use(
    pu_background, reference=_ref_pu_background, check=_check_pu_background, name="pu_background"
)

The PU loss, with your step 2 (or ours). The play example prints its value on six made-up pixels.

In [ ]:
def pu_loss(
    logits: torch.Tensor, labels: torch.Tensor, ignore_mask: torch.Tensor, prior: float
) -> torch.Tensor:
    """Balanced non-negative PU loss: labels 1 = painted vessel, 0 = unlabelled; `prior` = vessel share."""
    z, y = logits[~ignore_mask], labels[~ignore_mask]
    painted = z[y == 1]
    vessel_loss = F.softplus(-painted).mean()  # (1) painted vessels, target "vessel"
    all_as_bg = F.softplus(z).mean()  # every pixel, target "background"
    painted_as_bg = F.softplus(painted).mean()  # painted vessels, target "background"
    bg_loss = pu_background_fn(all_as_bg, painted_as_bg, prior)  # (2) the true background, estimated
    return 0.5 * vessel_loss + 0.5 * torch.as_tensor(bg_loss).clamp(min=0)  # (3) balanced, clipped at zero


_pu_logits = torch.tensor([2.0, -1.0, 0.5, -2.0, 3.0, 0.0])
_pu_labels = torch.tensor([1.0, 0.0, 0.0, 0.0, 1.0, 0.0])  # two painted vessels, four unlabelled
print(f"play example · PU loss {float(pu_loss(_pu_logits, _pu_labels, torch.zeros(6, dtype=bool), 0.3)):.4f}")

This helper cell defines `train_pu_head` and `recipe_comparison`, the 2 × 3 figure of the three heads.

In [ ]:
# @title PU helpers: train the head, compare the three recipes { display-mode: "form" }
BARS = {
    "recall_hidden_fit": "hidden vessels\n(training images)",
    "recall": "recall\n(held-out)",
    "precision": "precision\n(held-out)",
}


def path_chart(names: list[str] | None = None) -> None:
    """Grouped bars: recall on the hidden training vessels, held-out recall and precision, per head."""
    names = names or [approach_names[k] for k in ("naive", "masked", "pu")] + ["full masks (section 2)"]
    x, width = np.arange(len(BARS)), 0.8 / len(names)
    fig, ax = plt.subplots(figsize=(7.2, 3.2))
    for j, name in enumerate(names):
        shift = (j - (len(names) - 1) / 2) * width
        ax.bar(x + shift, [path_reports[name][b] for b in BARS], width, color=path_colours[name], label=name)
    ax.set_xticks(x, list(BARS.values()), fontsize=8)
    ax.set_ylim(0, 1)
    ax.set_ylabel("share, 0 to 1", fontsize=8)
    ax.set_title("Path 1: what each head finds,\nand how much of what it marks is right")
    ax.legend(fontsize=7, frameon=False, loc="upper left", bbox_to_anchor=(1, 1))
    plt.tight_layout()
    plt.show()


def hidden_error_map(item: dict, pred: np.ndarray) -> np.ndarray:
    """A training image scored on what the label file did not say: hidden vessels found (violet) or missed
    (green), background called vessel (orange); the painted vessels are not scored here (light grey)."""
    r = region(item)
    painted, hidden = item["visible"] & r, item["hidden"] & r
    out = overlay(item["image"], pred & hidden, THEME["works"], unscored=~r | painted)
    out = overlay(out, pred & ~item["mask"] & r, THEME["fails"])
    return overlay(out, hidden & ~pred, THEME["truth"])


def recipe_comparison() -> None:
    """2 x 4 figure. Column 1: what the heads were given (the label file of training image 0) and the truth of
    the held-out demo image. Then one column per head (naive, masked, PU). Row 1: training image 0, scored on
    the hidden vessels and the background. Row 2: the held-out demo image. Stored predictions; no training."""
    k_demo = next(i for i, e in enumerate(spec_eval) if e is demo)
    train, r_demo = miss_train[0], region(demo)
    fig, axes = plt.subplots(2, 4, figsize=(12.4, 7.0))
    axes[0, 0].imshow(overlay(train["image"], train["visible"] & region(train), THEME["scribble"]))
    axes[0, 0].set_title("training image · the label file\n(painted vessels only)", fontsize=9)
    tag_panel(axes[0, 0], "input")
    axes[1, 0].imshow(overlay(demo["image"], demo["mask"], THEME["truth"], unscored=~r_demo))
    axes[1, 0].set_title(f"held-out image · ground truth\nreally vessel: {demo['mask'][r_demo].mean():.0%}",
                         fontsize=9)
    tag_panel(axes[1, 0], "truth")
    lab = {"naive": "naive", "masked": "masked", "pu": "PU"}.get
    for col, key in enumerate(("naive", "masked", "pu"), start=1):
        report = path_reports[approach_names[key]]
        fit, held = report["fit_preds"][0], report["held_preds"][k_demo]
        r_fit = region(train)
        bg_called = (fit & ~train["mask"] & r_fit).sum() / max((~train["mask"] & r_fit).sum(), 1)
        axes[0, col].imshow(hidden_error_map(train, fit))
        axes[0, col].set_title(
            f"{lab(key)} head · training image\nhidden vessels found "
            f"{hidden_recall(fit, train['hidden'] & r_fit):.0%}\nbackground called vessel {bg_called:.0%}",
            fontsize=8,
        )
        tag_panel(axes[0, col], "errors")
        axes[1, col].imshow(error_map(demo["image"], held, demo))
        axes[1, col].set_title(
            f"{lab(key)} head · held-out image\nIoU {score(held, demo)['iou']:.2f} · recall "
            f"{hidden_recall(held, demo['mask'] & r_demo):.0%}\ncalls {held[r_demo].mean():.0%} vessel "
            f"(really {demo['mask'][r_demo].mean():.0%})",
            fontsize=8,
        )
        tag_panel(axes[1, col], "errors")
    for ax in axes.ravel():
        if not ax.texts:  # untagged panels only
            ax.axis("off")
    add_key(
        fig,
        [
            ("painted vessel: in the label file (input)", THEME["scribble"]),
            ("ground truth (held-out image)", THEME["truth"]),
            "found",
            "invented",
            "missed",
            ("not scored (outside the fundus; row 1 also the painted vessels)", "#E0E0E0"),
        ],
        note="Row 1 asks what each head did where the label file was silent; row 2 asks how it does on a new "
        "image.",
    )
    plt.show()


def train_pu_head(labelled_share: float):
    """Prior = painted share ÷ `labelled_share`; trains the head on the PU labels of 5.1 and adds it to
    `path_reports`. Returns the head's tensors."""
    if not isinstance(labelled_share, (int, float)) or not 0 < labelled_share <= 1:
        raise SystemExit("LABELLED_SHARE must be a number above 0 and at most 1, e.g. 0.5.")
    painted_share = sum(int((lab == 1).sum()) for lab in pu_labels) / sum(
        int((lab >= 0).sum()) for lab in pu_labels
    )
    prior = painted_share / labelled_share
    if not 0 < prior < 1:
        raise SystemExit(
            f"painted share {painted_share:.3f} ÷ LABELLED_SHARE {labelled_share} gives a prior of "
            f"{prior:.2f}, which must lie between 0 and 1: choose a larger LABELLED_SHARE."
        )
    head, sec = train_head_with(miss_feats, pu_labels, lambda z, y, ig: pu_loss(z, y, ig, prior), HEAD_STEPS)
    t0 = time.monotonic()
    path_reports[approach_names["pu"]] = pu = path_report(head)
    RESULTS["pu_recall_hidden_fg"] = pu["recall_hidden_fit"]
    RESULTS["pu_head"] = public(pu) | {
        "long_side": side,
        "n_train": len(miss_train),
        "labelled_share": float(labelled_share),
        "prior": float(prior),
        "train_sec": round(float(sec), 2),
        "infer_sec": round(time.monotonic() - t0, 1),
    }
    print(
        f"PU head · painted {painted_share:.1%} of the fundus ÷ labelled share {labelled_share:.2f} → "
        f"{prior:.1%} of the fundus is vessel (the prior) · trained in {sec:.1f} s on {DEVICE}"
    )
    print_reports({k: path_reports[approach_names[k]] for k in ("naive", "masked", "pu")})
    return head

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F3E5F5;border-left:6px solid #4A148C;border-radius:6px;padding:12px 14px;margin:6px 0;color:#4A148C;font-family:inherit;line-height:1.5"><b>🤔 Your prediction</b><p>The annotator says "I painted about half". Suppose they had said "a third" instead. Will the PU head mark
more or less of the fundus than with "half"? Write one sentence why.</p></div>"""))

The next cell trains the PU head (seconds) with `LABELLED_SHARE`, the share of the vessels the annotator
painted, then draws two pictures: the comparison of the three heads, and one chart with their recall
and precision.

How to read the comparison. The first column is the reference: on top, the label file the heads were
trained on (only the painted vessels); below, the true vessels of a held-out image. Then one column per
head. Row 1 scores each head on the training image, only where the label file was silent: a hidden vessel
it found is $\color{#6A1B9A}{\blacksquare}$ violet, one it missed $\color{#2E7D32}{\blacksquare}$ green, and
background it called vessel $\color{#E65100}{\blacksquare}$ orange. Row 2 is the same head on the held-out
image. What to look for: who finds the hidden vessels in row 1, and how much
$\color{#E65100}{\blacksquare}$ orange that costs in both rows.

In [ ]:
# @title Train the PU head, then compare the three recipes { display-mode: "form" }
# the share of the vessels the annotator painted (the dataset hides half of them: 0.5); try 0.3, 0.8
LABELLED_SHARE = 0.5  # @param {type:"slider", min:0.2, max:1, step:0.05}
pu_head = train_pu_head(LABELLED_SHARE)
recipe_comparison()
path_chart()

**Q11.** 🟡 Read the figure column by column. For each head, how many of the hidden vessels did it find (row
1), and what did it pay for that on the held-out image (row 2)? Which mistake would cost more in your own
problem: missing vessels, or inventing them?

<details><summary>What to look for</summary>

- **Naive:** finds few hidden vessels. It was told they are background, so little
  $\color{#E65100}{\blacksquare}$ orange, much $\color{#2E7D32}{\blacksquare}$ green.
- **Masked:** finds most of them, but nothing told it where a vessel ends, so the area around each vessel
  often turns $\color{#E65100}{\blacksquare}$ orange.
- **PU:** it hears about every pixel, so it lands between the two. Where exactly depends on the run: read
  your numbers.

</details>

**Q12.** 🟡 Compare with your prediction. Set `LABELLED_SHARE` to 0.3, run the cell, then to 0.8 and run it
again. Which way does the head move on recall, and on the marked share?

<details><summary>What to look for</summary>

- A smaller `LABELLED_SHARE` claims more hidden vessels (a larger prior), and the head marks more.
- At 1.0 it claims nothing is hidden, and moves back towards the naive head.
- A wrong guess moves the head in a predictable direction instead of breaking it. With a small fully
  labelled check set you can pick the share that matches it.

</details>

**Q13.** 🟡 The masked head rests on one assumption: the hidden vessels look like the painted ones, because
we hid segments at random. Name a real annotation habit that breaks it. What would the masked head learn
then?

<details><summary>A possible answer</summary>

Annotators skip what is hard or tedious: the thinnest capillaries, the low-contrast periphery, the
vessels inside a bleeding. Then the painted vessels are thick and clear, and the unpainted ones are thin
and faint.

- **Masked head:** never hears "background" on the thin vessels, but never hears "vessel" either. It learns
  "vessel = thick and clear" and still misses most of them.
- **PU head:** worse off. Its correction assumes the hidden vessels look like the painted ones, so it
  pushes the thin ones towards background (the PU literature calls this breaking "selected completely at
  random").
- **Fix:** keep a small fully labelled set and check the recall on the kind of structure your annotators
  skip (the ⚠️ in section 3).

</details>

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#E0F2F1;border-left:6px solid #004D40;border-radius:6px;padding:12px 14px;margin:6px 0;color:#004D40;font-family:inherit;line-height:1.5"><b>✅ Takeaway</b><ul>
<li><strong>A label file where unpainted = background</strong> → mark the unpainted pixels "ignore" (−1), so the hidden
  vessels stop being punished.</li>
<li><strong>The loss ignores almost everything</strong> → keep some known background close to the objects, and watch the
  "marked" share, not the training loss: hidden vessels come back, but the
  <span style="color:#E65100">■</span> orange area grows.</li>
<li><strong>Only positives are labelled and you can estimate their share</strong> → use the PU loss, and sweep
  <code>LABELLED_SHARE</code> to see how much a wrong guess moves recall.</li>
<li><strong>Annotators skip the thin or faint cases</strong> → neither fix works and none adds information about the
  hidden vessels: keep a small fully labelled check set, or try a second model (section 6).</li>
</ul></div>"""))

## 6. Path 2 · Recover supervision

Path 1 kept the label file and changed the loss. Path 2 keeps the loss and fills in the labels.

A second model, the **teacher** (the model whose output we use as labels), guesses every pixel nobody
painted, and the head trains on those guesses as if a person had drawn them. A guess used this way is a
**pseudo-label**. It is like handing your half-done colouring-in to a fast, confident colleague: the page
comes back full, and every confident mistake is now in your training set.

We start from the scribbles of section 4 (a few strokes per image, no more annotation than rung 2). Two
teachers get the job: **SAM**, prompted with clicks taken from the strokes, and **Grad-CAM**, the heat
map of a small classifier that only learnt whether a tile contains a vessel.

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#E8F5E9;border-left:6px solid #1B5E20;border-radius:6px;padding:12px 14px;margin:6px 0;color:#1B5E20;font-family:inherit;line-height:1.5"><b>🎯 Objective</b><p>Turn SAM pseudo-labels and Grad-CAM maps into training signal, and know when to distrust them.</p></div>"""))

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#EDE7F6;border-left:6px solid #311B92;border-radius:6px;padding:12px 14px;margin:6px 0;color:#311B92;font-family:inherit;line-height:1.5"><b>📘 Theory: when pseudo-labels work</b><p>A pseudo-label works when the teacher's guesses are mostly right <em>and</em> its mistakes are not systematic.</p>
<ul>
<li>Pets: SAM's mask of a cat is a fine pseudo-label.</li>
<li>Retina: SAM's "vessel" mask is a fat region, and a head trained on it learns the region, not the line.</li>
<li>Guard: keep only the guesses the teacher is sure of and leave the rest unlabelled (6.2). A person's
  strokes always overrule the teacher.</li>
</ul></div>"""))

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F5F5F5;border-left:6px solid #616161;border-radius:6px;padding:12px 14px;margin:6px 0;color:#616161;font-family:inherit;line-height:1.5"><details><summary><b>🔍 Pseudo-label papers and confirmation bias</b></summary><p>A model trained on its teacher's errors learns to repeat them. The literature calls this <em>confirmation
bias</em>. The usual guard is a confidence threshold.</p>
<ul>
<li>Lee, "Pseudo-Label", ICML 2013 workshop: the original recipe.</li>
<li>FixMatch, Sohn et al., NeurIPS 2020: <a style="color:inherit;text-decoration:underline" href="https://arxiv.org/abs/2001.07685">arXiv:2001.07685</a></li>
</ul></details></div>"""))

This helper cell adds the colours of this path, a probability colour scale and the stroke-click sampler.

In [ ]:
# @title Path 2 helpers: colours, probability scale, clicks along the strokes { display-mode: "form" }
approach_colours |= {
    "sam_pseudo": "#0277BD",
    "gradcam_pseudo": "#827717",
}
approach_names |= {
    "sam_pseudo": "pseudo-labels: SAM from stroke clicks",
    "gradcam_pseudo": "pseudo-labels: Grad-CAM of a tile classifier",
}

# every map of this section uses the shared score scale of section 0: blue = 1, white = 0.5, dark grey = 0
# (blue, so a score map never shares a colour with the violet "found" of the error maps)
prob_key = [
    ("score map: 1 = sure it is vessel", THEME["score"]),
    ("score map: 0.5 = unsure", "white"),
    ("score map: 0 = sure it is not", "#424242"),
]
error_key = ["found", "invented", "missed"]  # the error-map legend: violet, orange, green


def prob_rgb(prob: np.ndarray) -> np.ndarray:
    """A probability map as RGB on the shared score scale: blue = 1, white = 0.5 (unsure), dark grey = 0."""
    return (score_cmap(np.clip(prob, 0, 1))[..., :3] * 255).astype(np.uint8)


def stroke_points(item: dict, value: int, n: int) -> list[list[float]]:
    """n (x, y) clicks spread along the strokes of one kind (1 vessel, 2 background) by farthest-point
    sampling from the first stroke pixel: no randomness, no ground truth."""
    ys, xs = np.nonzero(item["scribbles"] == value)
    if len(ys) == 0:
        return []
    pts = np.stack([xs, ys], 1).astype(float)
    chosen = [0]
    dist = np.linalg.norm(pts - pts[0], axis=1)
    while len(chosen) < min(n, len(pts)):
        k = int(dist.argmax())
        chosen.append(k)
        dist = np.minimum(dist, np.linalg.norm(pts - pts[k], axis=1))
    return [[float(x), float(y)] for x, y in pts[chosen]]


def pseudo_numbers(masks: list[np.ndarray], items: list[dict]) -> dict:
    """Diagnosis only (uses the hidden truth): mean IoU, recall (share of the true vessels the mask covers)
    and marked (share of the fundus the mask covers) of pseudo-label masks, inside the fundus."""
    rows = []
    for m, it in zip(masks, items, strict=True):
        r = region(it)
        rows.append(
            {
                "iou": score(m, it)["iou"],
                "recall": hidden_recall(m & r, it["mask"] & r),
                "marked": float((m & r)[r].mean()),
            }
        )
    return {k: float(np.mean([row[k] for row in rows])) for k in rows[0]}

### 6.1 SAM fills in the gaps

We take the scribbles of section 4 again: a few vessel strokes and a few background strokes on each of the
same five training images.

- Twenty clicks spread along the vessel strokes go in as "vessel", twenty along the background strokes as
  "not vessel".
- SAM's answer for every pixel is the pseudo-label.
- Only the strokes go in. The ground truth appears in the figure so you can judge the guesses, and never
  in training.

This helper cell defines `sam_pseudo_labels`, which asks SAM for probability maps (or loads ours).

In [ ]:
# @title SAM pseudo-label helper: probability maps from the stroke clicks { display-mode: "form" }
def sam_pseudo_labels(run_sam: bool, n_clicks: int):
    """SAM's probability map for each training image from `n_clicks` + `n_clicks` clicks on the strokes
    (or our precomputed maps). Prints the numbers and draws one image. Returns the maps, the clicks, the
    seconds and SAM's quality at the 0.5 cut (IoU, recall, marked)."""
    if not isinstance(run_sam, bool):
        raise SystemExit("RUN_SAM must be True or False.")
    t0 = time.monotonic()
    probs, clicks_used = [], []  # per training image: SAM's probability map, (vessel, background) clicks
    if run_sam:
        for it in tqdm(scrib_train, desc="SAM on the training images"):
            pos, neg = stroke_points(it, 1, n_clicks), stroke_points(it, 2, n_clicks)
            labels = [1] * len(pos) + [0] * len(neg)
            probs.append(sam.probability(it["image"], points=pos + neg, labels=labels))
            clicks_used.append((pos, neg))
        src = f"SAM 2.1 run here, {n_clicks} + {n_clicks} clicks per image"
    else:
        ours = PRECOMPUTED / "pseudo_labels" / "specialised"
        if not (ours / "prompts.json").is_file():
            raise SystemExit(f"{ours} is missing. Run setup_check.ipynb first, or set RUN_SAM = True.")
        prompts = json.loads((ours / "prompts.json").read_text())["prompts"]
        for it in scrib_train:
            probs.append(read_png(ours / f"{it['name']}.png") / 255)
            p = prompts[it["name"]]
            pairs = list(zip(p["points_xy"], p["labels"], strict=True))
            clicks_used.append(tuple([xy for xy, lab in pairs if lab == k] for k in (1, 0)))
        src = "our precomputed maps (same recipe, clicks drawn at random from the strokes)"
    sec = round(time.monotonic() - t0, 1)

    # diagnosis only: SAM's guesses at the usual 0.5 cut, scored against the hidden truth (never for training)
    quality = pseudo_numbers([p >= 0.5 for p in probs], scrib_train)
    # how sure SAM is, inside the fundus (no truth needed): the spread of its probabilities
    spread = np.quantile(
        np.concatenate([p[region(it)] for p, it in zip(probs, scrib_train, strict=True)]), [0.05, 0.5, 0.95]
    )
    train_truth = float(np.mean([it["mask"][region(it)].mean() for it in scrib_train]))
    print(
        f"SAM pseudo-labels on {len(scrib_train)} training images · {src} · {sec} s on {DEVICE}\n"
        f"how sure SAM is: 90 % of the fundus pixels get a probability between {spread[0]:.2f} and "
        f"{spread[2]:.2f} (middle value {spread[1]:.2f})\n"
        f"checked against the truth (never used for training), cutting at 0.5: the pseudo-labels "
        f"{called_vessel(quality['marked'], train_truth)} · recall {quality['recall']:.2f} · "
        f"IoU {quality['iou']:.3f}"
    )
    scrib_ex, (pos, neg) = scrib_train[0], clicks_used[0]
    fig, axes = plt.subplots(1, 3, figsize=(10.2, 3.6))
    axes[0].imshow(scrib_ex["image"])
    for (x, y), kind in [(xy, 1) for xy in pos] + [(xy, 0) for xy in neg]:
        if kind:
            axes[0].plot(x, y, marker="*", ms=10, color=THEME["scribble"], mec="white", ls="")
        else:
            axes[0].plot(x, y, marker="X", ms=8, color="#424242", mec="white", ls="")
    axes[0].set_title(
        f"{scrib_ex['name']} · what SAM gets:\n{len(pos)} vessel + {len(neg)} background clicks", fontsize=9
    )
    im = axes[1].imshow(np.clip(probs[0], 0, 1), cmap=score_cmap, vmin=0, vmax=1)
    axes[1].set_title("what SAM answers: probability of vessel\n(pale = unsure, near 0.5)", fontsize=9)
    fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.02, ticks=[0, 0.5, 1]).ax.tick_params(labelsize=7)
    axes[2].imshow(error_map(scrib_ex["image"], probs[0] >= 0.5, scrib_ex))
    axes[2].set_title("pseudo-label (probability ≥ 0.5)\ncompared with the ground truth", fontsize=9)
    for ax, role in zip(axes, ("input", "map", "errors"), strict=True):
        ax.axis("off")
        tag_panel(ax, role)
    add_key(
        fig,
        [("click", "vessel click"), ("background click (X)", "#424242"), *error_key],
        note="The colour bar reads the middle panel: blue = sure it is vessel, white = unsure, "
        "dark grey = sure it is not.",
    )
    plt.show()
    return probs, clicks_used, sec, quality

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F3E5F5;border-left:6px solid #4A148C;border-radius:6px;padding:12px 14px;margin:6px 0;color:#4A148C;font-family:inherit;line-height:1.5"><b>🤔 Your prediction</b><p>Section 2 showed what SAM does with clicks on a vessel tree. Now it gets forty clicks per image, half of
them saying "not here". Will its mask follow the vessels, or cover a region around them? Write one guess.</p></div>"""))

The next cell draws three panels, left to right: what SAM gets (the photo and the clicks), what it answers
(a probability per pixel), and the pseudo-label you get by cutting that probability at 0.5, checked
against the truth. The printout says the same in numbers:

- **How sure SAM is:** 90 % of the pixels fall between the two printed values. Values near 0 or 1 mean
  sure, near 0.5 unsure.
- **Calls … vessel:** how much of the fundus the pseudo-label calls vessel, next to how much really is.
- **Colours in the last panel:** $\color{#6A1B9A}{\blacksquare}$ violet = a real vessel found,
  $\color{#E65100}{\blacksquare}$ orange = a vessel SAM invented, $\color{#2E7D32}{\blacksquare}$ green = a
  vessel SAM left out. Every $\color{#E65100}{\blacksquare}$ orange and
  $\color{#2E7D32}{\blacksquare}$ green pixel becomes a wrong training label if we take the guesses as
  they are.

This cell asks SAM for a probability map on each training image from the stroke clicks and shows one image
(seconds per image on a T4, much slower on a CPU, where `RUN_SAM = False` loads our maps, made the same way
with clicks drawn at random).

In [ ]:
# @title Run SAM on the stroke clicks to get pseudo-labels (False = load our maps) { display-mode: "form" }
RUN_SAM = True  # @param {type:"boolean"}
N_PSEUDO_CLICKS = 20  # @param {type:"integer"}
pseudo_probs, pseudo_clicks, pseudo_sec, sam_quality = sam_pseudo_labels(RUN_SAM, N_PSEUDO_CLICKS)
scrib_ex = scrib_train[0]

**Q14.** 🟡 Look at the error map and the probability panel. Where does SAM's guess go wrong: around
the thick vessels, along the thin ones, in the background between the strokes? Is SAM sure of itself (deep
$\color{#1565C0}{\blacksquare}$ blue, $\color{#424242}{\blacksquare}$ dark grey) or unsure (pale)? What
does that mean for a cut at exactly 0.5?

<details><summary>What to look for</summary>

- SAM looks for one object that contains the positive clicks (section 2.3). Forty clicks on a vessel tree
  do not describe one object, so SAM settles for the object that holds most of them, often the whole
  fundus. Its edge is careful, but the inside is a region ($\color{#E65100}{\blacksquare}$ orange), and
  thin branches between the clicks can
  drop out ($\color{#2E7D32}{\blacksquare}$ green).
- If the probability panel is mostly pale, the values sit just around 0.5 and the exact cut decides almost
  everything: a little below and the whole fundus is "vessel", a little above and nothing is. The crisp
  mask hides that doubt.
- Whatever you see on your run, the threshold is your choice, not a property of SAM.

</details>

### 6.2 From a probability map to training targets

A **target** is the label the head trains on: 1 vessel, 0 background, −1 ignored (the loss skips it). Two
numbers turn SAM's probability into a target: the **cut** and the **unsure band**. With a cut of 0.5 and a
band of 0.1:

1. A pixel at 0.9 is above 0.6, so it becomes vessel (1).
2. A pixel at 0.55 is within 0.1 of the cut, so it is ignored (−1).
3. A pixel at 0.3 is below 0.4, so it becomes background (0).

Two rules sit on top: a person's stroke always wins over SAM, and pixels outside the fundus are always −1.

This cell draws the three zones for a cut of 0.5 and a band of 0.1.

In [ ]:
# @title The cut and the unsure band, drawn { display-mode: "form" }
def draw_cut_and_band(cut: float, band: float) -> None:
    """One bar over SAM's probability axis, split into the three target zones."""
    zones = [
        (0, cut - band, THEME["background"], "0 · background"),
        (cut - band, cut + band, "white", "−1 · ignored"),
        (cut + band, 1, THEME["works"], "1 · vessel"),
    ]
    fig, ax = plt.subplots(figsize=(7.2, 1.6))
    for left, right, colour, label in zones:
        ax.barh(0, right - left, left=left, color=colour, edgecolor="black", height=0.5)
        ax.text(
            (left + right) / 2, 0, label, ha="center", va="center", fontsize=8,
            color="black" if colour == "white" else "white",
        )
    ax.set_xlim(0, 1)
    ax.set_yticks([])
    ax.set_xlabel("SAM's probability for a pixel")
    ax.set_title(
        f"cut {cut}, band {band}: below {cut - band:g} a pixel becomes background,\n"
        f"from {cut + band:g} vessel, and in between it is ignored",
        fontsize=9,
    )
    plt.tight_layout()
    plt.show()


draw_cut_and_band(0.5, 0.1)

This helper cell defines `pseudo_targets`, the rule above in code. Open it if you want to read it.

In [ ]:
# @title pseudo_targets: the cut, the band and the two rules in code { display-mode: "form" }
def pseudo_targets(
    prob: np.ndarray, scribbles: np.ndarray, region: np.ndarray, threshold: float = 0.5, band: float = 0.0
) -> np.ndarray:
    """Targets (int8) from a pseudo-label probability map: 1 where prob >= threshold + band, 0 where
    prob < threshold - band, -1 in between. Strokes overrule the map (1 vessel stroke -> 1, 2 background
    stroke -> 0); outside `region` -> -1.
    e.g. threshold 0.5, band 0.1, no strokes: prob [[0.9, 0.55, 0.3, 0.05]] -> [[1, -1, 0, 0]]"""
    targets = np.full(prob.shape, -1, np.int8)
    targets[prob >= threshold + band] = 1
    targets[prob < threshold - band] = 0
    targets[scribbles == 1] = 1
    targets[scribbles == 2] = 0
    targets[~np.asarray(region, bool)] = -1
    return targets

This helper cell defines `train_sam_head`, which makes targets at the 0.5 cut and trains or loads a head.

In [ ]:
# @title SAM-head helper: pseudo-label targets, then train or load { display-mode: "form" }
def train_sam_head(train_yourself: bool):
    """The head trained on SAM's pseudo-labels at threshold 0.5, no band. Returns the head's tensors, the
    training seconds and the targets."""
    if not isinstance(train_yourself, bool):
        raise SystemExit("TRAIN_SAM_HEAD must be True or False.")
    # the supervision of this path: the strokes, and SAM's guesses from clicks on the strokes (L-005)
    targets = [
        pseudo_targets(p, it["scribbles"], region(it), 0.5, 0.0)
        for p, it in zip(pseudo_probs, scrib_train, strict=True)
    ]
    if train_yourself:
        head, sec = train_head(scrib_feats, targets, HEAD_STEPS)
        src = f"trained here on {len(scrib_train)} images, {HEAD_STEPS} steps, SAM pseudo-labels at 0.5"
    else:
        head, entry = load_head(specialised, "pseudo", side)
        sec = entry["train_sec"]
        src = f"our checkpoint {entry['file']} ({len(entry['train_names'])} images)"
    print(f"SAM pseudo-label head @ {side} px · {src} · {sec:.1f} s on {DEVICE}")
    if sec > 60:
        print("ℹ️ Slow session? Set TRAIN_SAM_HEAD = False above to load our checkpoint next time.")
    return head, sec, targets

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F3E5F5;border-left:6px solid #4A148C;border-radius:6px;padding:12px 14px;margin:6px 0;color:#4A148C;font-family:inherit;line-height:1.5"><b>🤔 Your prediction</b><p>The scribble head of section 4 heard about the strokes only. This head hears about every pixel of the
fundus, most of it in SAM's words. Will it mark more of the held-out fundus than the scribble head, or
less? Will its IoU be higher? Write two guesses.</p></div>"""))

This cell turns SAM's maps into targets at the usual cut (threshold 0.5, no band) and trains the section-2
head on them (`TRAIN_SAM_HEAD = False` loads our "pseudo" checkpoint instead, trained the same way).

In [ ]:
# @title Train the SAM pseudo-label head (False = load our checkpoint) { display-mode: "form" }
TRAIN_SAM_HEAD = True  # @param {type:"boolean"}
sam_head, sam_sec, sam_targets = train_sam_head(TRAIN_SAM_HEAD)

This cell scores the new head on the held-out images and prints it under the scribble head, which saw the
same strokes but kept every other pixel unknown. Columns: **IoU** (overlap with the truth), **recall** (the
share of the true vessels it finds) and **calls vessel** (the share of the fundus it calls vessel; the last
line says how much really is).

In [ ]:
# @title Score the SAM pseudo-label head on the held-out images { display-mode: "form" }
t0 = time.monotonic()
path2_preds = {
    "scribbles": ladder_preds["scribbles"],
    "sam_pseudo": [
        predict_head(sam_head, features_for(specialised, it, side), it["mask"].shape) for it in spec_eval
    ],
}
sam_infer = round(time.monotonic() - t0, 1)
path2_report = {k: rung_report(v) for k, v in path2_preds.items()}
RESULTS["pseudo_label_iou"] = path2_report["sam_pseudo"]["iou"]  # held-out IoU of the head (SPEC §9)
RESULTS["sam_pseudo_head"] = path2_report["sam_pseudo"] | {
    "long_side": side,
    "n_train": len(scrib_train),
    "threshold": 0.5,
    "band": 0.0,
    "pseudo_label_iou_fit": sam_quality["iou"],
    "pseudo_sec": pseudo_sec,
    "sam_run_here": RUN_SAM,
    "trained_here": TRAIN_SAM_HEAD,
    "train_sec": round(float(sam_sec), 2),
    "infer_sec": sam_infer,
}


def print_path2(reports: dict) -> None:
    """One line per head on the held-out images: IoU, recall and the share of the fundus it calls vessel."""
    print(f"{'approach (held-out images)':<46}{'IoU':>6}{'recall':>8}{'calls vessel':>14}")
    for k, v in reports.items():
        name = approach_names.get(k, k)
        print(f"{name:<46}{v['iou']:>6.3f}{v['recall']:>8.3f}{v['marked']:>14.0%}")
    print(f"{'really vessel (the ground truth)':<46}{'':>14}{fg_share:>14.0%}")


print_path2(path2_report)

This helper cell defines `tune_pseudo_labels`, which trains a head on targets made with your cut and band.

In [ ]:
# @title Pseudo-label tuning helper: targets, head, diagnosis columns { display-mode: "form" }
def target_errors(targets: list[np.ndarray]) -> dict:
    """Diagnosis only (uses the hidden truth): share of the fundus ignored, of the true vessel pixels called
    background, and of the true background pixels called vessel, over the training images."""
    pairs = list(zip(targets, scrib_train, strict=True))
    return {
        "ignored": float(np.mean([(t < 0)[region(it)].mean() for t, it in pairs])),
        "vessel_as_bg": float(
            np.mean([((t == 0) & it["mask"]).sum() / max(it["mask"].sum(), 1) for t, it in pairs])
        ),
        "bg_as_vessel": float(
            np.mean(
                [
                    ((t == 1) & ~it["mask"] & region(it)).sum() / max((~it["mask"] & region(it)).sum(), 1)
                    for t, it in pairs
                ]
            )
        ),
    }


def tune_pseudo_labels(threshold: float, band: float) -> None:
    """Targets from SAM's maps with this cut and unsure band, a head trained on them, and its numbers
    printed under the head at 0.5."""
    ok = all(isinstance(v, (int, float)) for v in (threshold, band))
    if not ok or band < 0 or threshold - band <= 0 or threshold + band > 1:
        raise SystemExit(
            "PSEUDO_THRESHOLD must lie between 0 and 1, and UNSURE_BAND ≥ 0 must keep both edges inside."
        )
    side = specialised["feat_side"]
    tuned_targets = [
        pseudo_targets(p, it["scribbles"], region(it), threshold, band)
        for p, it in zip(pseudo_probs, scrib_train, strict=True)
    ]
    tuned_head, tuned_sec = train_head(scrib_feats, tuned_targets, HEAD_STEPS)
    tuned_preds = [
        predict_head(tuned_head, features_for(specialised, it, side), it["mask"].shape) for it in spec_eval
    ]
    tuned_report = rung_report(tuned_preds)
    RESULTS["sam_pseudo_tuned"] = tuned_report | {"threshold": threshold, "band": band}
    yours = f"yours (cut {threshold:g}, band {band:g})"
    print("1 · the labels the head trains on (training images, checked against the truth):")
    print(f"{'':<30}{'ignored':>10}{'true vessel':>14}{'true background':>18}")
    print(f"{'':<30}{'':>10}{'labelled bg':>14}{'labelled vessel':>18}")
    for name, t in {"usual (cut 0.5, no band)": sam_targets, yours: tuned_targets}.items():
        e = target_errors(t)
        print(f"{name:<30}{e['ignored']:>10.0%}{e['vessel_as_bg']:>14.0%}{e['bg_as_vessel']:>18.0%}")
    print(f"\n2 · the head trained on them, on the held-out images (trained in {tuned_sec:.1f} s):")
    print_path2({"usual (cut 0.5, no band)": path2_report["sam_pseudo"], yours: tuned_report})
    if threshold == 0.5 and band == 0:
        print("\nℹ️ Your setting equals the usual one: move a slider, then run this cell again.")
    k_demo = next(i for i, it in enumerate(spec_eval) if it is demo)
    ex = scrib_train[0]

    def label_view(t: np.ndarray) -> np.ndarray:
        """Training labels as three flat colours: vessel teal, background dark grey, ignored light grey."""
        out = np.full((*t.shape, 3), 224, np.uint8)  # ignored (−1) and outside the fundus
        out[t == 0] = (hex_rgb("#616161") * 255).astype(np.uint8)
        out[t == 1] = (hex_rgb(THEME["scribble"]) * 255).astype(np.uint8)
        return out

    def label_title(name: str, t: np.ndarray) -> str:
        r = region(ex)
        return (f"labels, {name}\nvessel {(t == 1)[r].mean():.0%} · background {(t == 0)[r].mean():.0%} · "
                f"ignored {(t < 0)[r].mean():.0%}")

    show_masks(
        [
            (label_title("usual", sam_targets[0]), label_view(sam_targets[0])),
            (label_title("yours", tuned_targets[0]), label_view(tuned_targets[0])),
            (rung_title("head, usual", path2_preds["sam_pseudo"][k_demo], demo),
             error_map(demo["image"], path2_preds["sam_pseudo"][k_demo], demo)),
            (rung_title("head, yours", tuned_preds[k_demo], demo),
             error_map(demo["image"], tuned_preds[k_demo], demo)),
        ],
        width=2.9,
        roles=["input", "input", "errors", "errors"],
        legend=[("label: vessel", THEME["scribble"]), ("label: background", "#616161"),
                ("label: ignored (−1)", "#E0E0E0"), "found", "invented", "missed"],
        note=f"Left: what each head is told on training image {ex['name']}. "
        "Right: what each head then predicts on the held-out demo image.",
    )

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F3E5F5;border-left:6px solid #4A148C;border-radius:6px;padding:12px 14px;margin:6px 0;color:#4A148C;font-family:inherit;line-height:1.5"><b>❓ Question</b><p>Run the next cell, then move the cut.</p>
<p><strong>Q15.</strong> 🟡 Tune the pseudo-labels on the <strong>training images</strong>, never on the test images. Raise or lower
<code>PSEUDO_THRESHOLD</code> and give the doubtful pixels an <code>UNSURE_BAND</code> to ignore, picking values from the spread
printed in 6.1 and judging them by how many training targets are wrong (the two error columns). Then look
at the held-out line once: does it move the same way? Is there a setting that beats 0.5 on your run?</p>
<ul>
<li><code>PSEUDO_THRESHOLD</code>: try 0.55, 0.6, 0.7 (or the median printed in 6.1).</li>
<li><code>UNSURE_BAND</code>: try 0.03; pixels within 0.03 of the threshold are ignored.</li>
</ul>
<p>Both are sliders in the next cell: move one, then run it again.</p></div>"""))

This cell trains the same head on labels made with your setting and compares it with the usual one (cut
0.5, no band), in two steps:

1. **The labels**, checked against the hidden truth on the training images: the share of the fundus
   **ignored** (−1), the share of the true vessel pixels **labelled background**, and the share of the true
   background **labelled vessel**. Lower is better for the last two.
2. **The heads** trained on those labels, on the held-out images, and a picture: the labels of one training
   image for both settings, then both heads' predictions on the demo image.

Colab does not rerun the cell when you move a slider: move it, then press ▶.

In [ ]:
# @title Q15 — tune the pseudo-label threshold and the unsure band { display-mode: "form" }
PSEUDO_THRESHOLD = 0.5  # @param {type:"slider", min:0.45, max:0.75, step:0.025}
UNSURE_BAND = 0.0  # @param {type:"slider", min:0, max:0.1, step:0.01}
tune_pseudo_labels(PSEUDO_THRESHOLD, UNSURE_BAND)

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F3E5F5;border-left:6px solid #4A148C;border-radius:6px;padding:12px 14px;margin:6px 0;color:#4A148C;font-family:inherit;line-height:1.5"><b>👥 Pair moment (optional, 4 min)</b><p>One of you tries a low cut (<code>PSEUDO_THRESHOLD = 0.55</code>), the other a high one (<code>0.7</code>). Compare your two
error columns (true vessel labelled background, true background labelled vessel): which kind of wrong
label did each of you buy, and what happened to the held-out line?</p></div>"""))

<details><summary>What to look for</summary>

- The threshold trades one kind of wrong label for the other. A low cut turns more of SAM's region into
  "vessel" (bg → vessel rises). A high cut keeps only SAM's surest pixels as vessel and the rest becomes
  background (vessel → bg rises): the section-3 mistake again, with SAM as the annotator who stopped early.
  When the probabilities crowd around one value, a step of 0.05 can flip most of the fundus.
- The band ignores the doubtful pixels instead of guessing, so both error columns can fall at once. The
  price is fewer labelled pixels (the "ignored" column).
- Whether the held-out numbers follow the training diagnosis depends on your run. In a real project you
  pick the setting on a few fully labelled images, never on the test images.

</details>

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#FDECEA;border-left:6px solid #8B1A10;border-radius:6px;padding:12px 14px;margin:6px 0;color:#8B1A10;font-family:inherit;line-height:1.5"><b>⚠️ Pitfall: over-trusting SAM&#x27;s confident masks</b><ol>
<li>SAM draws a crisp edge and fills the whole region. <strong>Consequence:</strong> it looks finished, with no "I don't
   know" anywhere.</li>
<li>The head cannot tell a guess from a drawing. <strong>Consequence:</strong> it learns SAM's habit (a region where you
   wanted a line) as faithfully as the vessels.</li>
</ol></div>"""))

This simulation invents a teacher that is right about the strokes but confidently calls the brightest blob
of the fundus (usually the optic disc) vessel. It trains one head on the strokes alone and one with that
blind spot, and shows both on the same held-out image (no ground truth, seconds per head). What to look
for: how much $\color{#6A1B9A}{\blacksquare}$ violet each head puts inside the
$\color{#E65100}{\blacksquare}$ orange blob.

In [ ]:
# @title Playful pitfall: a teacher with one confident blind spot { display-mode: "form" }
def bright_blob(item: dict) -> np.ndarray:
    """The brightest smooth blob of the fundus (usually the optic disc), found from the image alone."""
    g = ndi.gaussian_filter(item["image"][..., 1].astype(float), sigma=max(item["mask"].shape) / 60)
    g[~region(item)] = 0
    peak = np.unravel_index(int(g.argmax()), g.shape)
    lab, _ = ndi.label(g >= 0.9 * g.max())
    return ndi.binary_dilation(lab == lab[peak], iterations=max(1, max(item["mask"].shape) // 100))


def stroke_targets(item: dict, blind_spot: bool) -> np.ndarray:
    """The strokes only (1 vessel, 0 background, -1 unknown); with `blind_spot`, the teacher also calls the
    bright blob vessel wherever no stroke says otherwise."""
    t = np.full(item["mask"].shape, -1, np.int8)
    t[item["scribbles"] == 1] = 1
    t[item["scribbles"] == 2] = 0
    if blind_spot:
        t[bright_blob(item) & (item["scribbles"] == 0)] = 1
    return t


clean_head, _ = train_head(scrib_feats, [stroke_targets(it, False) for it in scrib_train], HEAD_STEPS)
biased_head, _ = train_head(scrib_feats, [stroke_targets(it, True) for it in scrib_train], HEAD_STEPS)
feat_demo = features_for(specialised, demo, side)
p_clean = predict_head(clean_head, feat_demo, demo["mask"].shape)
p_biased = predict_head(biased_head, feat_demo, demo["mask"].shape)
blob_demo = bright_blob(demo)
print(
    f"blind spot ({blob_demo.mean():.1%} of the image) marked as vessel · head on strokes alone "
    f"{p_clean[blob_demo].mean():.0%} · head on the biased teacher {p_biased[blob_demo].mean():.0%}\n"
    f"share of the fundus marked as vessel · strokes alone {p_clean[region(demo)].mean():.0%} · "
    f"biased teacher {p_biased[region(demo)].mean():.0%}"
)
show_masks(
    [
        (
            "the teacher's blind spot\n(it calls this vessel)",
            overlay(demo["image"], blob_demo, THEME["fails"]),
        ),
        ("head trained on\nthe strokes alone", overlay(demo["image"], p_clean, THEME["works"])),
        ("head trained on\nthe biased teacher", overlay(demo["image"], p_biased, THEME["works"])),
    ],
    width=3.0,
    legend=[
        ("invented", "orange: the blind spot, called vessel by the teacher"),
        ("mask", "violet: what the head marks as vessel"),
    ],
    note="no ground truth in this picture: the teacher's mistake is invented on purpose",
)

<details><summary>What to look for</summary>

- Compare the blind-spot share in the printout with the $\color{#6A1B9A}{\blacksquare}$ violet inside the
  $\color{#E65100}{\blacksquare}$ orange blob. If the biased head
  marks the blob far more than the clean head, it copied the teacher's mistake onto an image it never saw
  labelled.
- It may also mark other bright patches, because it learnt "bright and round means vessel", not "this exact
  blob".
- If both heads mark about the same share on your run, the strokes alone already outweighed the blind spot.

</details>

### 6.3 Grad-CAM: where a classifier looks

The second teacher does not segment at all.

1. We train a small classifier to answer one yes/no question.
2. We ask it *where* in the image its "yes" came from. Grad-CAM turns that answer into a heat map.
3. A threshold turns the heat map into a pseudo-mask.

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#EDE7F6;border-left:6px solid #311B92;border-radius:6px;padding:12px 14px;margin:6px 0;color:#311B92;font-family:inherit;line-height:1.5"><b>📘 Theory: Grad-CAM, and why it fits this task poorly</b><p>Grad-CAM shows which parts of the input made a classifier say "yes"
(<a style="color:inherit;text-decoration:underline" href="https://arxiv.org/abs/1610.02391">Selvaraju et al., 2017</a>). It needs a question with a real "no", such as
"is there a defect in this image?". Every retina image has vessels, so we ask about 64 px tiles instead:
"does this tile contain a vessel?", with labels read off the strokes. The map has one value per 16 px patch
and cannot outline a thin line. Seeing it fail shows what to check.</p></div>"""))

This helper cell defines the tile classifier, the tiling, the tile labels and its Grad-CAM map.

In [ ]:
# @title Grad-CAM helpers: tile classifier, tiles, labels from strokes, the map { display-mode: "form" }
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget


class TileClassifier(torch.nn.Module):
    """'Does this tile contain a vessel?' on frozen DINOv3 features: 3 x 3 conv (384 -> 32), ReLU, one logit
    per patch, averaged. Fully convolutional, so it also runs on a whole image."""

    def __init__(self, dim: int, hidden: int = 32):
        super().__init__()
        self.hidden = torch.nn.Conv2d(dim, hidden, 3, padding=1)
        self.act = torch.nn.ReLU()
        self.out = torch.nn.Conv2d(hidden, 1, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.out(self.act(self.hidden(x))).mean((2, 3))  # [B, 1]


def to_tiles(grid: torch.Tensor, t: int) -> torch.Tensor:
    """[C, h, w] -> [n, C, t, t]: non-overlapping t x t tiles, row by row (a ragged edge is dropped)."""
    c, h, w = grid.shape
    nh, nw = h // t, w // t
    g = grid[:, : nh * t, : nw * t].reshape(c, nh, t, nw, t)
    return g.permute(1, 3, 0, 2, 4).reshape(nh * nw, c, t, t)


def tile_labels(item: dict, grid: tuple[int, int], t: int) -> torch.Tensor:
    """Per tile, from the strokes only: 1 = a vessel stroke crosses it, 0 = only background strokes,
    -1 = no stroke (not used)."""
    has = [to_tiles(patch_share(item["scribbles"] == v, grid)[None], t).amax((1, 2, 3)) > 0 for v in (1, 2)]
    return torch.where(has[0], 1, torch.where(has[1], 0, -1))


def train_tile_classifier(x: torch.Tensor, y: torch.Tensor, steps: int, lr: float = 0.01):
    """Full-batch Adam with the balanced cross-entropy of every head; returns the classifier and seconds."""
    torch.manual_seed(SEED)
    clf = TileClassifier(x.shape[1]).to(DEVICE)
    opt = torch.optim.Adam(clf.parameters(), lr=lr)
    target = (y == 1).float()
    w = torch.where(y == 1, 0.5 / max(int((y == 1).sum()), 1), 0.5 / max(int((y == 0).sum()), 1))
    t0 = time.monotonic()
    for _ in range(steps):
        opt.zero_grad()
        bce = F.binary_cross_entropy_with_logits(clf(x)[:, 0], target, reduction="none")
        (bce * w).sum().backward()
        opt.step()
    return clf.eval(), time.monotonic() - t0


def cam_map(clf: TileClassifier, feat: torch.Tensor) -> np.ndarray:
    """Grad-CAM of the 'contains a vessel' score on a whole (standardised) feature grid: [h, w] in [0, 1]."""
    with GradCAM(model=clf, target_layers=[clf.act]) as cam:
        return cam(input_tensor=feat[None], targets=[ClassifierOutputTarget(0)])[0]


def held_out_vessel_tile_share(tile_patches: int) -> float:
    """Evaluation only: the share of held-out fundus tiles that truly contain a vessel."""
    shares = []
    for held in spec_eval:
        grid = tuple(features_for(specialised, held, side).shape[1:])
        vessel_share = patch_share(held["mask"] & region(held), grid)
        has_vessel = to_tiles(vessel_share[None], tile_patches).amax((1, 2, 3))
        inside = to_tiles(patch_share(region(held), grid)[None], tile_patches).mean((1, 2, 3)) > 0.5
        shares.append(float((has_vessel[inside] > 0).float().mean()))
    return float(np.mean(shares))


def cam_prob_maps(clf: TileClassifier, feats: list[torch.Tensor], items: list[dict]) -> list[np.ndarray]:
    """The Grad-CAM map of each training image, upsampled to the image size: [H, W] in [0, 1]."""
    maps = []
    for it, f in zip(items, feats, strict=True):
        small = torch.from_numpy(cam_map(clf, f))[None, None]
        up = F.interpolate(small, size=it["mask"].shape, mode="bilinear", align_corners=False)
        maps.append(up[0, 0].numpy())
    return maps

This cell cuts the five training images into 64 px tiles, labels each from the strokes (a vessel stroke
crosses it → "vessel", only background strokes → "no vessel", no stroke → unused), trains the classifier in
seconds and counts the tiles, with the held-out truth showing how rare a vessel-free tile really is.

In [ ]:
# @title Train the tile classifier and count the tiles { display-mode: "form" }
TILE_PATCHES = 1 if SMOKE else 4  # 64 px of the 1024 px crop either way (smoke thumbnails are 4× smaller)
CLS_STEPS = 30 if SMOKE else 200
all_feats = torch.cat([f.flatten(1) for f in scrib_feats], 1)
cls_mu, cls_sd = all_feats.mean(1).view(-1, 1, 1), all_feats.std(1).view(-1, 1, 1) + 1e-6
std_feats = [(f - cls_mu) / cls_sd for f in scrib_feats]
tile_x, tile_y = [], []
for tile_item, tile_feat in zip(scrib_train, std_feats, strict=True):
    tile_lab = tile_labels(tile_item, tuple(tile_feat.shape[1:]), TILE_PATCHES)
    tile_x.append(to_tiles(tile_feat, TILE_PATCHES)[tile_lab >= 0])
    tile_y.append(tile_lab[tile_lab >= 0])
tile_x, tile_y = torch.cat(tile_x), torch.cat(tile_y)
clf, cls_sec = train_tile_classifier(tile_x, tile_y, CLS_STEPS)
with torch.no_grad():
    fit_acc = float(((clf(tile_x)[:, 0] > 0).long() == tile_y).float().mean())
vessel_tile_share = held_out_vessel_tile_share(TILE_PATCHES)
print(
    f"tile classifier · {int((tile_y == 1).sum())} 'vessel' tiles, {int((tile_y == 0).sum())} 'no vessel' "
    f"tiles from the strokes · fit accuracy {fit_acc:.0%} · {cls_sec:.1f} s on {DEVICE}"
)
print(f"held-out truth (evaluation only): {vessel_tile_share:.0%} of the fundus tiles contain a vessel")

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F3E5F5;border-left:6px solid #4A148C;border-radius:6px;padding:12px 14px;margin:6px 0;color:#4A148C;font-family:inherit;line-height:1.5"><b>🤔 Your prediction</b><p>The classifier only ever answered "vessel in this tile, yes or no". Will its heat map look like a vessel
tree, like blobs, or like noise? Write one guess.</p></div>"""))

This cell cuts each Grad-CAM map at half its peak into a pseudo-mask (strokes still win), trains the same
head on it, scores it on the held-out images and shows one training image next to SAM's pseudo-label.

In [ ]:
# @title Grad-CAM pseudo-labels: map, cut, head, score { display-mode: "form" }
CAM_THRESHOLD = 0.5  # share of the map's peak; Grad-CAM maps are rescaled to [0, 1] per image
t0 = time.monotonic()
cam_probs = cam_prob_maps(clf, std_feats, scrib_train)
cam_sec = round(time.monotonic() - t0, 1)
# the supervision: strokes, and the tile classifier's heat map (trained on stroke-derived tile labels)
cam_targets = [
    pseudo_targets(p, it["scribbles"], region(it), CAM_THRESHOLD, 0.0)
    for p, it in zip(cam_probs, scrib_train, strict=True)
]
cam_quality = pseudo_numbers([t == 1 for t in cam_targets], scrib_train)
cam_head, cam_head_sec = train_head(scrib_feats, cam_targets, HEAD_STEPS)
t0 = time.monotonic()
path2_preds["gradcam_pseudo"] = [
    predict_head(cam_head, features_for(specialised, it, side), it["mask"].shape) for it in spec_eval
]
cam_infer = round(time.monotonic() - t0, 1)
path2_report["gradcam_pseudo"] = rung_report(path2_preds["gradcam_pseudo"])
RESULTS["gradcam_iou"] = path2_report["gradcam_pseudo"]["iou"]  # held-out IoU of the head (SPEC §9)
RESULTS["gradcam_note"] = (
    f"Patch-level proxy '{specialised['gradcam']['patch_level_task']}' (tiles labelled from the strokes), "
    f"because every image contains the structure and there is no image-level negative class: an awkward "
    f"fit, {vessel_tile_share:.0%} of the held-out tiles contain foreground"
)
RESULTS["gradcam_head"] = path2_report["gradcam_pseudo"] | {
    "long_side": side,
    "n_train": len(scrib_train),
    "tile_patches": TILE_PATCHES,
    "n_tiles": [int((tile_y == 1).sum()), int((tile_y == 0).sum())],
    "classifier_fit_acc": fit_acc,
    "held_vessel_tile_share": vessel_tile_share,
    "threshold": CAM_THRESHOLD,
    "pseudo_label_iou_fit": cam_quality["iou"],
    "train_sec": round(float(cls_sec + cam_head_sec), 2),
    "pseudo_sec": cam_sec,
    "infer_sec": cam_infer,
}
cam_truth = float(np.mean([it["mask"][region(it)].mean() for it in scrib_train]))
print(
    f"Grad-CAM pseudo-labels made in {cam_sec} s. Checked against the truth on the training images "
    f"(never used for training):\n"
    f"  Grad-CAM labels: IoU {cam_quality['iou']:.3f} · recall {cam_quality['recall']:.2f} · "
    f"{called_vessel(cam_quality['marked'], cam_truth)}\n"
    f"  SAM labels (cut 0.5): IoU {sam_quality['iou']:.3f} · recall {sam_quality['recall']:.2f} · "
    f"{called_vessel(sam_quality['marked'], cam_truth)}"
)
print(f"Grad-CAM head trained in {cam_head_sec:.1f} s on {DEVICE}")
print_path2(path2_report)
show_masks(
    [
        (f"{scrib_ex['name']}", scrib_ex["image"]),
        ("Grad-CAM map", prob_rgb(cam_probs[0])),
        (
            "Grad-CAM pseudo-label\nvs ground truth",
            error_map(scrib_ex["image"], cam_targets[0] == 1, scrib_ex),
        ),
        ("SAM pseudo-label\nvs ground truth", error_map(scrib_ex["image"], sam_targets[0] == 1, scrib_ex)),
    ],
    width=2.8,
    roles=["input", "map", "errors", "errors"],
    legend=[*prob_key, *error_key],
    note="Grad-CAM map: blue = strongest evidence for vessel; the pseudo-label keeps every pixel above "
    "half the peak.",
)

**Q16.** 🟡 Compare the two pseudo-labels of the same training image. What shape does the Grad-CAM
evidence have, and where does it sit? Which of the two would you rather correct by hand, and why?

<details><summary>What to look for</summary>

- Shape: blobs one or a few patches wide, where the classifier found the most "vessel-like" patches,
  usually the thick, dark vessels. It does not follow a line, because its resolution is the 16 px patch and
  its question does not need a line. After the cut, the blobs become fat regions with the thin vessels
  missing. SAM fails differently, as one big region with a careful edge.
- Which is better: neither is the vessel map. A coarse but *local* guess can teach the head more than a
  sharp but global one, so compare the held-out lines above before you pick.
- Other tasks: where the class really is a blob (a defect, a nucleus) and some images contain none, the
  same recipe fits much better.

</details>

### 6.4 Path 2, side by side

This cell prints the scribble head of section 4 and the two pseudo-label heads in one table, draws their
error maps on the held-out demo image and writes the annotation effort for section 8: the same strokes
for all three. What to look for: $\color{#6A1B9A}{\blacksquare}$ violet is what each head got right,
$\color{#E65100}{\blacksquare}$ orange is what it invented, $\color{#2E7D32}{\blacksquare}$ green is
what it missed. Recall is the share of the true vessels a head finds; "calls … vessel" is the share of the
fundus it calls vessel, next to the share that really is.

In [ ]:
# @title Path 2 side by side: table and error maps { display-mode: "form" }
RESULTS["path2_effort_min"] = {
    k: round(effort_minutes(n_strokes=n_strokes), 2) for k in ("sam_pseudo", "gradcam_pseudo")
}
print_path2(path2_report)
print(
    f"{'everything is foreground':<46}{RESULTS['specialised_everything_iou']:>6.3f}\n"
    f"annotation for all three: {n_strokes} strokes on {len(scrib_train)} images "
    f"(≈ {effort_minutes(n_strokes=n_strokes):.1f} min); the pseudo-labels add compute, not annotation"
)


def head_title(label: str, pred: np.ndarray, item: dict) -> str:
    """Panel title: the head, then its IoU, recall and the share of this fundus it calls vessel."""
    r = region(item)
    p, t = pred & r, item["mask"] & r
    return (
        f"{label}\nIoU {score(pred, item)['iou']:.2f} · recall {hidden_recall(p, t):.2f}\n"
        f"calls {100 * p[r].mean():.0f} % vessel (really {100 * t[r].mean():.0f} %)"
    )


show_masks(
    [
        (
            head_title(approach_names.get(k, k).split(":")[-1].strip(), p[k_demo], demo),
            error_map(demo["image"], p[k_demo], demo),
        )
        for k, p in path2_preds.items()
    ]
    + [("ground truth\n(every vessel, green)", overlay(demo["image"], demo["mask"], THEME["truth"]))],
    width=2.8,
    roles=["errors"] * len(path2_preds) + ["truth"],
    legend=error_key,
    note="held-out demo image · recall = true vessels found · calls … vessel = share of the fundus it "
    "calls vessel",
)

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#E0F2F1;border-left:6px solid #004D40;border-radius:6px;padding:12px 14px;margin:6px 0;color:#004D40;font-family:inherit;line-height:1.5"><b>✅ Takeaway: Path 2</b><ul>
<li><strong>You are about to train on a teacher's guesses</strong> → score them on a few fully labelled images first, and
  keep the person's strokes stronger than the teacher.</li>
<li><strong>Its probabilities crowd around 0.5</strong> → do not trust a hard cut. Tune the threshold on training images
  and ignore the doubtful band instead of guessing.</li>
<li><strong>Every image has the structure, so there is no "no" class</strong> → Grad-CAM is a poor fit. Use it when you
  have real image-level labels (defects, tumours).</li>
<li><strong>You need more labels but cannot pay for strokes</strong> → pseudo-labels add compute, not annotation, and
  bring the teacher's bias with them.</li>
</ul></div>"""))

## 7. Path 3 · Skip specialist training

Paths 1 and 2 trained a head once from strokes on five images, and that head then segmented every new image
on its own. Path 3 trains nothing: the strokes a person draws on an image become SAM's prompts on *that
same image*, and SAM's answer is the result.

It is the difference between teaching an intern once and pointing at every image yourself: no training time
and no training data, but the pointing never stops.

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#E8F5E9;border-left:6px solid #1B5E20;border-radius:6px;padding:12px 14px;margin:6px 0;color:#1B5E20;font-family:inherit;line-height:1.5"><b>🎯 Objective</b><p>Prompt the foundation model directly with scribble points and boxes, no training, and know when that beats
training a head.</p></div>"""))

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#EDE7F6;border-left:6px solid #311B92;border-radius:6px;padding:12px 14px;margin:6px 0;color:#311B92;font-family:inherit;line-height:1.5"><b>📘 Theory: what a SAM prompt says</b><p>Section 2.3 showed that many scattered clicks make SAM answer with one big region. Two things are new here:</p>
<ul>
<li>The prompt is split into many small ones, one box per stroke, each its own object. SAM answers many local
  questions instead of one global one, and the masks are combined.</li>
<li>Nothing is trained. The image encoder runs once per image, and every extra prompt costs only a fast
  decoder pass.</li>
</ul></div>"""))

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F5F5F5;border-left:6px solid #616161;border-radius:6px;padding:12px 14px;margin:6px 0;color:#616161;font-family:inherit;line-height:1.5"><details><summary><b>🔍 SAM papers</b></summary><ul>
<li>SAM, Kirillov et al., ICCV 2023: <a style="color:inherit;text-decoration:underline" href="https://arxiv.org/abs/2304.02643">arXiv:2304.02643</a></li>
<li>SAM 2, Ravi et al. 2024: <a style="color:inherit;text-decoration:underline" href="https://arxiv.org/abs/2408.00714">arXiv:2408.00714</a></li>
<li>SAM 3 (concept prompts, used in 7.4): Meta, 2025.</li>
</ul></details></div>"""))

This helper cell defines the stroke prompts, SAM on many boxes, the two recipes and a box-drawing figure.

In [ ]:
# @title Path 3 helpers: colours, strokes, SAM on many prompts, recipes, printout { display-mode: "form" }
from matplotlib.patches import Rectangle

approach_colours |= {
    "sam_clicks": "#546E7A",
    "sam_boxes": "#880E4F",
    "sam3_text": "#5D4037",
}
approach_names |= {
    "sam_clicks": "direct: SAM from stroke clicks",
    "sam_boxes": "direct: SAM, one box + click per stroke",
    "sam3_text": "direct: SAM 3 from a word",
}


def stroke_masks(item: dict, value: int = 1) -> list[np.ndarray]:
    """The separate strokes of one kind (1 vessel, 2 background) as boolean masks: the 8-connected pieces
    of the scribble map."""
    lab, n = ndi.label(item["scribbles"] == value, structure=np.ones((3, 3)))
    return [lab == k for k in range(1, n + 1)]


def stroke_click(stroke: np.ndarray) -> list[float]:
    """(x, y) of the stroke pixel nearest the stroke's centre: a click that is always on the stroke."""
    ys, xs = np.nonzero(stroke)
    k = int(np.argmin((xs - xs.mean()) ** 2 + (ys - ys.mean()) ** 2))
    return [float(xs[k]), float(ys[k])]


@torch.no_grad()
def sam_many(image: np.ndarray, boxes: list[list[float]], clicks: list[list[float]] | None = None, chunk=32):
    """SAM with one object per box (plus one positive click inside it, if given), decoded in chunks after a
    single image encoding. Returns the highest probability over the objects: a pixel is foreground as soon
    as one object covers it."""
    best = np.zeros(image.shape[:2], np.float32)
    for s in range(0, len(boxes), chunk):
        kw = {"input_boxes": [boxes[s : s + chunk]]}
        if clicks is not None:
            c = clicks[s : s + chunk]
            kw |= {"input_points": [[[p] for p in c]], "input_labels": [[[1]] * len(c)]}
        inputs = sam.proc(images=image, return_tensors="pt", **kw)
        if sam._cache is None or sam._cache[0] is not image:  # the same image cache as Sam.probability
            sam._cache = (image, sam.model.get_image_embeddings(inputs["pixel_values"].to(sam.device)))
        out = sam.model(
            image_embeddings=sam._cache[1],
            input_points=inputs["input_points"].to(sam.device) if clicks is not None else None,
            input_labels=inputs["input_labels"].to(sam.device) if clicks is not None else None,
            input_boxes=inputs["input_boxes"].to(sam.device),
            multimask_output=False,
        )
        logits = sam.proc.post_process_masks(out.pred_masks.cpu(), inputs["original_sizes"], binarize=False)[
            0
        ]
        best = np.maximum(best, torch.sigmoid(logits[:, 0]).amax(0).numpy())
    return best


def direct_clicks(items: list[dict], n: int) -> tuple[list[np.ndarray], list[tuple]]:
    """Recipe 1: n vessel clicks and n background clicks spread along the strokes of each image, one SAM
    prompt per image. Returns the masks and each image's (vessel, background) clicks."""
    preds, clicks = [], []
    for it in tqdm(items, desc="SAM, stroke clicks", leave=False):
        pos, neg = stroke_points(it, 1, n), stroke_points(it, 2, n)
        preds.append(
            sam.probability(it["image"], points=pos + neg, labels=[1] * len(pos) + [0] * len(neg)) >= 0.5
        )
        clicks.append((pos, neg))
    return preds, clicks


def direct_boxes(items: list[dict], pad: int, with_click: bool) -> tuple[list[np.ndarray], list[tuple]]:
    """Recipe 2: one box around every vessel stroke (grown by `pad` px; `stroke_box` is set in 7.2), plus a
    click at the stroke's centre if `with_click`; the masks of all strokes are combined. Returns the masks
    and each image's (boxes, clicks)."""
    preds, prompts = [], []
    for it in tqdm(items, desc="SAM, one box per stroke", leave=False):
        strokes = stroke_masks(it, 1)
        boxes = [[float(v) for v in stroke_box(s, pad)] for s in strokes]
        clicks = [stroke_click(s) for s in strokes] if with_click else None
        preds.append(
            sam_many(it["image"], boxes, clicks) >= 0.5 if boxes else np.zeros(it["mask"].shape, bool)
        )
        prompts.append((boxes, clicks))
    return preds, prompts


def show_boxes(
    panels: list[tuple], width: float = 3.2, legend: list | None = None, note: str | None = None,
    roles: list | None = None,
):
    """One row of panels (title, RGB image, boxes or None[, clicks]); boxes and clicks are drawn in teal.
    `legend` and `note` are the colour key of add_key() under the figure; `roles` one corner tag per panel."""
    fig, axes = plt.subplots(1, len(panels), figsize=(width * len(panels), width))
    for ax, (title, img, boxes, *extra), role in zip(
        np.atleast_1d(axes), panels, roles or [None] * len(panels), strict=True
    ):
        ax.imshow(img)
        ax.set_title(title, fontsize=9)
        ax.axis("off")
        tag_panel(ax, role)
        for x0, y0, x1, y1 in boxes or []:
            ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, lw=1, color=THEME["scribble"]))
        for x, y in extra[0] if extra and extra[0] else []:
            ax.plot(x, y, marker="*", markersize=7, color=THEME["scribble"], markeredgecolor="white")
    add_key(fig, legend, note)
    plt.show()


def print_path3(reports: dict) -> None:
    """One line per approach: IoU, recall and the share of the fundus the mask calls vessel."""
    print(f"{'approach (held-out images)':<46}{'IoU':>6}{'recall':>8}{'calls vessel':>14}")
    for k, v in reports.items():
        print(f"{approach_names.get(k, k):<46}{v['iou']:>6.3f}{v['recall']:>8.3f}{v['marked']:>14.0%}")
    print(f"{'really vessel (the ground truth)':<46}{'':>14}{fg_share:>14.0%}")


def try_boxes(box_pad: int, with_click: bool) -> None:
    """Run the box recipe with a margin of `box_pad` px (of the 1024 px crop) and the centre click on or off
    on every evaluation image and print it under the default. Stores the numbers in
    RESULTS["sam_direct_boxes_try"]."""
    if not isinstance(box_pad, int) or box_pad < 0 or not isinstance(with_click, bool):
        raise SystemExit("BOX_PAD_TRY must be a whole number ≥ 0 and WITH_CLICK_TRY True or False.")
    pad = max(box_pad // 4, 0) if SMOKE else box_pad  # smoke thumbnails are 4× smaller
    preds, _ = direct_boxes(spec_eval, pad, with_click)
    RESULTS["sam_direct_boxes_try"] = rung_report(preds) | {"pad": box_pad, "with_click": with_click}
    print_path3(
        {
            f"pad {BOX_PAD} px, with click (default)": path3_report["sam_boxes"],
            f"pad {box_pad} px, {'with' if with_click else 'no'} click (yours)": RESULTS[
                "sam_direct_boxes_try"
            ],
        }
    )

### 7.1 Clicks on the vessels, and "not here"

The news is **fairness**: the heads of sections 4 to 6 never saw a stroke on the held-out images, but Path 3
gets strokes on every one of them. It gets more help per image, and pays for it at use time.

The first recipe is section 6.1 without the head: twenty clicks along the vessel strokes and twenty along
the background strokes of each *held-out* image, and SAM's mask is the answer. The next cell prints that
one row. *Marked* is the share of the fundus (the round, scored part of the photo) the mask covers.

In [ ]:
# @title SAM from twenty vessel and twenty background clicks per image { display-mode: "form" }
N_DIRECT_CLICKS = 20  # clicks per kind (vessel / background) and image, taken from the strokes
t0 = time.monotonic()
path3_preds = {"zero_shot": ladder_preds["zero_shot"], "scribbles": ladder_preds["scribbles"]}
path3_preds["sam_clicks"], click_prompts = direct_clicks(spec_eval, N_DIRECT_CLICKS)
clicks_sec = round(time.monotonic() - t0, 1)
path3_report = {k: rung_report(v) for k, v in path3_preds.items()}
print(
    f"SAM from {N_DIRECT_CLICKS} + {N_DIRECT_CLICKS} stroke clicks on {len(spec_eval)} images · "
    f"{clicks_sec} s on {DEVICE}"
)
print_path3({"sam_clicks": path3_report["sam_clicks"]})
print(f"{'section 2: SAM, 20 vessel clicks only (IoU)':<46}{RESULTS['sam_prompt_ladder']['20 points']:>6.3f}")
print(f"{'everything is foreground (IoU)':<46}{RESULTS['specialised_everything_iou']:>6.3f}")

**What to look for.** Read *calls vessel* first. A value near 100 % means the "not here" clicks only cut a
few
holes in one big region (section 2.3), and the IoU stays near the "everything" line.

### 7.2 One small box per stroke

The second recipe splits the question:

1. Each vessel stroke gets its own box, a few pixels larger than the stroke, and a click at its centre.
2. SAM answers each box as a separate object.
3. The vessel map is the union of all the answers.

One stroke is a short piece of one vessel, so each box asks a local question SAM can answer: "the thing
along this line, inside this rectangle".

This helper cell defines `stroke_box`: the box around one stroke, grown by a margin.

In [ ]:
# @title stroke_box: the box around one stroke { display-mode: "form" }
def stroke_box(stroke: np.ndarray, pad: int) -> list[float]:
    """The box [x0, y0, x1, y1] around the True pixels of `stroke`, grown by `pad` pixels on every side and
    clipped to the image: 0 <= x <= width - 1, 0 <= y <= height - 1. np.nonzero returns rows (y) first."""
    ys, xs = np.nonzero(stroke)
    h, w = stroke.shape
    return [
        float(max(xs.min() - pad, 0)),
        float(max(ys.min() - pad, 0)),
        float(min(xs.max() + pad, w - 1)),
        float(min(ys.max() + pad, h - 1)),
    ]

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F3E5F5;border-left:6px solid #4A148C;border-radius:6px;padding:12px 14px;margin:6px 0;color:#4A148C;font-family:inherit;line-height:1.5"><b>🤔 Your prediction</b><p>The strokes cover only some of the vessels (section 4 dropped 70 % of the segments), and each box sits on a
stroke. Two guesses:</p>
<ul>
<li>Will this recipe mark more of the fundus than the clicks of 7.1, or less?</li>
<li>Will it beat simply thickening the strokes by a few pixels, with no model at all?</li>
</ul></div>"""))

This cell asks SAM for one object per vessel stroke (a box with 4 px of margin plus a centre click) on every
evaluation image, and prints it next to the strokes thickened by 3 px, the "no model" reference. The figure
shows the demo image: SAM's input on the left, its answer in the middle, the thickened strokes on the right.

In [ ]:
# @title SAM with one box and one click per stroke, next to the thickened strokes { display-mode: "form" }
BOX_PAD = 1 if SMOKE else 4  # px around each stroke (smoke thumbnails are 4× smaller)
THICKEN = 1 if SMOKE else 3  # px: the "no model" reference, the vessel strokes thickened
t0 = time.monotonic()
path3_preds["sam_boxes"], box_prompts = direct_boxes(spec_eval, BOX_PAD, with_click=True)
boxes_sec = round(time.monotonic() - t0, 1)
path3_report["sam_boxes"] = rung_report(path3_preds["sam_boxes"])
strokes_alone = [ndi.binary_dilation(it["scribbles"] == 1, iterations=THICKEN) for it in spec_eval]
strokes_report = rung_report(strokes_alone)
n_boxes = [len(b) for b, _ in box_prompts]
print(
    f"SAM, one box + click per vessel stroke · {int(np.mean(n_boxes))} boxes per image on average · "
    f"{boxes_sec} s on {DEVICE}"
)
print_path3(path3_report | {f"no model: the strokes thickened by {THICKEN} px": strokes_report})
boxes, centre_clicks = box_prompts[k_demo]
box_nums = image_numbers(path3_preds["sam_boxes"][k_demo], demo)
thick_nums = image_numbers(strokes_alone[k_demo], demo)
show_boxes(
    [
        (f"SAM's input: {len(boxes)} boxes\nand a centre click in each", demo["image"], boxes, centre_clicks),
        (
            f"SAM, one object per box\nIoU {box_nums['iou']:.2f} · calls {box_nums['marked']:.0%} vessel",
            error_map(demo["image"], path3_preds["sam_boxes"][k_demo], demo),
            None,
        ),
        (
            f"baseline: the strokes themselves, no SAM\nIoU {thick_nums['iou']:.2f} · "
            f"calls {thick_nums['marked']:.0%} vessel",
            error_map(demo["image"], strokes_alone[k_demo], demo),
            None,
        ),
    ],
    width=3.0,
    roles=["input", "errors", "errors"],
    legend=["box", "click", "found", "invented", "missed"],
    note="Left is what SAM is given; the other two panels compare with the true vessels.",
)

**What to look for.** $\color{#2E7D32}{\blacksquare}$ Green is missed vessels. Each box finds only what
lies inside it, so expect a lot of
$\color{#2E7D32}{\blacksquare}$ green between the boxes. Compare the middle and right panels: is SAM's
answer better than the plain
thickened strokes?

**Q17.** 🟡 Compare the box recipe with the clicks of 7.1 and with the thickened strokes: which column
moved, recall or "calls vessel"? Then grow the boxes (try 0, 16 and 32 px of the 1024 px crop) and untick
the
centre click in the next cell. What does a larger box let SAM find, and what does it cost?

This cell reruns the box recipe with your margin and click setting and prints it under the default.

In [ ]:
# @title Q17 — larger or smaller boxes, with or without the centre click { display-mode: "form" }
BOX_PAD_TRY = 16  # @param {type:"slider", min:0, max:48, step:8}
WITH_CLICK_TRY = True  # @param {type:"boolean"}
try_boxes(BOX_PAD_TRY, WITH_CLICK_TRY)

<details><summary>What to look for</summary>

- **Recall is capped.** Each box finds only what lies inside it: the vessels a person stroked, plus the
  branches that cross their boxes. A larger box reaches more branches (recall rises) but also more
  background (*calls vessel* rises, and part of it is invented). At some size it becomes section 2's big
  box.
- **The centre click** tells SAM which thing inside the box is meant. It helps most when the box holds more
  than one candidate.
- **The baseline:** if the thickened strokes come close to or beat SAM, that is a real finding, not a bug.
  On thin structures, compare against the cheapest baseline built from the hints alone.

</details>

### 7.3 Path 3, side by side

This cell puts the two direct recipes next to the section-2 box, the scribble head and the SAM
pseudo-label head, and writes the numbers section 8 compares. The figure shows the demo image under each
approach: no training time, but annotation on *every* image you segment, at 2 s per stroke.

In [ ]:
# @title Path 3 side by side: table, annotation cost and error maps { display-mode: "form" }
path3_preds["sam_pseudo"] = path2_preds["sam_pseudo"]
path3_report["sam_pseudo"] = path2_report["sam_pseudo"]
eval_meta = specialised["eval"]
strokes_per_image = {
    "sam_clicks": float(np.mean([m.get("n_fg_strokes", 0) + m.get("n_bg_strokes", 0) for m in eval_meta])),
    "sam_boxes": float(np.mean([m.get("n_fg_strokes", 0) for m in eval_meta])),  # vessel strokes only
}
effort = {k: effort_minutes(n_strokes=round(v * len(eval_meta))) for k, v in strokes_per_image.items()}
sec = {"sam_clicks": clicks_sec, "sam_boxes": boxes_sec}
result_key = {"sam_clicks": "sam_direct_clicks", "sam_boxes": "sam_direct_boxes"}
for k in ("sam_clicks", "sam_boxes"):
    per_image = [score(p, it)["iou"] for p, it in zip(path3_preds[k], spec_eval, strict=True)]
    RESULTS[result_key[k]] = path3_report[k] | {
        "iou_std": float(np.std(per_image)),
        "train_sec": 0.0,
        "infer_sec": sec[k],
        "n_images": len(spec_eval),
        "strokes_per_image": strokes_per_image[k],
        "effort_min_per_image": round(effort_minutes(n_strokes=1) * strokes_per_image[k], 2),
        "effort_min": round(effort[k], 2),
    }
RESULTS["sam_direct_boxes"] |= {"pad": BOX_PAD, "with_click": True}
RESULTS["sam_direct_clicks"] |= {"n_clicks": N_DIRECT_CLICKS}
RESULTS["sam_direct_iou"] = path3_report["sam_boxes"]["iou"]  # headline: points + boxes (SPEC §6, 07)
RESULTS["sam_direct_clicks_iou"] = path3_report["sam_clicks"]["iou"]
RESULTS["sam_direct_strokes_alone"] = strokes_report | {"thicken_px": THICKEN}
RESULTS["path3_effort_min"] = {k: round(v, 2) for k, v in effort.items()}

order = ["zero_shot", "scribbles", "sam_pseudo", "sam_clicks", "sam_boxes"]
print_path3({k: path3_report[k] for k in order})
print(
    f"{'no model: strokes thickened (IoU)':<46}{strokes_report['iou']:>6.3f}\n"
    f"{'everything is foreground (IoU)':<46}{RESULTS['specialised_everything_iou']:>6.3f}"
)
print(
    f"\nannotation · the heads: {n_strokes} strokes on {len(scrib_train)} training images, once "
    f"(≈ {effort_minutes(n_strokes=n_strokes):.1f} min)"
)
for k in ("sam_clicks", "sam_boxes"):
    print(
        f"annotation · {approach_names[k]}: ≈ {RESULTS[result_key[k]]['effort_min_per_image']:.1f} min on "
        f"every image you segment ({effort[k]:.0f} min for these {len(spec_eval)})"
    )
short3 = {
    "zero_shot": "zero-shot\n(one box)",
    "scribbles": "scribble head\n(trained once)",
    "sam_pseudo": "SAM pseudo-label\nhead (trained once)",
    "sam_clicks": "SAM from\nstroke clicks",
    "sam_boxes": "SAM, one box\nper stroke",
}
show_masks(
    [
        (
            f"{short3[k]}\nIoU {score(path3_preds[k][k_demo], demo)['iou']:.2f}",
            error_map(demo["image"], path3_preds[k][k_demo], demo),
        )
        for k in order
    ]
    + [("ground truth\n(the true vessels)", overlay(demo["image"], demo["mask"], THEME["truth"]))],
    width=2.6,
    roles=["errors"] * len(order) + ["truth"],
    legend=["found", "invented", ("missed", "missed (truth only); the last panel shows every true vessel")],
    note="Same demo image under every approach; the last panel is the answer key.",
)

**What to look for.** The two trained heads (second and third panel) are not prompted on this image at use
time. The two SAM recipes are, with strokes drawn on it. Look at $\color{#E65100}{\blacksquare}$ orange
(invented) and $\color{#2E7D32}{\blacksquare}$ green (missed):
which approach mostly invents, which mostly misses?

### 7.4 SAM 3: a word instead of a stroke (needs SAM 3 access)

SAM 3 (Meta, 2025) takes a short noun phrase such as "retinal vessels" and segments every instance of it,
with no click at all. It is gated on Hugging Face, so it works only if your account has access and
`setup_check.ipynb` cached it. If not, the next cell says so in one line and you can skip to the takeaway;
nothing later depends on it.

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F3E5F5;border-left:6px solid #4A148C;border-radius:6px;padding:12px 14px;margin:6px 0;color:#4A148C;font-family:inherit;line-height:1.5"><b>❓ Question</b><p><strong>Q18.</strong> 🟡 Run the next cell, type "retinal vessels" and press "Ask SAM 3" on a few retina images. Then
try other words ("blood vessel", "vessel", "dark line", "optic disc"). Which words find the vessels? Does
the best word beat the stroke boxes of 7.2, which cost you strokes on every image? What does the answer
tell you about the images SAM 3 learnt its words from?</p></div>"""))

This cell shows whether SAM 3 is available, then opens a playground on the retina images: type a word and
press "Ask SAM 3", or click and drag to prompt SAM 2.1 instead. Tick "Show ground truth" to colour it.

In [ ]:
# @title Q18 — Ask SAM 3 with a word on the retina (needs SAM 3 access) { display-mode: "form" }
s3 = load_sam3()  # from 00: loads SAM 3 once from your cache, never from the internet
if "error" in s3:
    RESULTS["sam3_note"] = "SAM 3 not available in this session"
    print(f"ℹ️ {s3['error']}")
    print("The playground below still takes clicks and boxes (SAM 2.1). Nothing else needs SAM 3.")
else:
    print(f"SAM 3 ready · {SAM3_ID} · SAM License (Meta) · {str(s3['dtype']).split('.')[-1]} on {DEVICE}")
sam_playground(
    spec_eval,
    "Retina: ask SAM 3 with a word, or click",
    text_prompt=True,
    default_text="retinal vessels",
)

<details><summary>What to look for</summary>

- **Only known concepts work.** A concept prompt works for words SAM 3 met in training, named the way its
  captions named them. "Vessel" in a fundus photograph is a rare, specialist phrase next to "dog" or "car",
  so expect thick vessels, a blob, or nothing, and a lot of change with the wording.
- **Appearance can beat meaning.** A generic word like "dark line" sometimes does better than the correct
  medical term.
- **It is free.** A word costs no annotation, so even a weak answer is a free starting point for the strokes
  of 7.2.

</details>

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F5F5F5;border-left:6px solid #616161;border-radius:6px;padding:12px 14px;margin:6px 0;color:#616161;font-family:inherit;line-height:1.5"><details><summary><b>🔍 Optional: score a word on all evaluation images</b></summary><p>The playground shows one image at a time. The next cell scores one phrase on every evaluation image, with
the same IoU, recall and "calls vessel" columns as above, and stores it in <code>RESULTS["sam3_text"]</code> so
section 8 can add an optional row. It needs SAM 3 access and otherwise prints one line and stops.</p></details></div>"""))

In [ ]:
# @title Optional: score one SAM 3 phrase on all images (needs SAM 3 access) { display-mode: "form" }
SAM3_TEXT = "retinal vessels"  # @param ["retinal vessels", "blood vessel", "dark line"] {allow-input: true}
if not isinstance(SAM3_TEXT, str) or not SAM3_TEXT.strip():
    raise SystemExit('SAM3_TEXT must be a short phrase in quotes, e.g. "retinal vessels".')
RESULTS["sam3_text_iou"] = None
if "error" in s3:
    print("ℹ️ SAM 3 is not available in this session, so this optional cell stops here.")
else:
    try:
        t0, sam3_preds = time.monotonic(), []
        for it in tqdm(spec_eval, desc="SAM 3, one phrase", leave=False):
            found = sam3_masks(it["image"], SAM3_TEXT)
            sam3_preds.append(found.any(0) if len(found) else np.zeros(it["mask"].shape, bool))
        sam3_report = rung_report(sam3_preds)
        RESULTS["sam3_text_iou"] = sam3_report["iou"]
        RESULTS["sam3_text"] = sam3_report | {"text": SAM3_TEXT, "infer_sec": round(time.monotonic() - t0, 1)}
        print_path3(
            {
                f'SAM 3, "{SAM3_TEXT}"': sam3_report,
                "SAM 2.1, one box + click per stroke": path3_report["sam_boxes"],
            }
        )
    except Exception as err:  # noqa: BLE001 - an optional extension must never stop the notebook
        RESULTS["sam3_note"] = f"SAM 3 loaded but failed: {type(err).__name__}: {err}"
        print(f"⚠️ SAM 3 loaded but did not run here ({type(err).__name__}: {err}). It is optional; carry on.")

Want to try this on a photo of your own? Section 9 lets you draw the strokes yourself.

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#E0F2F1;border-left:6px solid #004D40;border-radius:6px;padding:12px 14px;margin:6px 0;color:#004D40;font-family:inherit;line-height:1.5"><b>✅ Takeaway: Path 3</b><ul>
<li><strong>Your structure is thin and spread over the image</strong> → more clicks only make a bigger region: split the
  prompt into one small box and a centre click per stroke.</li>
<li><strong>Few images, or each one matters</strong> → prompt SAM directly and review every mask.</li>
<li><strong>Images keep coming</strong> → train a head once (Path 1 or 2); direct prompting pays strokes on every image.</li>
<li><strong>A word is free but narrow</strong> → SAM 3 only knows the concepts it was trained on; check it against the
  strokes.</li>
</ul></div>"""))

## 8. Compare everything

Every section since 2 ended with its own small table, each against a different neighbour. Here they all
sit in one table, scored on the same held-out images against the same hidden ground truth.

The question is the one you face on a real project: not "which approach has the highest number" but
"which approach would I pay for". A recipe can win on accuracy and lose on annotation hours, or win on
average and fail badly on one image in five. Three pictures follow the table: a trade-off plot, a violin
plot, and the winner's best and worst images.

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#E8F5E9;border-left:6px solid #1B5E20;border-radius:6px;padding:12px 14px;margin:6px 0;color:#1B5E20;font-family:inherit;line-height:1.5"><b>🎯 Objective</b><p>fill the comparison table, predicting the winner first.</p></div>"""))

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#EDE7F6;border-left:6px solid #311B92;border-radius:6px;padding:12px 14px;margin:6px 0;color:#311B92;font-family:inherit;line-height:1.5"><b>📘 Theory: the five columns</b><p>Each row of the table is read through five columns:</p>
<ul>
<li><strong>Accuracy</strong>: mean IoU over the held-out images (0 = no overlap with the truth, 1 = perfect), plus
  clDice (does the mask follow the vessel centre lines?).</li>
<li><strong>Annotation effort</strong>: what a person had to draw, in minutes. A stroke or a box is seconds, a complete
  vessel map is hours.</li>
<li><strong>Reliability, or spread</strong>: the standard deviation of the per-image IoU, i.e. how far a typical image
  sits from the average. IoUs of 0.60, 0.62, 0.58 give a spread of 0.02: trust the average. IoUs of
  0.90, 0.30, 0.60 have the same average but a spread of 0.25: some images fail badly.</li>
<li><strong>Compute</strong>: seconds of training plus inference in this session.</li>
<li><strong>Scalability</strong>: a sentence, not a number. What does the next 1,000 images cost?</li>
</ul>
<p>The table also shows <strong>recall</strong> (the share of the true vessel pixels the mask finds) and <strong>worst</strong> (the
lowest IoU on any single image).</p></div>"""))

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#FFF8E1;border-left:6px solid #5D4037;border-radius:6px;padding:12px 14px;margin:6px 0;color:#5D4037;font-family:inherit;line-height:1.5"><b>🧾 Assumptions behind the minutes</b><ul>
<li>A scribble stroke, a click or a box takes about <strong>2 seconds</strong>.</li>
<li>A complete vessel map of one fundus image takes <strong>3 to 5 hours</strong> (FIVES paper, 2022). We count the
  midpoint, 4 hours, and a half-painted map as half of that.</li>
<li>Both numbers come from the dataset config, not from a measurement in this room. Change them there and
  every row moves.</li>
<li>Training minutes are paid once. The per-image column is what every <em>new</em> image costs.</li>
</ul></div>"""))

This helper cell gathers every approach's predictions, seconds, annotation minutes and scaling note.

In [ ]:
# @title Comparison helpers: every approach, its predictions, seconds and effort { display-mode: "form" }
approach_colours |= {"full_masks": THEME["works"]}
approach_names |= {"full_masks": "full masks: complete maps (section 2)"}
ALL_APPROACHES = [  # the short names Q19 accepts
    "zero_shot",
    "one_shot",
    "scribbles",
    "naive",
    "masked",
    "pu",
    "sam_pseudo",
    "gradcam_pseudo",
    "sam_clicks",
    "sam_boxes",
    "full_masks",
]
COMPARE_LABEL = {  # readable names for figure titles and the table
    "zero_shot": "zero-shot box",
    "one_shot": "one-shot (go-deeper)",
    "scribbles": "scribble head",
    "naive": "naive (gaps = bg)",
    "masked": "masked loss",
    "pu": "PU loss",
    "sam_pseudo": "SAM pseudo-labels",
    "gradcam_pseudo": "Grad-CAM labels",
    "sam_clicks": "SAM from clicks",
    "sam_boxes": "SAM from boxes",
    "full_masks": "full masks",
    "sam3_text": "SAM 3 from a word",
}
# An optional extra row: section 7's SAM 3 phrase, only if that cell ran (it needs SAM 3 access).
has_sam3 = (
    isinstance(RESULTS.get("sam3_text"), dict)
    and "sam3_preds" in globals()
    and len(sam3_preds) == len(spec_eval)
)
compare_order = [*ALL_APPROACHES, *(["sam3_text"] if has_sam3 else [])]
compare_reports = (  # mean IoU, clDice, recall, precision, marked share and held-out predictions per approach
    {k: ladder_report[k] | {"held_preds": ladder_preds[k]} for k in ("zero_shot", "one_shot", "scribbles")}
    | {k: path_reports[approach_names[k]] for k in ("naive", "masked", "pu")}
    | {k: path2_report[k] | {"held_preds": path2_preds[k]} for k in ("sam_pseudo", "gradcam_pseudo")}
    | {k: path3_report[k] | {"held_preds": path3_preds[k]} for k in ("sam_clicks", "sam_boxes")}
    | {"full_masks": path_reports["full masks (section 2)"]}
    | ({"sam3_text": RESULTS["sam3_text"] | {"held_preds": sam3_preds}} if has_sam3 else {})
)
compare_seconds = {  # training + pseudo-labels + inference, as recorded by each section in this session
    "zero_shot": RESULTS["specialised_zero_shot_seconds"] / len(LADDER),  # the box: 1 of section 2's prompts
    "one_shot": RESULTS["one_shot"]["infer_sec"],
    "full_masks": spec_heads[side][1] + RESULTS["scribbles_head"]["infer_sec"],  # same head, same features
} | ({"sam3_text": RESULTS["sam3_text"]["infer_sec"]} if has_sam3 else {}) | {
    k: sum(float(RESULTS[entry].get(part, 0.0)) for part in ("train_sec", "pseudo_sec", "infer_sec"))
    for k, entry in {
        "scribbles": "scribbles_head",
        "naive": "naive_head",
        "masked": "masked_head",
        "pu": "pu_head",
        "sam_pseudo": "sam_pseudo_head",
        "gradcam_pseudo": "gradcam_head",
        "sam_clicks": "sam_direct_clicks",
        "sam_boxes": "sam_direct_boxes",
    }.items()
}
compare_effort = (  # annotation minutes for this run (section 4's effort_minutes and its assumptions)
    RESULTS["ladder_effort_min"]
    | RESULTS["path1_effort_min"]
    | RESULTS["path2_effort_min"]
    | RESULTS["path3_effort_min"]
    | {"full_masks": round(effort_minutes(n_full_masks=len(spec_train)), 2)}
    | {"sam3_text": 0.0}  # a phrase typed once; no strokes
)
compare_per_image = {  # annotation minutes for every *new* image; trained heads need none
    k: 0.0 for k in compare_order
} | {
    "zero_shot": effort_minutes(n_strokes=1),  # one box
    "sam_clicks": RESULTS["sam_direct_clicks"]["effort_min_per_image"],
    "sam_boxes": RESULTS["sam_direct_boxes"]["effort_min_per_image"],
}
SCALES = {
    "zero_shot": "a box on every new image, SAM per image, nothing to train",
    "one_shot": "one complete mask once (go-deeper); new images cost one feature pass",
    "scribbles": "minutes of strokes once; new images cost one feature pass",
    "naive": "half-painted maps once (hours); new images cost one feature pass",
    "masked": "half-painted maps plus a few strokes once; new images cost one feature pass",
    "pu": "half-painted maps once plus a guessed share; new images cost one feature pass",
    "sam_pseudo": "strokes once, SAM at training time only; new images cost one feature pass",
    "gradcam_pseudo": "strokes once, a small classifier at training time; new images cost one feature pass",
    "sam_clicks": "strokes on every new image, SAM per image; the clicking never stops",
    "sam_boxes": "strokes on every new image, SAM per box; the clicking never stops",
    "full_masks": "hours of painting per training image; new images cost one feature pass",
    "sam3_text": "one phrase for every image, SAM 3 per image; no strokes at all",
}
use_cldice = "cldice" in specialised.get("metric_extras", [])
print(f"{len(compare_order)} approaches, each scored on the same {len(spec_eval)} held-out images:")
for k in compare_order:
    print(f"  {k:<16}{approach_names[k]}")
cost = specialised["annotation_effort"]
print(
    f"effort assumptions ({specialised['name']}): {cost['seconds_per_stroke']} s per stroke, click or box · "
    f"{cost['hours_per_full_mask'][0]}–{cost['hours_per_full_mask'][-1]} h per complete mask "
    f"(we count {np.mean(cost['hours_per_full_mask']):.0f} h)"
)

### 8.1 Look first, then predict

This cell shows every approach's mask on one held-out image, next to the image and its hidden ground truth.

What to look for: who follows the thin vessels, who paints blobs, who marks almost nothing.
$\color{#6A1B9A}{\blacksquare}$ Violet is
always a model's mask and $\color{#2E7D32}{\blacksquare}$ green is the ground truth. A title's colour is
the approach's colour in the
charts below.

In [ ]:
# @title All approaches on one held-out image { display-mode: "form" }
k_show = 0  # which held-out image to look at
it_show = spec_eval[k_show]
truth_img = overlay(it_show["image"], it_show["mask"], THEME["truth"])
gallery = [
    (f"{it_show['name']} · image", it_show["image"], "#311B92"),
    ("ground truth (scoring only)", truth_img, THEME["truth"]),
]
for k in compare_order:
    pred = compare_reports[k]["held_preds"][k_show] & region(it_show)
    shown = overlay(it_show["image"], pred, THEME["works"])
    gallery.append((f"{COMPARE_LABEL[k]}\n{k}", shown, approach_colours[k]))
g_cols = 5
g_rows = -(-len(gallery) // g_cols)
fig, axes = plt.subplots(g_rows, g_cols, figsize=(2.6 * g_cols, 2.9 * g_rows))
for ax in np.ravel(axes):
    ax.axis("off")
for ax, (title, img, colour) in zip(np.ravel(axes), gallery, strict=False):
    ax.imshow(img)
    ax.set_title(title, fontsize=8, color=colour)
add_key(fig, ["mask", "truth"], "title colour = the approach's colour in the charts below")
plt.show()

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F3E5F5;border-left:6px solid #4A148C;border-radius:6px;padding:12px 14px;margin:6px 0;color:#4A148C;font-family:inherit;line-height:1.5"><b>🤔 Your prediction (1 min)</b><p><strong>Q19.</strong> 🟡 Before the table appears: which approach will have the <strong>highest mean IoU</strong>, and which will be
the <strong>steadiest</strong> (the smallest spread of IoU from image to image)?</p>
<ul>
<li>Choose your two picks from the dropdowns in the next cell and run it; leave them blank to skip. The
  short names are the second line of each picture title above. The table further down shows where
  they landed.</li>
</ul></div>"""))

In [ ]:
# Q19 — your two predictions (pick from the dropdowns, or leave blank to skip)
MY_WINNER = ""  # @param ["", "zero_shot", "one_shot", "scribbles", "naive", "masked", "pu", "sam_pseudo", "gradcam_pseudo", "sam_clicks", "sam_boxes", "full_masks"]
MY_STEADIEST = ""  # @param ["", "zero_shot", "one_shot", "scribbles", "naive", "masked", "pu", "sam_pseudo", "gradcam_pseudo", "sam_clicks", "sam_boxes", "full_masks"]
for label, pick in (("MY_WINNER", MY_WINNER), ("MY_STEADIEST", MY_STEADIEST)):
    if not isinstance(pick, str) or (pick and pick not in ALL_APPROACHES):
        raise SystemExit(f'{label} must be "" or one of: {", ".join(ALL_APPROACHES)}.')
print(f"your picks · highest IoU: {MY_WINNER or '(none)'} · steadiest: {MY_STEADIEST or '(none)'}")

This helper cell defines `spread`, the standard deviation of one approach's per-image IoUs.

In [ ]:
# @title spread: the standard deviation of the per-image IoU { display-mode: "form" }
def spread(per_image_iou: list[float]) -> float:
    """Reference: numpy's standard deviation, which divides by n by default.
    e.g. [0.1, 0.3, 0.2, 0.6] -> 0.19"""
    return float(np.std(np.asarray(per_image_iou, dtype=float)))

### 8.2 The table

This cell scores every approach image by image against the hidden ground truth and shows the table,
sorted by IoU, then where your two picks landed. The best value of each column is in bold.

In [ ]:
# @title Score every approach and show the table { display-mode: "form" }
comparison = []
iou_per_image = {}  # approach -> its IoU on each held-out image (drawn as violins below)
for k in compare_order:
    rep = compare_reports[k]
    pairs = zip(rep["held_preds"], spec_eval, strict=True)
    per_image = [iou(p & region(it), it["mask"] & region(it)) for p, it in pairs]
    iou_per_image[k] = per_image
    comparison.append(
        {
            "approach": k,
            "name": approach_names[k],
            "iou": float(np.mean(per_image)),
            **({"cldice": float(rep["cldice"])} if use_cldice else {}),
            "recall": float(rep["recall"]),
            "precision": float(rep["precision"]),
            "effort_min": float(compare_effort[k]),
            "effort_min_per_new_image": round(float(compare_per_image[k]), 2),
            "reliability": float(spread(per_image)),
            "worst_iou": float(np.min(per_image)),
            "seconds": round(float(compare_seconds[k]), 1),
            "scalability": SCALES[k],
        }
    )
RESULTS["comparison_table"] = comparison
RESULTS["comparison_assumptions"] = dict(specialised["annotation_effort"]) | {
    "reliability": f"standard deviation of the IoU over {len(spec_eval)} held-out images",
    "seconds": f"training + pseudo-labels + inference in this session, on {DEVICE}",
}

from html import escape

ranked = sorted(comparison, key=lambda r: r["iou"], reverse=True)
thick_iou = RESULTS["sam_direct_strokes_alone"]["iou"]
table_cols = [  # header, row key, number format, which value is the best one
    ("IoU", "iou", "{:.3f}", max),
    *([("clDice", "cldice", "{:.3f}", max)] if use_cldice else []),
    ("recall", "recall", "{:.3f}", max),
    ("± spread", "reliability", "{:.3f}", min),
    ("worst", "worst_iou", "{:.3f}", max),
    ("effort min", "effort_min", "{:.1f}", min),
    ("/new img", "effort_min_per_new_image", "{:.2f}", min),
    ("sec", "seconds", "{:.1f}", min),
]


def comparison_html(rows: list[dict]) -> str:
    """The table as HTML: a colour chip per approach, the best value of each column in bold."""
    best = {}
    for _, key, fmt, pick in table_cols:
        top = fmt.format(pick(r[key] for r in rows))
        ties = sum(fmt.format(r[key]) == top for r in rows)
        best[key] = top if ties <= 3 else None  # when many rows tie, nobody is "the best"
    th = "padding:4px 8px;border-bottom:2px solid #311B92;color:#311B92;text-align:"
    td = "padding:3px 8px;border-bottom:1px solid #D1C4E9;text-align:"
    head = f'<th style="{th}left">approach</th>'
    head += "".join(f'<th style="{th}right">{h}</th>' for h, *_ in table_cols)
    lines = []
    for r in rows:
        chip = (
            '<span style="display:inline-block;width:11px;height:11px;border-radius:3px;margin-right:6px;'
            f'background:{approach_colours[r["approach"]]}"></span>'
        )
        cells = f'<td style="{td}left;white-space:nowrap">{chip}{escape(COMPARE_LABEL[r["approach"]])} '
        cells += f'<span style="color:#757575">{r["approach"]}</span></td>'
        for _, key, fmt, _pick in table_cols:
            text = fmt.format(r[key])
            text = f"<b>{text}</b>" if text == best[key] else text
            cells += f'<td style="{td}right">{text}</td>'
        lines.append(f"<tr>{cells}</tr>")
    for label, value in (
        ("no model: strokes thickened (section 7)", thick_iou),
        ("everything is foreground", RESULTS["specialised_everything_iou"]),
    ):
        grey = "color:#757575;font-style:italic"
        lines.append(
            f'<tr><td style="{td}left;{grey}">{label}</td><td style="{td}right;{grey}">{value:.3f}</td>'
            f'<td colspan="{len(table_cols) - 1}" style="{td}left"></td></tr>'
        )
    scales = "".join(f"<li><b>{r['approach']}</b>: {escape(r['scalability'])}</li>" for r in rows)
    return (
        '<table style="border-collapse:collapse;font-size:13px;margin:6px 0">'
        f"<tr>{head}</tr>{''.join(lines)}</table>"
        '<div style="font-size:12px;color:#616161;margin:2px 0 6px">recall = share of the true vessel pixels '
        "found · ± spread = std of the per-image IoU · worst = lowest per-image IoU · effort min = "
        "annotation minutes once · /new img = annotation minutes per new image · bold = best of its column · "
        "the colour chip is the approach's colour in the charts below"
        + (" · clDice = how well the mask follows the vessel centre lines (section 2.2)" if use_cldice else "")
        + "</div>"
        '<details style="background:#F5F5F5;border-left:6px solid #616161;border-radius:6px;padding:8px 12px;'
        'color:#616161;font-size:13px"><summary><b>🔍 How each one scales</b></summary>'
        f'<ul style="margin:6px 0">{scales}</ul></details>'
    )


try:
    from IPython.display import HTML, display

    display(HTML(comparison_html(ranked)))
except ImportError:  # no IPython: a plain list with the same numbers
    for r in ranked:
        nums = " · ".join(f"{h} {fmt.format(r[key])}" for h, key, fmt, _ in table_cols)
        print(f"{r['approach']:<16}{nums}")
    print(f"no model: strokes thickened (section 7) {thick_iou:.3f}")
    print(f"everything is foreground {RESULTS['specialised_everything_iou']:.3f}")

winner = ranked[0]["approach"]
steadiest = min(comparison, key=lambda r: r["reliability"])["approach"]
by_spread = [r["approach"] for r in sorted(comparison, key=lambda r: r["reliability"])]
print(f"highest IoU: {winner} · steadiest: {steadiest}")
if MY_WINNER in by_spread:
    rank = [r["approach"] for r in ranked].index(MY_WINNER) + 1
    print(f"your IoU pick {MY_WINNER} ranks {rank} of {len(ranked)}")
if MY_STEADIEST in by_spread:
    rank = by_spread.index(MY_STEADIEST) + 1
    print(f"your steadiness pick {MY_STEADIEST} ranks {rank} of {len(by_spread)}")
if not (MY_WINNER or MY_STEADIEST):
    print("no prediction written in Q19; fine, but next time guess first: it is the cheapest lesson here")

### 8.3 The trade-off in one picture

This cell draws the table twice. On the left, each approach is a dot: how much annotation it needed
(log scale) against how well it did (IoU), with a bar for its spread. On the right, the seconds each
approach took.

What to look for: the dots near the top-left are accurate and cheap. A short bar is a steady approach. The
dashed line is the score of marking everything, so a dot near it has learned almost nothing.

In [ ]:
# @title Trade-off plot: accuracy against annotation, and compute { display-mode: "form" }
fig, (ax, bx) = plt.subplots(1, 2, figsize=(10, 4.2), gridspec_kw={"width_ratios": [1.25, 1]})
top = min(1.0, 1.25 * max(r["iou"] + r["reliability"] for r in comparison) + 0.05)
gap = 0.045 * top  # labels of approaches with about the same effort are stacked at least this far apart
groups, last = [], None
for r in sorted(comparison, key=lambda r: r["effort_min"]):
    lx = np.log10(max(r["effort_min"], 0.01))
    if last is None or lx - last > 0.35:
        groups.append([])
    groups[-1].append(r)
    last = lx
label_y = {}
for group in groups:
    y_prev = -1.0
    for r in sorted(group, key=lambda r: r["iou"]):
        label_y[r["approach"]] = y_prev = max(r["iou"], y_prev + gap)
for r in comparison:
    x, colour = max(r["effort_min"], 0.01), approach_colours[r["approach"]]
    ax.errorbar(x, r["iou"], yerr=r["reliability"], fmt="o", ms=7, capsize=3, color=colour)
    ax.annotate(
        r["approach"],
        (x, r["iou"]),
        xytext=(x * 1.25, label_y[r["approach"]]),
        fontsize=7,
        color=colour,
        va="center",
    )
ax.axhline(
    RESULTS["specialised_everything_iou"], color=THEME["background"], ls="--", label='IoU of "everything"'
)
ax.plot([], [], "o", color="#616161", label="dot = mean IoU, bar = ± spread")
ax.set_xscale("log")
ax.set_xlim(right=ax.get_xlim()[1] * 4)  # room for the labels of the right-most points
ax.set_xlabel("annotation effort, ≈ minutes (log scale)")
ax.set_ylabel("mean IoU on the hidden ground truth (± spread)")
ax.set_ylim(0, top)
ax.set_title("Accuracy against annotation")
ax.legend(fontsize=7, frameon=False, loc="upper left")
names = [r["approach"] for r in ranked][::-1]
bx.barh(names, [r["seconds"] for r in ranked][::-1], color=[approach_colours[k] for k in names])
bx.set_xlabel(f"seconds, training + inference ({DEVICE})")
bx.tick_params(axis="y", labelsize=7)
bx.set_title("Compute (sorted by IoU, best at the top)")
add_key(fig, None, "colour = approach · left: higher is more accurate, further left is cheaper · right: shorter bar is faster")
plt.show()

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#FFF8E1;border-left:6px solid #5D4037;border-radius:6px;padding:12px 14px;margin:6px 0;color:#5D4037;font-family:inherit;line-height:1.5"><b>🧾 Reading the plot</b><ul>
<li><strong>Top-left</strong> would be the dream (accurate and nearly free; on thin structures nobody is there),
  <strong>bottom-left</strong> is cheap and weak, and the <strong>far right</strong> is the complete-mask head, hours away.</li>
<li><strong>The vertical bar is the spread</strong>: a slightly lower dot with a much shorter bar can be the safer choice.</li>
<li><strong>Dots at the same x</strong> are the fairest comparison: the same strokes, used in different ways.</li>
<li><strong>Right panel</strong>: seconds matter little for the heads and a lot for anything that runs SAM on every image.</li>
</ul></div>"""))

**Q20.** 🟡 Pick one approach for each situation and name the column that decided it.

- (a) A screening study of 1,000 new images next month.
- (b) One difficult image a clinician needs today.
- (c) A project where a missed vessel is much worse than a false alarm.

Columns you can use: IoU, recall, spread, worst, effort min, /new img. Did the IoU winner win any of them?

<details><summary>A possible answer</summary>

- (a) **/new img** decides. Strokes or a box on every image cost 1,000 times their minutes; a trained
  head pays its annotation once. Among the heads, pick by IoU and spread.
- (b) One image today: a per-image row needs no training. Compare it with the thickened-strokes line.
- (c) IoU is the wrong column. Rank by **recall** and **worst**, and accept an over-marking head if a
  person checks it.

The IoU winner can win none of the three. That is the point of a table with more than one column.

</details>

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F3E5F5;border-left:6px solid #4A148C;border-radius:6px;padding:12px 14px;margin:6px 0;color:#4A148C;font-family:inherit;line-height:1.5"><b>👥 Pairs, 4 min: defend a row</b><p>Use your Q20 answers. <strong>A</strong> defends one row for situation (a), <strong>B</strong> defends another row for situation
(c), one minute each, naming the column that decides. Then swap: attack the other's row.</p>
<ul>
<li>Which column did the defence rest on, and which column of the same row would hurt it most?</li>
<li>If you changed any setting earlier (threshold, number of clicks, box size), did a row of your table
  move compared with your neighbour's?</li>
</ul></div>"""))

<details><summary>What pairs usually find</summary>

- A good defence names a column: **/new img** for (a), **recall** or **worst** for (c).
- Every row has a weak column (hours of painting, clicking on every image, over-marking). Find it.
- Two rows whose IoUs differ by less than their spread are a tie: choose by the cheaper column.

</details>

### 8.4 Who is steady, and who fails on some images

This cell draws, for each approach, one dot per held-out image and a violin around the dots. The bar is the
median image, and approaches are sorted by mean IoU, best on the left.

What to look for: a tall violin means the approach is good on some images and bad on others. Compare two
approaches with the same mean: the one with the lower bottom dots is the riskier one.

In [ ]:
# @title Violin plot: each approach's IoU over the held-out images { display-mode: "form" }
fig, ax = plt.subplots(figsize=(10, 3.8))
for x, r in enumerate(ranked):
    vals = np.asarray(iou_per_image[r["approach"]], float)
    colour = approach_colours[r["approach"]]
    if len(vals) >= 2 and np.ptp(vals) > 1e-9:  # a violin needs some spread to draw
        body = ax.violinplot(vals, positions=[x], widths=0.8, showextrema=False)["bodies"][0]
        body.set_facecolor(colour)
        body.set_edgecolor(colour)
        body.set_alpha(0.3)
    ax.scatter(np.full(len(vals), x), vals, s=12, color=colour, zorder=3)
    ax.plot([x - 0.3, x + 0.3], [np.median(vals)] * 2, color=colour, lw=2.5, zorder=4)
line_all = RESULTS["specialised_everything_iou"]
ax.axhline(line_all, color=THEME["background"], ls="--", label='mean IoU of "everything"')
ax.plot([], [], "o", ms=3, color="#616161", label="dot = one held-out image")
ax.plot([], [], "-", lw=2.5, color="#616161", label="bar = median image")
ax.set_xticks(range(len(ranked)), [r["approach"] for r in ranked], rotation=45, ha="right", fontsize=7)
ax.set_ylabel("IoU on one held-out image")
ax.set_title("Per-image IoU, best mean on the left")
ax.legend(fontsize=7, frameon=False, loc="upper right")
add_key(fig, None, "colour = approach, as in the table · wide or tall violin = unsteady")
plt.show()

This cell shows the IoU winner on its **worst**, **typical** (median) and **best** held-out image, as error
maps. What to look for: what the winner invents ($\color{#E65100}{\blacksquare}$ orange) and what it misses
($\color{#2E7D32}{\blacksquare}$ green) when it fails, against
when it works.

In [ ]:
# @title Worst, typical and best image of the IoU winner { display-mode: "form" }
w_iou = np.asarray(iou_per_image[winner], float)
w_order = np.argsort(w_iou)
panels = []
w_picks = (("worst", w_order[0]), ("typical (median)", w_order[len(w_order) // 2]), ("best", w_order[-1]))
for tag, n in w_picks:
    it_n, pred_n = spec_eval[n], compare_reports[winner]["held_preds"][n]
    panels.append((f"{winner}\n{tag} · IoU {w_iou[n]:.2f}", error_map(it_n["image"], pred_n, it_n)))
show_masks(panels, width=3.4, legend=["found", "invented", "missed", "unscored"])

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#E0F2F1;border-left:6px solid #004D40;border-radius:6px;padding:12px 14px;margin:6px 0;color:#004D40;font-family:inherit;line-height:1.5"><b>✅ When you see X, do Y: choosing an approach</b><ul>
<li><strong>When</strong> two approaches differ by less than their spread <strong>→</strong> call it a tie and pick the cheaper one.</li>
<li><strong>When</strong> the best approach needs strokes or a box on every new image and you have thousands of images
  <strong>→</strong> compare the /new img column first; a trained head pays its annotation once.</li>
<li><strong>When</strong> a missed structure costs more than a false alarm <strong>→</strong> rank by recall and worst image, not by
  mean IoU.</li>
<li><strong>When</strong> the IoU winner is also the most expensive row <strong>→</strong> ask what the next-cheapest row loses
  before you pay for it.</li>
</ul>
<p>Section 9 runs the cheapest rungs on your own images; section 10 turns this table into rules.</p></div>"""))

## 9. Your own images

Everything so far ran on our images, where a hidden ground truth told us who was right. This section is
the dress rehearsal for your next problem.

Pick one to three photos of your own (a leaf, a crack in a wall, cells, a satellite tile, a pet) and draw a
few strokes on what you want and on what you do not want. Then run three rungs of the ladder on them.

There is no ground truth here. You judge the masks the way you would on a real project: by eye, against
your strokes, and by where the three approaches disagree.

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#E8F5E9;border-left:6px solid #1B5E20;border-radius:6px;padding:12px 14px;margin:6px 0;color:#1B5E20;font-family:inherit;line-height:1.5"><b>🎯 Objective</b><p>scribble on your own images and run the ladder.</p></div>"""))

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#FDECEA;border-left:6px solid #8B1A10;border-radius:6px;padding:12px 14px;margin:6px 0;color:#8B1A10;font-family:inherit;line-height:1.5"><b>⚠️ Pitfall: privacy</b><ul>
<li>Your images stay in your Colab session: they go to this session's <code>/content</code> folder, and nothing is
  written to Drive.</li>
<li>The canvas runs in your browser and sends your strokes only to your own Colab kernel.</li>
<li>We never use a public share link (such as Gradio's <code>share=True</code>): that would put your images on a
  public URL.</li>
<li>Even so, pick images you are allowed to process in a cloud notebook: no patient data, nothing under
  an NDA.</li>
</ul></div>"""))

This helper cell defines image picking, the drawing canvas and the stroke decoding (0 nothing, 1 the thing,
2 background, as in section 4).

In [ ]:
# @title Your images: example, canvas, picking images, stroke decoding { display-mode: "form" }
import base64
import io
import uuid
from string import Template

from IPython.display import HTML, display

own_strokes: dict[int, np.ndarray] = {}  # image number -> strokes saved from the canvas (0 / 1 thing / 2 bg)
CANVAS_PX = 640  # long side of the canvas on screen; your strokes are scaled back to the image
BRUSH_PX = 6  # pen width on screen
_CANVAS = Template("""
<div><b>$title</b><br>
<div style="position:relative;width:${cw}px;height:${ch}px">
<img src="data:image/jpeg;base64,$b64" width="$cw" height="$ch" style="position:absolute;left:0;top:0">
<canvas id="fmc$uid" width="$cw" height="$ch"
 style="position:absolute;left:0;top:0;cursor:crosshair;touch-action:none"></canvas>
</div>
<button id="fmt$uid">🖍️ the thing</button> <button id="fmb$uid">🖍️ background</button>
<button id="fmx$uid">clear</button> <button id="fms$uid">💾 save strokes</button>
<span id="fmm$uid">pen: the thing</span></div>
<script>
(function () {
  var c = document.getElementById("fmc$uid"), ctx = c.getContext("2d");
  var msg = document.getElementById("fmm$uid"), pen = "$thing", down = false;
  ctx.lineWidth = $brush; ctx.lineCap = "round"; ctx.lineJoin = "round";
  function xy(e) { var r = c.getBoundingClientRect(); return [e.clientX - r.left, e.clientY - r.top]; }
  c.addEventListener("pointerdown", function (e) {
    down = true; c.setPointerCapture(e.pointerId);
    var p = xy(e); ctx.strokeStyle = pen; ctx.beginPath(); ctx.moveTo(p[0], p[1]);
    ctx.lineTo(p[0], p[1]); ctx.stroke();
  });
  c.addEventListener("pointermove", function (e) {
    if (!down) return; var p = xy(e); ctx.lineTo(p[0], p[1]); ctx.stroke();
  });
  c.addEventListener("pointerup", function () { down = false; });
  document.getElementById("fmt$uid").onclick = function () {
    pen = "$thing"; msg.textContent = "pen: the thing";
  };
  document.getElementById("fmb$uid").onclick = function () {
    pen = "$bg"; msg.textContent = "pen: background";
  };
  document.getElementById("fmx$uid").onclick = function () {
    ctx.clearRect(0, 0, c.width, c.height); msg.textContent = "cleared (press save to send the empty canvas)";
  };
  document.getElementById("fms$uid").onclick = function () {
    if (!(window.google && google.colab && google.colab.kernel)) {
      msg.textContent = "saving needs Colab: use the PNG fallback below"; return;
    }
    msg.textContent = "saving...";
    google.colab.kernel.invokeFunction("fm_workshop.save_strokes", [$k, c.toDataURL("image/png")], {})
      .then(function () { msg.textContent = "✅ saved, now run the cell below the canvases"; })
      .catch(function (err) { msg.textContent = "❌ " + err; });
  };
})();
</script>
""")


def shrink(arr: np.ndarray, max_side: int) -> np.ndarray:
    """Resize so the long side is at most `max_side`: bilinear for an RGB image, nearest for a 2-D map."""
    h, w = arr.shape[:2]
    if max(h, w) <= max_side:
        return arr
    s = max_side / max(h, w)
    size = (max(1, round(w * s)), max(1, round(h * s)))
    resample = Image.BILINEAR if arr.ndim == 3 else Image.NEAREST
    return np.array(Image.fromarray(arr.astype(np.uint8)).resize(size, resample)).astype(arr.dtype)


def own_item(name: str, image: np.ndarray, source: str, auto=None, mask=None) -> dict:
    """One image of this section: `auto` = generated strokes (example only), `mask` = its ground truth
    (example only, for scoring, never for training: L-005)."""
    return {
        "name": name,
        "image": image,
        "source": source,
        "auto": auto,
        "mask": mask,
        "fov": np.ones(image.shape[:2], bool),  # no border to ignore: the whole image counts
        "scribbles": None,  # filled in 9.2: 0 nothing, 1 the thing, 2 background
    }


def shrink_strokes(strokes: np.ndarray, max_side: int) -> np.ndarray:
    """Like `shrink` for strokes, but thin strokes survive: a pixel keeps a stroke if its block has one"""
    if max(strokes.shape) <= max_side:
        return strokes
    out = np.zeros(shrink(strokes, max_side).shape, np.uint8)
    for v in (2, 1):  # the thing wins over the background where they meet
        layer = Image.fromarray(((strokes == v) * 255).astype(np.uint8))
        out[np.array(layer.resize((out.shape[1], out.shape[0]), Image.BOX)) > 0] = v
    return out


def example_items(max_side: int) -> list[dict]:
    """Our example: two held-out pets (compact) and one held-out retina image (thin vessels), with the strokes
    our scribble generator drew from their masks."""
    out = []
    for it in [load_item(pets_task, meta) for meta in pets_task["held"][:2]] + [spec_eval[0]]:
        item = own_item(
            it["name"],
            shrink(it["image"], max_side),
            "example",
            auto=shrink_strokes(it["scribbles"], max_side),
            mask=shrink(it["mask"], max_side),
        )
        if it.get("fov") is not None:  # the retina's black border is not scored
            item["fov"] = shrink(it["fov"], max_side)
        out.append(item)
    return out


def show_canvas(k: int, image: np.ndarray, title: str) -> None:
    """A drawing canvas over image k: two pens, clear, save. Saving calls `save_strokes` in this kernel."""
    small = shrink(image, CANVAS_PX)
    buf = io.BytesIO()
    Image.fromarray(small).save(buf, format="JPEG", quality=85)
    s = CANVAS_PX / max(image.shape[:2])
    ch, cw = round(image.shape[0] * s), round(image.shape[1] * s)
    html = _CANVAS.substitute(
        title=title,
        k=k,
        cw=cw,
        ch=ch,
        b64=base64.b64encode(buf.getvalue()).decode(),
        uid=uuid.uuid4().hex[:8],
        thing=THEME["scribble"],
        bg=THEME["background"],
        brush=BRUSH_PX,
    )
    display(HTML(html))


def decode_strokes(png: bytes, shape: tuple[int, int]) -> np.ndarray:
    """A stroke map (0 nothing, 1 the thing, 2 background) at `shape` from a PNG: either the canvas layer
    (transparent, teal = thing, grey = background) or a copy of your image painted elsewhere with pure red
    (thing) and pure blue (background)."""
    img = Image.open(io.BytesIO(png)).convert("RGBA").resize((shape[1], shape[0]), Image.NEAREST)
    r, g, b, a = (np.asarray(img, int)[..., i] for i in range(4))
    red, blue = (r > 200) & (g < 60) & (b < 60), (b > 200) & (r < 60) & (g < 60)
    out = np.zeros(shape, np.uint8)
    if (a < 255).any():  # a transparent layer: every opaque pixel is a stroke
        on = a > 127
        out[on] = 2  # grey (or blue)
        out[on & (((r < 79) & (g > 79)) | red)] = 1  # teal (or red)
    else:  # a painted copy of the photo: only the pure pen colours count
        out[red], out[blue] = 1, 2
    return out


def strokes_png(strokes: np.ndarray) -> bytes:
    """The other way round, as the canvas sends it: a transparent PNG with teal and grey strokes."""
    rgba = np.zeros((*strokes.shape, 4), np.uint8)
    for v, colour in ((1, THEME["scribble"]), (2, THEME["background"])):
        rgba[strokes == v] = [*np.round(hex_rgb(colour) * 255).astype(int), 255]
    buf = io.BytesIO()
    Image.fromarray(rgba).save(buf, format="PNG")
    return buf.getvalue()


def save_strokes(k, data_url: str) -> None:
    """Called by a canvas's 💾 button: decode the strokes of image k and keep them in this session."""
    k = int(k)
    own_strokes[k] = decode_strokes(base64.b64decode(data_url.split(",", 1)[1]), own[k]["image"].shape[:2])


STROKE_KEY = [("thing stroke", THEME["scribble"]), ("background stroke", THEME["background"])]


def stroke_panel(it: dict) -> np.ndarray:
    """The image with its strokes, thickened to stay visible: the thing in teal, the background in grey."""
    grow = max(1, max(it["image"].shape[:2]) // 256)
    thick = {v: ndi.binary_dilation(it["scribbles"] == v, iterations=grow) for v in (1, 2)}
    drawn = overlay(it["image"], thick[2], THEME["background"], 0.9)
    return overlay(drawn, thick[1], THEME["scribble"], 0.9, dim=False)


def stroke_targets(strokes: np.ndarray, region: np.ndarray) -> np.ndarray:
    """Training targets from strokes: 1 the thing, 0 background, -1 (ignored) everywhere you drew nothing.
    e.g. strokes [[1, 0, 2]] -> [[1, -1, 0]]"""
    targets = np.full(strokes.shape, -1, np.int8)
    targets[strokes == 2] = 0
    targets[strokes == 1] = 1
    targets[~np.asarray(region, bool)] = -1
    return targets


def show_strokes(items: list[dict]) -> None:
    """Each image next to its strokes."""
    panels = []
    for it in items:
        panels += [(it["name"], it["image"]), (f"strokes · {it['stroke_source']}", stroke_panel(it))]
    show_masks(panels, width=2.6, legend=STROKE_KEY, note="strokes are drawn thicker so you can see them")


def pick_images(ask_upload: bool, use_example: bool, my_images: list) -> list[dict]:
    """The images of this section: uploaded in Colab if `ask_upload`, else the files listed in `my_images`
    (up to 3, each shrunk to OWN_MAX_SIDE), else our example. Prints and shows them."""
    if (
        not isinstance(ask_upload, bool)
        or not isinstance(use_example, bool)
        or not isinstance(my_images, list)
    ):
        raise SystemExit(
            "ASK_UPLOAD, USE_EXAMPLE must be True or False; MY_IMAGES a list of paths in quotes."
        )
    missing = [p for p in my_images if not Path(p).is_file()]
    if missing:
        raise SystemExit(f"not found: {missing}. Upload with the folder icon on the left and copy each path.")
    seed_everything()
    own_strokes.clear()  # a fresh start: strokes from an earlier run belong to earlier images
    own_paths = [Path(p) for p in my_images]
    if ask_upload and IN_COLAB and not use_example and not own_paths:  # opt-in: Run all never waits here
        from google.colab import files

        print("Pick 1 to 3 images (JPEG or PNG), or press Cancel to use our example.")
        upload_dir = Path("/content/own_images")  # this session's disk only; gone when the session ends
        upload_dir.mkdir(exist_ok=True)
        try:
            uploaded = files.upload()
        except Exception as err:  # noqa: BLE001 - a cancelled upload raises in some Colab versions
            print(f"(upload stopped: {type(err).__name__})")
            uploaded = {}
        for fname, data in uploaded.items():
            (upload_dir / fname).write_bytes(data)
            own_paths.append(upload_dir / fname)
    if len(own_paths) > 3:
        print(f"ℹ️ {len(own_paths)} images: keeping the first 3, so every step stays short.")
        own_paths = own_paths[:3]
    images = []
    for p in own_paths if not use_example else []:
        try:
            images.append(
                own_item(p.stem, shrink(np.array(Image.open(p).convert("RGB")), OWN_MAX_SIDE), "yours")
            )
        except Exception as err:  # noqa: BLE001 - any unreadable file gets the same plain message
            print(
                f"⚠️ {p.name}: could not read it as an image ({type(err).__name__}), skipped. Use JPEG or PNG."
            )
    if not images:
        images = example_items(OWN_MAX_SIDE)
        print("ℹ️ Using our example: two held-out pets and one retina image, with generated strokes.")
    sizes = ", ".join(f"{it['name']} {it['image'].shape[1]}×{it['image'].shape[0]}" for it in images)
    print(f"{len(images)} image(s) · {sizes} · {images[0]['source']}")
    show_masks([(it["name"], it["image"]) for it in images], width=3.0)
    return images


def stroke_effort(items: list[dict]) -> tuple[dict[int, int], float]:
    """Number of strokes per kind (1 thing, 2 background) over `items`, and the minutes they cost to draw."""
    n_strokes = {v: sum(len(stroke_masks(it, v)) for it in items) for v in (1, 2)}
    return n_strokes, effort_minutes(n_strokes=n_strokes[1] + n_strokes[2])


def collect_strokes(stroke_pngs: list) -> list[dict]:
    """Give every image of own its strokes (your PNGs, then the canvas, then the generated strokes of the
    example), print the counts and show them. Returns the images the ladder will run on: the ones with a
    thing stroke, or our example if there are none (or no background stroke at all)."""
    if not isinstance(stroke_pngs, list) or len(stroke_pngs) > len(own):
        raise SystemExit(
            f"MY_STROKE_PNGS must be a list of at most {len(own)} paths, one per image, in order."
        )
    for k, p in enumerate(stroke_pngs):
        if p and not Path(p).is_file():
            raise SystemExit(f"{p} not found. Upload it with the folder icon on the left and copy its path.")
        if p:
            own_strokes[k] = decode_strokes(Path(p).read_bytes(), own[k]["image"].shape[:2])
    for k, it in enumerate(own):
        if k in own_strokes:
            it["scribbles"], it["stroke_source"] = own_strokes[k], "drawn by you"
        elif it["auto"] is not None:  # the example, not redrawn: generated strokes through the canvas decoder
            it["scribbles"] = decode_strokes(strokes_png(it["auto"]), it["image"].shape[:2])
            it["stroke_source"] = "generated"
        else:
            it["scribbles"], it["stroke_source"] = np.zeros(it["image"].shape[:2], np.uint8), "none yet"
    no_thing = [it["name"] for it in own if not (it["scribbles"] == 1).any()]
    if no_thing:
        print(
            f"⚠️ no 'thing' stroke on {no_thing}: skipped. Draw one, press 💾 save, and run this cell again."
        )
    ladder = [it for it in own if (it["scribbles"] == 1).any()]
    if not any((it["scribbles"] == 2).any() for it in ladder):
        print(
            "⚠️ No background stroke on any image, and the head needs both kinds. Draw at least one background"
        )
        print("   stroke and save. Until then the ladder runs on our example, so the notebook keeps going.")
        ladder = []
    if not ladder:
        ladder = example_items(OWN_MAX_SIDE)
        for it in ladder:
            it["scribbles"] = decode_strokes(strokes_png(it["auto"]), it["image"].shape[:2])
            it["stroke_source"] = "generated"
    n_strokes, minutes = stroke_effort(ladder)
    print(
        f"{len(ladder)} image(s) · {n_strokes[1]} thing strokes, {n_strokes[2]} background strokes · "
        f"≈ {minutes:.1f} min at {specialised['annotation_effort']['seconds_per_stroke']} s per stroke"
    )
    show_strokes(ladder)
    return ladder


OWN_MAX_SIDE = 256 if SMOKE else 1024  # long side, in pixels, after shrinking


if IN_COLAB:
    from google.colab import output

    output.register_callback("fm_workshop.save_strokes", save_strokes)  # the canvas talks to this kernel only

### 9.1 Pick one to three images

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#FFE8D6;border-left:6px solid #7A2E00;border-radius:6px;padding:12px 14px;margin:6px 0;color:#7A2E00;font-family:inherit;line-height:1.5"><b>✏️ Your turn (optional, 2 min): choose your images</b><p>By default the next cell uses our example: two held-out pets and one retina image, with strokes drawn by
our scribble generator. To use your own images:</p>
<ol>
<li>Tick <code>ASK_UPLOAD</code> in the next cell and run it.</li>
<li>Pick one to three JPEG or PNG files in the upload button that appears (press Cancel to keep the example).</li>
</ol>
<ul>
<li>Choose images that show the <em>same kind of thing</em>: the head is trained on all of them together.</li>
<li>Already uploaded them with the folder icon on the left? Put their paths in <code>MY_IMAGES</code> instead.</li>
</ul>
<p>Nothing here is mandatory.</p></div>"""))

This cell uploads your images if `ASK_UPLOAD` is ticked (in Colab), shrinks them to at most 1024 px, and
shows them. Tick `USE_EXAMPLE` to keep our example.

In [ ]:
# @title Choose your images { display-mode: "form" }
ASK_UPLOAD = False  # @param {type:"boolean"}
USE_EXAMPLE = False  # @param {type:"boolean"}
MY_IMAGES = []  # @param {type:"raw"}
own = pick_images(ASK_UPLOAD, USE_EXAMPLE, MY_IMAGES)

### 9.2 Draw your strokes 🖍️

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#FFE8D6;border-left:6px solid #7A2E00;border-radius:6px;padding:12px 14px;margin:6px 0;color:#7A2E00;font-family:inherit;line-height:1.5"><b>✏️ Your turn (optional, 4 min): scribble</b><p>Run the next cell: a canvas appears over each image.</p>
<ol>
<li>With the <strong>thing</strong> pen (<span style="color:#00838F">■</span> teal), draw two to five short strokes <em>on</em>
   what you want segmented, on
   different parts of it.</li>
<li>With the <strong>background</strong> pen (<span style="color:#9E9E9E">■</span> grey), draw two to five strokes on
   what you do not want, especially near
   the boundary and on anything that looks similar.</li>
<li>Press <strong>💾 save strokes</strong> under each canvas, then run the "collect strokes" cell further down.</li>
</ol>
<p>Strokes, not outlines: a few seconds per image, the annotation budget of section 4. On the example,
drawing is optional: an image you do not save keeps its generated strokes.</p></div>"""))

This cell draws one canvas per image. The save button sends your strokes to this Colab session.

In [ ]:
# @title Draw your strokes on a canvas { display-mode: "form" }
for k, it in enumerate(own):
    show_canvas(k, it["image"], f"image {k + 1} of {len(own)} · {it['name']}")
if not IN_COLAB:
    print("ℹ️ Not in Colab: the canvases show, but saving needs Colab. Use the PNG fallback below.")

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#FDECEA;border-left:6px solid #8B1A10;border-radius:6px;padding:12px 14px;margin:6px 0;color:#8B1A10;font-family:inherit;line-height:1.5"><b>⚠️ Pitfall: no canvas, or the save button does nothing?</b><p>Some browsers or extensions block the canvas. The fallback:</p>
<ol>
<li>Open a copy of your image in any paint program (Preview, Paint, a phone's markup tool).</li>
<li>Draw the thing strokes in $\color{#FF0000}{\blacksquare}$ <strong>pure red</strong> (255, 0, 0) and the background
   strokes in $\color{#0000FF}{\blacksquare}$ <strong>pure blue</strong>
   (0, 0, 255), and save it as PNG.</li>
<li>Upload it with the folder icon on the left and put its path in <code>MY_STROKE_PNGS</code> below, one per image,
   in the order above (<code>""</code> skips one). Any size works.</li>
</ol></div>"""))

This cell gathers the strokes of every image, counts them, and shows them next to the photos.

What to look for: $\color{#00838F}{\blacksquare}$ teal strokes on the thing,
$\color{#9E9E9E}{\blacksquare}$ grey strokes on what you do not want. If an image shows none,
the next cells use our generated strokes for it.

In [ ]:
# @title Collect your strokes { display-mode: "form" }
MY_STROKE_PNGS = []  # @param {type:"raw"}
ladder_own = collect_strokes(MY_STROKE_PNGS)

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#FFE8D6;border-left:6px solid #7A2E00;border-radius:6px;padding:12px 14px;margin:6px 0;color:#7A2E00;font-family:inherit;line-height:1.5"><b>✏️ Your turn (optional, 2 min): prompt SAM yourself</b><p>Before the ladder runs, try SAM by hand on your images. Click on the thing, shift-click on what you do not
want, drag a box, or type a few words and press "Ask SAM 3". This is the same canvas idea as above, with
SAM answering at once.</p></div>"""))

In [ ]:
# @title Prompt SAM on your images { display-mode: "form" }
sam_playground(own, "Your images: click, box, or a word", text_prompt=True)

### 9.3 The ladder on your images

Three rungs, the same recipes as before, nothing new to learn.

- **Zero-shot**: SAM with one box, the box a person would draw around the thing (here: around your thing
  strokes, grown by 5 % of the image).
- **Your scribble head**: the section-4 head, a linear layer on DINOv3 patch features. It is trained with
  the partial cross-entropy on your strokes only (every unlabelled pixel ignored), on all your images
  together.
- **SAM from your strokes**: ten clicks along your thing strokes and ten along your background strokes,
  one SAM prompt per image (Path 3, section 7.1).

This helper cell defines the three rungs, a stroke check (no ground truth needed) and the panels.

In [ ]:
# @title The ladder on your images: three rungs, scoring, panels { display-mode: "form" }
OWN_APPROACHES = ("zero_shot", "scribbles", "sam_clicks")  # the colours and names of sections 4 and 7


def strokes_box(item: dict, pad: float) -> list[float]:
    """Zero-shot's box: around every 'thing' stroke, grown by `pad` × the long side, inside the image."""
    ys, xs = np.nonzero(item["scribbles"] == 1)
    h, w = item["scribbles"].shape
    g = pad * max(h, w)
    x0, y0 = max(0.0, xs.min() - g), max(0.0, ys.min() - g)
    return [float(x0), float(y0), float(min(w - 1.0, xs.max() + g)), float(min(h - 1.0, ys.max() + g))]


def own_ladder(items: list[dict], n_clicks: int, box_pad: float, feat_side: int, steps: int):
    """The three rungs on `items`. Returns masks per approach (one per image), seconds per approach, and the
    seconds the head took to train."""
    preds, sec = {k: [] for k in OWN_APPROACHES}, dict.fromkeys(OWN_APPROACHES, 0.0)
    for it in tqdm(items, desc="SAM on your images", leave=False):  # the image is encoded once, then reused
        t0 = time.monotonic()
        preds["zero_shot"].append(sam.probability(it["image"], box=strokes_box(it, box_pad)) >= 0.5)
        t1 = time.monotonic()
        pos, neg = stroke_points(it, 1, n_clicks), stroke_points(it, 2, n_clicks)
        prob = sam.probability(it["image"], points=pos + neg, labels=[1] * len(pos) + [0] * len(neg))
        preds["sam_clicks"].append(prob >= 0.5)
        sec["zero_shot"] += t1 - t0  # includes the image encoding
        sec["sam_clicks"] += time.monotonic() - t1
    t0 = time.monotonic()
    feats = [dino.features(it["image"], min(feat_side, max(it["image"].shape[:2]))) for it in items]
    # the rung's supervision: your strokes only; every unlabelled pixel is ignored (-1)
    labels = [stroke_targets(it["scribbles"], region(it)) for it in items]
    head, train_sec = train_head(feats, labels, steps)
    pairs = zip(feats, items, strict=True)
    preds["scribbles"] = [predict_head(head, f, it["image"].shape[:2]) for f, it in pairs]
    sec["scribbles"] = time.monotonic() - t0
    return preds, sec, train_sec


def stroke_agreement(pred: np.ndarray, item: dict) -> tuple[float, float]:
    """Share of the thing-stroke pixels the mask covers, and share of the background-stroke pixels it leaves
    out (nan if the image has no background stroke). Both are 1.0 for a mask that respects every stroke."""
    s = item["scribbles"]
    spared = 1.0 - float(pred[s == 2].mean()) if (s == 2).any() else float("nan")
    return float(pred[s == 1].mean()), spared


def score_own_ladder(items: list[dict], preds: dict, sec: dict) -> dict:
    """Per approach: the share of the image marked, the two stroke checks, the seconds, and (the example
    only, its ground truth scores and never trains: L-005) the IoU inside the evaluation region."""
    has_truth = all(it["mask"] is not None for it in items)
    report = {}
    for k in OWN_APPROACHES:
        agree = [stroke_agreement(p, it) for p, it in zip(preds[k], items, strict=True)]
        report[k] = {
            "marked": float(np.mean([p.mean() for p in preds[k]])),
            "thing_strokes_covered": float(np.mean([a[0] for a in agree])),
            "bg_strokes_spared": float(np.nanmean([a[1] for a in agree])),
            "seconds": round(sec[k], 1),
        }
        if has_truth:
            pairs = zip(preds[k], items, strict=True)
            report[k]["iou"] = float(
                np.mean([iou(p & region(it), it["mask"] & region(it)) for p, it in pairs])
            )
    return report


def show_own_ladder(items: list[dict], preds: dict, report: dict) -> None:
    """The table of the three rungs, then one row of panels per image: strokes, the three masks, and the
    ground truth when there is one."""
    has_truth = all(it["mask"] is not None for it in items)
    print(
        f"{'approach':<36}{'marks':>7}{'thing ✓':>9}{'bg ✓':>7}{'s':>6}"
        + (f"{'IoU':>7}" if has_truth else "")
    )
    for k in OWN_APPROACHES:
        v = report[k]
        print(
            f"{approach_names[k]:<36}{v['marked']:>7.0%}{v['thing_strokes_covered']:>9.0%}"
            f"{v['bg_strokes_spared']:>7.0%}{v['seconds']:>6.1f}" + (f"{v['iou']:>7.3f}" if has_truth else "")
        )
    print("marks = share of the image the mask covers")
    print("thing ✓ = share of your thing-stroke pixels inside the mask (100 % = all covered)")
    print("bg ✓ = share of your background-stroke pixels outside the mask (100 % = none marked)")
    key = [*STROKE_KEY, "mask", *(["truth"] if has_truth else [])]
    for n, it in enumerate(items):
        panels = [(f"{it['name']}\nyour strokes", stroke_panel(it))]
        for k in OWN_APPROACHES:
            r_n = region(it)
            tag = f"\nIoU {iou(preds[k][n] & r_n, it['mask'] & r_n):.2f}" if has_truth else ""
            shown = overlay(it["image"], preds[k][n], THEME["works"])
            panels.append((f"{approach_names[k].split(':')[0]}{tag}", shown))
        if has_truth:
            panels.append(("ground truth\n(scoring only)", overlay(it["image"], it["mask"], THEME["truth"])))
        show_masks(panels, width=2.6, legend=key)


def run_own_ladder(items: list[dict], n_clicks: int, box_pad: float) -> dict:
    """Q21: the three rungs on `items`, the table and the panels. Stores RESULTS["own_images"] and
    RESULTS["own_images_ok"]. Returns the masks per approach (one per image)."""
    if not isinstance(n_clicks, int) or not 1 <= n_clicks <= 50:
        raise SystemExit("N_OWN_CLICKS must be a whole number from 1 to 50.")
    if not isinstance(box_pad, (int, float)) or not 0 <= box_pad <= 0.5:
        raise SystemExit("OWN_BOX_PAD must be a number from 0 to 0.5 (a share of the image's long side).")
    seed_everything()
    preds, sec, train_sec = own_ladder(items, n_clicks, box_pad, 256 if SMOKE else 512, HEAD_STEPS)
    if train_sec > 60:
        print("ℹ️ Slow session? The head trained for over a minute: fewer or smaller images make it faster.")
    report = score_own_ladder(items, preds, sec)
    show_own_ladder(items, preds, report)
    n_strokes, minutes = stroke_effort(items)
    RESULTS["own_images"] = {
        "source": items[0]["source"],
        "n_images": len(items),
        "n_strokes": n_strokes[1] + n_strokes[2],
        "effort_min": round(minutes, 2),
        "head_train_sec": round(float(train_sec), 2),
        "approaches": report,
    }
    RESULTS["own_images_ok"] = all(len(preds[k]) == len(items) for k in OWN_APPROACHES)  # SPEC §9
    return preds

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F3E5F5;border-left:6px solid #4A148C;border-radius:6px;padding:12px 14px;margin:6px 0;color:#4A148C;font-family:inherit;line-height:1.5"><b>🤔 Your prediction (1 min)</b><p>Look at your images and strokes above. Is your thing an "object" in SAM's sense (compact, a clear inside
and outside), or more like the vessels (thin, scattered, a texture)?</p>
<ul>
<li>Write down which of the three rungs you expect to give the cleanest mask, and which the worst.</li>
<li>Then run the next cell (seconds on a T4, much slower on CPU). Each row is one image: your strokes, the
  three masks in <span style="color:#6A1B9A">■</span> violet, and on the example the ground truth in
  <span style="color:#2E7D32">■</span> green with the IoU.</li>
<li>What to look for: where the three <span style="color:#6A1B9A">■</span> violet masks differ, and which one
  respects your strokes.</li>
</ul>
<p><strong>Q21.</strong> 🟡 Which rung gave the mask you would keep, image by image, and did it match your prediction?
Where a rung failed, was it:</p>
<ul>
<li>SAM's idea of an object (the box or the clicks grabbed the wrong thing, or too much)?</li>
<li>the 16 px patches of the head (blocky edges, small parts lost)?</li>
<li>your strokes (a part of the thing you never touched)?</li>
</ul></div>"""))

In [ ]:
# @title Q21 — the ladder on your images: box, scribble head, SAM clicks { display-mode: "form" }
# Clicks per kind (thing / background) and image, spread along your strokes.
N_OWN_CLICKS = 10  # @param {type:"slider", min:1, max:50, step:1}
# Zero-shot box: your thing strokes' bounding box, grown by this share of the long side.
OWN_BOX_PAD = 0.05  # @param {type:"slider", min:0, max:0.5, step:0.05}
own_preds = run_own_ladder(ladder_own, N_OWN_CLICKS, OWN_BOX_PAD)

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F5F5F5;border-left:6px solid #616161;border-radius:6px;padding:12px 14px;margin:6px 0;color:#616161;font-family:inherit;line-height:1.5"><details><summary><b>🔍 How to read the stroke checks and the masks</b></summary><ul>
<li>The two percentages are weak checks: SAM's clicks sit on your strokes and the head trained on them, so
  both should respect them. A low number means the approach ignored what you drew.</li>
<li>The masks matter more. A compact thing makes the rungs agree, as on the pets. A thin or textural thing
  brings section 2 back: a box grabs a region, the head draws at patch resolution.</li>
</ul></details></div>"""))

### 9.4 No ground truth: look where they disagree

Without a ground truth, run two or three approaches and look at the pixels where they disagree. That is
where the image is hard, and where one more stroke is worth the most.

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#EDE7F6;border-left:6px solid #311B92;border-radius:6px;padding:12px 14px;margin:6px 0;color:#311B92;font-family:inherit;line-height:1.5"><b>📘 Theory: the disagreement map</b><p>The map is <span style="color:#D81B60">■</span> pink on every pixel that <strong>some</strong> of the three masks mark
and <strong>others</strong> do not. A thin rim
along the thing means the rungs agree on <em>what</em> it is and differ on the edge. Whole patches mean they
disagree on what belongs to the thing.</p></div>"""))

This helper cell defines `disagreement`: marked by some masks, not by all.

In [ ]:
# @title disagreement: marked by some masks, not by all { display-mode: "form" }
def disagreement(masks: list[np.ndarray]) -> np.ndarray:
    """True where at least one mask marks the pixel and at least one does not.
    e.g. three 1×4 masks [1, 1, 0, 0], [1, 0, 1, 0], [1, 1, 1, 0] -> [False, True, True, False]"""
    stack = np.stack([np.asarray(m, bool) for m in masks])
    return stack.any(0) & ~stack.all(0)

This cell computes the disagreement of the three rungs on each of your images and shows it in
$\color{#D81B60}{\blacksquare}$ pink.

What to look for: a thin $\color{#D81B60}{\blacksquare}$ pink rim, or whole
$\color{#D81B60}{\blacksquare}$ pink parts, and whether the $\color{#D81B60}{\blacksquare}$ pink sits on
your strokes.

In [ ]:
# @title Disagreement map of the three rungs { display-mode: "form" }
own_disagree = [disagreement([own_preds[k][n] for k in OWN_APPROACHES]) for n in range(len(ladder_own))]
RESULTS["own_images"]["disagreement"] = float(np.mean([d.mean() for d in own_disagree]))
DISAGREE = "#D81B60"  # pink: the rungs disagree (orange is kept for "invented" in the error maps)
panels = []
for it, d in zip(ladder_own, own_disagree, strict=True):
    print(f"{it['name']}: the three rungs disagree on {d.mean():.0%} of the image")
    panels += [
        (f"{it['name']}\nyour strokes", stroke_panel(it)),
        (f"disagreement · {d.mean():.0%} of the image", overlay(it["image"], d, DISAGREE)),
    ]
show_masks(
    panels,
    width=2.8,
    legend=[*STROKE_KEY, ("some rungs mark it, others do not", DISAGREE)],
)

**Q22.** 🟡 Where is the $\color{#D81B60}{\blacksquare}$ pink?

- Along the boundary of the thing?
- On whole parts of it?
- On something in the background that looks similar?

If you could add only two more strokes per image, where would you put them, and with which pen? On the
example, compare the retina's $\color{#D81B60}{\blacksquare}$ pink with the pets'. To test your answer,
draw those strokes on the canvas
of 9.2, save, and run the cells from the stroke collection again.

<details><summary>What to look for</summary>

- A thin $\color{#D81B60}{\blacksquare}$ pink rim: the rungs agree on *what* it is, and differ on the exact
  edge (SAM's sharp boundary
  against the head's 16 px squares). More strokes will not fix that.
- Whole $\color{#D81B60}{\blacksquare}$ pink parts: they disagree on what belongs to the thing, usually a
  part you never stroked. That is
  where the next stroke pays.
- After redrawing, SAM from clicks changes only near the new clicks. The head changes on every image,
  because it learns one rule from all strokes: a trained head spreads annotation everywhere.

</details>

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F3E5F5;border-left:6px solid #4A148C;border-radius:6px;padding:12px 14px;margin:6px 0;color:#4A148C;font-family:inherit;line-height:1.5"><b>👥 Pairs, 5 min: compare your images</b><p>Show your neighbour your strokes, your three masks and your disagreement map.</p>
<ul>
<li>Whose thing was easier for the foundation model, and what made it easier: its shape, its contrast, how
  "photo-like" it is?</li>
<li>Did the same rung win on both your images?</li>
<li>Agree on one sign you would look for, on a new problem, that says "a box will do" versus "I need
  strokes and a head".</li>
</ul></div>"""))

<details><summary>What pairs usually find</summary>

- Compact, everyday objects (a mug, a pet, a leaf on a plain background) make the rungs agree, and the
  zero-shot box is often the mask you would keep.
- Thin or textural things (cracks, cells, vessels, defects) make them disagree, and the trained head or
  many small prompts start to matter.
- The sign from section 2: no clear inside and outside, or smaller than a 16 px patch, means plan for
  strokes.

</details>

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#E0F2F1;border-left:6px solid #004D40;border-radius:6px;padding:12px 14px;margin:6px 0;color:#004D40;font-family:inherit;line-height:1.5"><b>✅ When you see X, do Y: on your own images</b><ul>
<li><strong>When</strong> your thing is compact and photo-like and the three rungs agree <strong>→</strong> the zero-shot box is
  probably enough; stop there.</li>
<li><strong>When</strong> the rungs disagree on whole parts of the thing <strong>→</strong> add a thing stroke on the missed part or
  a background stroke on the false alarm, then rerun.</li>
<li><strong>When</strong> only a thin rim disagrees <strong>→</strong> that is the 16 px patch against SAM's sharp edge; more strokes
  will not fix it.</li>
<li><strong>When</strong> there is no ground truth <strong>→</strong> use your strokes and the disagreement map as a sanity check,
  then label a few images completely so the next decision is measured.</li>
</ul>
<p>Section 10 turns all of this into rules you can take to the next problem.</p></div>"""))

## 10. Guiding principles & wrap-up

A table is good for one decision; a rule is what you take to the next problem. This last section reads the
numbers *your* session produced in sections 1 to 8 and writes them up as a handful of principles, each
with its evidence.

They are not general truths about foundation models. They are what this run, on this task, with these
settings, showed. If your neighbour's session ran with other settings, some sentences may read
differently, and that is fine.

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#E8F5E9;border-left:6px solid #1B5E20;border-radius:6px;padding:12px 14px;margin:6px 0;color:#1B5E20;font-family:inherit;line-height:1.5"><b>🎯 Objective</b><p>turn the results into rules you can reuse.</p></div>"""))

This cell draws the whole story in one picture: the pets one box got almost right, the same box on the
specialised task, and the best approach of every path on the same hidden ground truth.

What to look for: the gap between the two one-box bars (where it fell short), then how far each path climbs
back towards the complete-masks bar at the bottom, and how far each bar sits above the dashed line (marking
everything).

In [ ]:
# @title The story in one picture { display-mode: "form" }
table = {r["approach"]: r for r in RESULTS["comparison_table"]}  # section 8's rows, by short name
everything_iou = RESULTS["specialised_everything_iou"]
PATHS = {  # the rows of section 8, grouped the way the notebook introduced them
    "ladder (section 4)": ("scribbles",),
    "path 1 (section 5)": ("naive", "masked", "pu"),
    "path 2 (section 6)": ("sam_pseudo", "gradcam_pseudo"),
    "path 3 (section 7)": ("sam_clicks", "sam_boxes"),
}
story = [("pets · one box (section 1)", RESULTS["pets_zero_shot_iou"], THEME["works"])]
story.append(
    (f"{specialised['name']} · one box (section 2)", table["zero_shot"]["iou"], approach_colours["zero_shot"])
)
for path, keys in PATHS.items():
    best = max(keys, key=lambda k: table[k]["iou"])
    story.append((f"best of {path}: {best}", table[best]["iou"], approach_colours[best]))
story.append(("complete masks (section 2)", table["full_masks"]["iou"], approach_colours["full_masks"]))

fig, ax = plt.subplots(figsize=(8, 3.4))
labels, values, colours = zip(*story[::-1], strict=True)
ax.barh(labels, values, color=colours)
for y, v in enumerate(values):
    ax.text(v + 0.01, y, f"{v:.2f}", va="center", fontsize=8, color="#311B92")
ax.axvline(everything_iou, color=THEME["background"], ls="--", label='IoU of "everything" (specialised task)')
ax.set_xlim(0, 1.08)
ax.set_xlabel("mean IoU on the held-out images")
ax.set_title("Where it worked, where it fell short,\nhow far each path got back")
ax.tick_params(axis="y", labelsize=8)
ax.legend(fontsize=7, frameon=False, loc="lower right")
add_key(fig, None, "bar colour = the approach's colour in section 8 (violet = the pets) · number = mean IoU")
plt.show()

### 10.1 The principles, from your numbers

The next cell writes six rules as cards. Each carries a number from your own session, which tells you how
strongly to believe it, and says so where a lesson did not hold in your run.

In [ ]:
# @title Principles from this run { display-mode: "form" }
import textwrap


def p_check_first() -> str:
    """Outcome 1: is the foundation model good enough out of the box?"""
    pets, spec = RESULTS["pets_zero_shot_iou"], RESULTS["specialised_zero_shot_iou"]
    rule = "Test the foundation model on your own task first, and read its score against marking everything."
    fact = (
        f"One box scored IoU {pets:.2f} on pets and {spec:.2f} on {specialised['name']}, where a mask over "
        f"the whole image scores {everything_iou:.2f}."
    )
    if spec < everything_iou + 0.1:
        return f"{rule} {fact} Out of the box it was close to doing nothing useful here."
    return f"{rule} {fact} It did part of the job out of the box; adapt only if the rest matters."


def p_resolution() -> str:
    """Section 2: the patch is 16 px; does a larger input help?"""
    res = sorted((int(k.split()[0]), v) for k, v in RESULTS["resolution_effect"].items())
    (lo, a), (hi, b) = res[0], res[-1]
    if specialised.get("thin_structures"):
        rule = "Thin structures defeat patch features: check the resolution first."
    else:
        rule = "Check the resolution first: a patch feature describes a 16 px block, not a pixel."
    fact = (
        f"The same head on complete masks scored IoU {a['iou']:.2f} at {lo} px and {b['iou']:.2f} at {hi} px "
        f"on pets the same kind of head scored {RESULTS['pets_head_iou']:.2f}."
    )
    if b["iou"] - a["iou"] < 0.05:
        return f"{rule} {fact} More pixels helped little, so the limit is the patch: plan for cheap labels."
    return f"{rule} {fact} More pixels helped, so try the larger input before anything more expensive."


def p_missing_is_unknown() -> str:
    """Sections 3 and 5: missing labels are unknown, not background."""
    naive, masked = RESULTS["naive_recall_hidden_fg"], RESULTS["masked_recall_hidden_fg"]
    rule = "An unlabelled pixel is unknown, not background: tell the loss so."
    fact = (
        f"On the same half-painted maps, the head that read the gaps as background found {naive:.0%} of the "
        f"hidden foreground, the masked loss {masked:.0%}."
    )
    if masked <= naive:
        return f"{rule} {fact} Here ignoring the gaps did not recover more; check what the strokes cover."
    pn, pm = table["naive"]["precision"], table["masked"]["precision"]
    if pm < pn:
        return f"{rule} {fact} The price was precision ({pn:.2f} → {pm:.2f}): fewer misses, more alarms."
    return f"{rule} {fact} And it kept its precision ({pn:.2f} → {pm:.2f})."


def p_cheapest_baseline() -> str:
    """Section 7: compare SAM with the hints alone."""
    alone = RESULTS["sam_direct_strokes_alone"]
    sam_rows = [k for k in ("zero_shot", "sam_clicks", "sam_boxes") if k in table]
    best = max(sam_rows, key=lambda k: table[k]["iou"])
    gain = table[best]["iou"] - alone["iou"]
    rule = "Compare the foundation model with the cheapest baseline built from the same hints."
    fact = (
        f"The strokes thickened by {alone['thicken_px']} px, no model at all, scored IoU {alone['iou']:.2f}; "
        f"the best SAM recipe, {best}, scored {table[best]['iou']:.2f}."
    )
    if gain <= 0:
        return f"{rule} {fact} The model added nothing the strokes did not already say."
    if gain < 0.05:
        return f"{rule} {fact} The model added only {gain:.2f}."
    return f"{rule} {fact} The model earned its place, by {gain:.2f}."


def p_one_number() -> str:
    """Section 8: no single column decides."""
    rows = RESULTS["comparison_table"]
    by_iou = max(rows, key=lambda r: r["iou"])
    by_recall = max(rows, key=lambda r: r["recall"])
    steady = min(rows, key=lambda r: r["reliability"])
    rule = "Choose by what a mistake costs, not by one number."
    fact = (
        f"Highest IoU: {by_iou['approach']} ({by_iou['iou']:.2f}); highest recall: {by_recall['approach']} "
        f"({by_recall['recall']:.2f}); smallest spread: {steady['approach']} ({steady['reliability']:.2f})."
    )
    leaders = {by_iou["approach"], by_recall["approach"], steady["approach"]}
    if len(leaders) == 1:
        return f"{rule} {fact} This time one approach led all three; check that before assuming it."
    lead = f"{len(leaders)} different approaches lead the three columns"
    return f"{rule} {fact} {lead}, so the right row depends on which mistake costs most."


def p_minutes_before_hours() -> str:
    """Section 8: annotation effort, once and per new image."""
    full = table["full_masks"]
    cheap_rows = [r for r in RESULTS["comparison_table"] if r["effort_min"] < full["effort_min"] / 10]
    cheap = max(cheap_rows or RESULTS["comparison_table"], key=lambda r: r["iou"])
    per_image = max(RESULTS["comparison_table"], key=lambda r: r["effort_min_per_new_image"])
    rule = "Spend minutes before hours: start with strokes, and buy complete masks only if the table says so."
    fact = (
        f"Complete masks cost {full['effort_min']:.0f} min of annotation for IoU {full['iou']:.2f}; "
        f"{cheap['approach']} needed {cheap['effort_min']:.1f} min for IoU {cheap['iou']:.2f}."
    )
    if cheap["iou"] >= full["iou"]:
        verdict = "The cheap labels matched or beat the expensive ones."
    else:
        ratio = full["effort_min"] / max(cheap["effort_min"], 0.01)
        verdict = f"The hours bought {full['iou'] - cheap['iou']:.2f} IoU for {ratio:.0f}× the effort."
    scale = (
        f"And count per new image: {per_image['approach']} costs "
        f"{per_image['effort_min_per_new_image']:.2f} min on every one, a trained head none."
    )
    return f"{rule} {fact} {verdict} {scale}"


principles = [
    p_check_first(),
    p_resolution(),
    p_missing_is_unknown(),
    p_cheapest_baseline(),
    p_one_number(),
    p_minutes_before_hours(),
]
RESULTS["principles"] = principles
headline = f"Principles from this session ({specialised['name']}, {len(spec_eval)} held-out images, {DEVICE})"
try:
    from html import escape

    from IPython.display import HTML, display

    cards = ""
    for n, text in enumerate(principles, start=1):
        rule, _, evidence = text.partition(". ")  # the rule is the first sentence, the rest is the evidence
        cards += (
            f'<div style="flex:1 1 300px;background:#EDE7F6;border-left:6px solid {THEME["works"]};'
            'border-radius:6px;padding:10px 12px;color:#311B92;line-height:1.45">'
            f'<div style="font-weight:bold;font-size:15px">{n}. {escape(rule.rstrip("."))}.</div>'
            f'<div style="font-size:12px;margin-top:5px;color:#4527A0">{escape(evidence)}</div></div>'
        )
    display(
        HTML(
            f'<div style="font-weight:bold;color:#311B92;margin:4px 0">{escape(headline)}</div>'
            f'<div style="display:flex;flex-wrap:wrap;gap:8px">{cards}</div>'
        )
    )
except ImportError:  # no IPython: plain text
    print(headline + ":\n")
    for n, text in enumerate(principles, start=1):
        print(textwrap.fill(text, width=100, initial_indent=f"{n}. ", subsequent_indent="   "), end="\n\n")

### 10.2 Your framework: six questions for any task

The six principles are the same six questions, asked in the order you would meet them on a new problem.
Each one has a number you can measure in an afternoon.

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#E0F2F1;border-left:6px solid #004D40;border-radius:6px;padding:12px 14px;margin:6px 0;color:#004D40;font-family:inherit;line-height:1.5"><b>✅ Six questions to ask on any new task</b><ol>
<li><strong>Is the out-of-the-box result above the "everything" line?</strong> Run one box on about 20 images and compare
   with marking everything. If not, plan to adapt (sections 1 and 2).</li>
<li><strong>Is the structure smaller than a 16 px patch?</strong> Train the same head at two input sizes. If the larger
   one barely helps, buy cheap labels, not a bigger model (section 2).</li>
<li><strong>What do the gaps in the labels mean?</strong> Unlabelled is unknown, not background. Tell the loss, and
   score recall on the hidden part (sections 3 and 5).</li>
<li><strong>Does the model beat the hints alone?</strong> Thicken your strokes with no model and compare (section 7).</li>
<li><strong>Which mistake costs most?</strong> A missed structure or a false alarm decides whether you rank by recall,
   precision or IoU (section 8).</li>
<li><strong>How many minutes does every new image cost?</strong> Strokes once is cheap, strokes on every image is not
   (section 8).</li>
</ol>
<p>Answer them in this order, write each answer as a number, and keep the table. That table is your
experiment.</p></div>"""))

This helper cell defines `show_my_plan`, which prints your plan next to the principle you picked.

In [ ]:
# @title show_my_plan: print your problem, your first step and the principle { display-mode: "form" }
def show_my_plan(problem: str, principle: int, first_step: str) -> None:
    """Print the problem and the first step, and the principle (1 to 6) they rest on. Checks nothing."""
    if not isinstance(principle, int) or not 0 <= principle <= len(principles):
        raise SystemExit(f"MY_PRINCIPLE must be a whole number from 0 to {len(principles)}.")
    if not (isinstance(problem, str) and isinstance(first_step, str)):
        raise SystemExit('MY_PROBLEM and MY_FIRST_STEP must be text in quotes, e.g. "my problem".')
    if problem or first_step:
        print(f"problem: {problem or '(not written)'}\nfirst step: {first_step or '(not written)'}")
        if principle:
            print(textwrap.fill(principles[principle - 1], width=100, initial_indent="because: "))
    else:
        print("nothing written; fine. Keep the question for the train home: what would you measure first?")

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#FFE8D6;border-left:6px solid #7A2E00;border-radius:6px;padding:12px 14px;margin:6px 0;color:#7A2E00;font-family:inherit;line-height:1.5"><b>✏️ Your turn (2 min)</b><p><strong>Q23.</strong> 🟢 Think of one segmentation or detection problem of your own (work, research, or the images of
section 9). Which of the six questions would you ask first, and what number would you measure to answer it?</p>
<ul>
<li>Write the problem and your first step on Monday, as a rule with a number, in the form of the next cell
  (for example "one box on 20 images, compared with the 'everything' line").</li>
<li>The cell prints them next to the principle you picked and checks nothing.</li>
</ul></div>"""))

In [ ]:
# @title Q23 — your problem and your first step (leave empty to skip) { display-mode: "form" }
MY_PROBLEM = ""  # @param {type:"string"}
# The number of the question or principle above that changes your plan (1 to 6), or 0 for none.
MY_PRINCIPLE = 0  # @param {type:"slider", min:0, max:6, step:1}
MY_FIRST_STEP = ""  # @param {type:"string"}
show_my_plan(MY_PROBLEM, MY_PRINCIPLE, MY_FIRST_STEP)

<details><summary>A possible answer</summary>

Take cracks in drone photos of bridges. They are thin, like vessels, so principle 2 changes the plan.

1. Run one box and one feature head on ten images at two input sizes.
2. See whether the larger size moves IoU at all.
3. If it does not, the patch is the limit: spend the first afternoon on strokes for twenty images.

The number to measure is the gap between the head and the "everything" line on a few images you label
completely. What carries over is writing the step with the number that would prove it wrong.

</details>

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F3E5F5;border-left:6px solid #4A148C;border-radius:6px;padding:12px 14px;margin:6px 0;color:#4A148C;font-family:inherit;line-height:1.5"><b>❓ Show of hands</b><ul>
<li>Which principle would you have got wrong this morning, before section 1?</li>
<li>Which one would you most like to check on your own data next week?</li>
</ul></div>"""))

### 10.3 How far from the state of the art?

The best IoU of the day is far from a finished vessel map, so it is fair to ask whether the methods are
worth anything. The next cell puts your bars next to three reference points that were measured outside this
notebook:

- **A small U-Net trained from scratch on the same five complete masks** (a plain convolutional network
  that works on the pixels, no foundation model). We trained it on our 1024 px crops for about three
  minutes on a laptop and scored it on the same 25 held-out images, with the cut chosen on the training
  images.
- **ESRVS** (Xu & Zhang, PRCV 2026, [arXiv:2607.24453](https://arxiv.org/abs/2607.24453)): DINOv3
  features and pseudo-labels, the ingredients of section 6, with **one** labelled image and 31 unlabelled
  ones, and a real decoder (Mask2Former) at 1024 px.
- **The best published fully supervised models on HRF**, trained on 15 complete masks: Dice 0.81 to 0.82
  (Little W-Net, Galdran et al. 2020, [arXiv:2009.01907](https://arxiv.org/abs/2009.01907), and Shin et
  al., MedIA 2019, in its Table 2).

The papers report Dice; the cell converts it to IoU (IoU = Dice ÷ (2 − Dice)). Their splits and image sizes
differ from ours, so read them as the right order of magnitude, not as an exact ranking.

What to look for: the gap between your best bar and the references, and which reference uses the same
ingredients as the notebook.

In [ ]:
# @title Your results next to the state of the art { display-mode: "form" }
def dice_to_iou(dice: float) -> float:
    """IoU from a Dice score of the same masks: IoU = Dice / (2 - Dice)."""
    return dice / (2 - dice)


sota_references = [  # (label, IoU, where it comes from): measured outside this notebook, see the text above
    ("small U-Net from scratch · same 5 complete masks", 0.581, "our run on these crops, 2026-10-26"),
    ("ESRVS · DINOv3 + pseudo-labels · 1 labelled image", dice_to_iou(0.7263), "arXiv:2607.24453, Table 3"),
    ("best published · 15 complete masks", dice_to_iou(0.8104), "Little W-Net, arXiv:2009.01907, Table 2"),
]
sota_rows = [(label, value, colour, False) for label, value, colour in story[1:]]  # the specialised task
sota_rows += [(label, value, "#BDBDBD", True) for label, value, _ in sota_references]
fig, ax = plt.subplots(figsize=(8, 4.2))
for y, (_label, value, colour, outside) in enumerate(sota_rows[::-1]):
    edge = "#616161" if outside else colour
    ax.barh(y, value, color=colour, hatch="//" if outside else None, edgecolor=edge)
    ax.text(value + 0.01, y, f"{value:.2f}", va="center", fontsize=8, color="#311B92")
ax.set_yticks(range(len(sota_rows)), [r[0] for r in sota_rows[::-1]], fontsize=8)
ax.axvline(everything_iou, color=THEME["background"], ls="--")
ax.set_xlim(0, 1)
ax.set_xlabel("mean IoU on held-out retina images")
ax.set_title("Your session (solid) and three reference points (hatched)")
add_key(
    fig,
    [
        ("solid: your session (colour = the approach, as in section 8)", "#455A64"),
        ("measured outside this notebook (hatched)", "#BDBDBD"),
        ('IoU of marking everything (dashed line)', THEME["background"]),
    ],
)
plt.show()
for label, value, source in sota_references:
    print(f"{label:<52} IoU {value:.2f} · {source}")

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#E0F2F1;border-left:6px solid #004D40;border-radius:6px;padding:12px 14px;margin:6px 0;color:#004D40;font-family:inherit;line-height:1.5"><b>✅ Why it was not all for nothing</b><ul>
<li><strong>The limit you met is the patch, not the idea.</strong> Every head today decides per 16 px patch, and the
  vessels are 4 to 20 px wide (section 2.4). A small U-Net that sees the pixels reaches IoU 0.58 on the same
  five complete masks, about twice our full-mask head (0.28 on our run). Choosing the cut on the training
  images alone lifts our head from 0.28 to 0.33, so part of the gap is also a default threshold.</li>
<li><strong>Complete masks are what the U-Net needs.</strong> Those five masks cost 15 to 25 hours of annotation. Trained
  on the strokes alone, the same U-Net marked almost the whole fundus (IoU 0.11 on our run), below the
  scribble head on foundation-model features (0.17). Cheap labels need the foundation model.</li>
<li><strong>The same ingredients, at full strength, are close to the top.</strong> ESRVS combines DINOv3 features,
  pseudo-labels and one labelled image with a decoder that works at pixel level, and keeps about 97 % of
  its fully supervised Dice on HRF.</li>
<li><strong>What you would change first on a real project:</strong> a decoder that sees the pixels on top of the frozen
  features, the threshold chosen on a few fully labelled images, and the unlabelled images of your archive
  as pseudo-label material.</li>
</ul></div>"""))

### 10.4 What you can do now

At the end you can:

1. **Judge when a vision foundation model is good enough out of the box**, and recognise the signs that it
   is not (sections 1 and 2).
2. **Adapt it with cheap, incomplete supervision**, and explain what each approach uses and assumes
   (sections 3 to 7).
3. **Compare approaches** on accuracy, annotation effort, reliability and compute, and decide what to do
   next on a new problem, including your own images (sections 8 and 9).

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F5F5F5;border-left:6px solid #616161;border-radius:6px;padding:12px 14px;margin:6px 0;color:#616161;font-family:inherit;line-height:1.5"><details><summary><b>🔍 Go deeper · where to go next</b></summary><ul>
<li><strong>Label a small test set first</strong>: a few fully annotated images turn every later decision into a
  measurement, as the hidden ground truth did here.</li>
<li><strong>Let the model choose the next stroke</strong>: draw where the approaches disagree (section 9), retrain, repeat.</li>
<li><strong>A light decoder instead of a linear head</strong>: a small upsampling decoder on the same frozen features
  comes before any fine-tuning.</li>
<li><strong>Sources</strong>: <a style="color:inherit;text-decoration:underline" href="https://github.com/facebookresearch/dinov3">DINOv3</a> ·
  <a style="color:inherit;text-decoration:underline" href="https://github.com/facebookresearch/sam2">SAM 2</a> ·
  <a style="color:inherit;text-decoration:underline" href="https://github.com/jacobgil/pytorch-grad-cam">pytorch-grad-cam</a></li>
</ul></details></div>"""))

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#E8EAF6;border-left:6px solid #1A237E;border-radius:6px;padding:12px 14px;margin:6px 0;color:#1A237E;font-family:inherit;line-height:1.5"><b>🔀 Across approaches, one last time</b><p>The question on the first screen was what modern AI solves out of the box, when its limits start to
matter, and what to do next.</p>
<p>Your session's answer is in the six principles above, with numbers. The foundation model was a starting
point, not an answer, and every way out paid for its gain in a different currency: minutes of strokes, a
wrong assumption about the gaps, a second model's mistakes, or a person pointing at every image.</p>
<p>On your next problem, measure first, spend minutes before hours, and keep the table. Thank you for taking
part.</p></div>"""))

In [ ]:
# @title Credits: models and datasets of this session { display-mode: "form" }
cfg = specialised
print(f"DINOv3 ViT-S/16 · {DINO_SOURCE} · {DINO_LICENCE}")
print(f"SAM 2.1 hiera-small · {SAM_ID} · {SAM_LICENCE}")
print(f"{pets_task['name']} · {pets_task['licence']}")
print(f"{cfg['name']} · {cfg['licence']} ({cfg['licence_url']})")
print(textwrap.fill(cfg["citation"], width=100, initial_indent="  ", subsequent_indent="  "))
RESULTS["credits"] = {
    "models": {DINO_SOURCE: DINO_LICENCE, SAM_ID: SAM_LICENCE},
    "datasets": {pets_task["name"]: pets_task["licence"], specialised["name"]: specialised["licence"]},
}

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F5F5F5;border-left:6px solid #616161;border-radius:6px;padding:12px 14px;margin:6px 0;color:#616161;font-family:inherit;line-height:1.5"><details><summary><b>🔍 Credits &amp; licences</b></summary><ul>
<li><strong>DINOv3</strong> ViT-S/16 (Meta), via the <code>timm</code> re-host
  <a style="color:inherit;text-decoration:underline" href="https://huggingface.co/timm/vit_small_patch16_dinov3.lvd1689m">timm/vit_small_patch16_dinov3.lvd1689m</a>.</li>
<li>DINOv3 is licensed under the
  <a style="color:inherit;text-decoration:underline" href="https://huggingface.co/timm/vit_small_patch16_dinov3.lvd1689m">DINOv3 License</a>, Copyright © Meta
  Platforms, Inc. All Rights Reserved. It allows research and commercial use under Meta's conditions and
  acceptable-use policy; keep this notice if you redistribute the weights or anything derived from them.
  SAM 2.1 is Apache-2.0.</li>
<li><strong>SAM 2.1</strong> hiera-small (Meta),
  <a style="color:inherit;text-decoration:underline" href="https://huggingface.co/facebook/sam2.1-hiera-small">facebook/sam2.1-hiera-small</a>. SAM 3, if you used
  it, comes under Meta's own SAM 3 licence, which you accepted on Hugging Face.</li>
<li><strong>Oxford-IIIT Pet</strong>: Parkhi, Vedaldi, Zisserman and Jawahar, <em>Cats and Dogs</em>, CVPR 2012,
  <a style="color:inherit;text-decoration:underline" href="https://creativecommons.org/licenses/by-sa/4.0/">CC BY-SA 4.0</a>. The subset shipped with this workshop
  is resized and shared under the same licence.</li>
<li><strong>pytorch-grad-cam</strong> (Jacob Gildenblat and contributors), MIT.</li>
<li>The specialised dataset is printed by the credits cell above, read from the dataset config.</li>
<li>Every derived file (scribbles, hidden labels, pseudo-labels, features) carries its source's licence.
  The full list is in <code>data/ATTRIBUTION.md</code>.</li>
</ul></details></div>"""))